In [ ]:

import torch
import platform

print("=" * 60)
print("GRAC-LM v0.4 — NEW BASE MODEL")
print("=" * 60)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")
else:
    print("WARNING: GPU NOT DETECTED!")

print("=" * 60)


GRAC-LM v0.4 — NEW BASE MODEL
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA: 13.0
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:

!pip -q install -U datasets tokenizers safetensors
print("GRAC-LM v0.4 dependencies ready!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
GRAC-LM v0.4 dependencies ready!


In [ ]:

GRAC_V04_CONFIG = {
    "vocab_size": 8000,
    "context_length": 512,
    "embedding_dim": 640,
    "num_heads": 10,
    "num_layers": 10,
    "dropout": 0.1,

    # Initial training settings
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,

    "model_name": "GRAC-LM",
    "version": "0.4",
}

print("=" * 55)
print("GRAC-LM v0.4 — MODEL CONFIGURATION")
print("=" * 55)

for key, value in GRAC_V04_CONFIG.items():
    print(f"{key:30}: {value}")

assert GRAC_V04_CONFIG["embedding_dim"] % GRAC_V04_CONFIG["num_heads"] == 0

print("\nArchitecture configuration validated!")


GRAC-LM v0.4 — MODEL CONFIGURATION
vocab_size                    : 8000
context_length                : 512
embedding_dim                 : 640
num_heads                     : 10
num_layers                    : 10
dropout                       : 0.1
micro_batch_size              : 4
gradient_accumulation_steps   : 4
learning_rate                 : 0.0003
weight_decay                  : 0.1
model_name                    : GRAC-LM
version                       : 0.4

Architecture configuration validated!


In [ ]:

from google.colab import files
from tokenizers import Tokenizer
from pathlib import Path

print("GRAC-LM v0.4 — TOKENIZER SETUP")
print("=" * 50)

print("Upload your original grac_tokenizer.json")

uploaded = files.upload()

if "grac_tokenizer.json" not in uploaded:
    raise FileNotFoundError(
        "Please upload grac_tokenizer.json"
    )

tokenizer = Tokenizer.from_file(
    "/content/grac_tokenizer.json"
)

vocab_size = tokenizer.get_vocab_size()

print("\nTokenizer loaded successfully!")
print("Vocabulary size:", vocab_size)

assert vocab_size == GRAC_V04_CONFIG["vocab_size"], (
    f"Vocabulary mismatch: {vocab_size}"
)

test_text = "Hello! I am GRAC-LM v0.4, an AI language model."

encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded.ids, skip_special_tokens=False)

print("\nOriginal text:", test_text)
print("Token IDs:", encoded.ids)
print("Decoded text:", decoded)

print("\nGRAC tokenizer verification complete!")


GRAC-LM v0.4 — TOKENIZER SETUP
Upload your original grac_tokenizer.json


Saving grac_tokenizer.json to grac_tokenizer.json

Tokenizer loaded successfully!
Vocabulary size: 8000

Original text: Hello! I am GRAC-LM v0.4, an AI language model.
Token IDs: [43, 425, 82, 4, 293, 791, 329, 6619, 38, 16, 47, 48, 354, 19, 17, 23, 15, 334, 7321, 3206, 3406, 17]
Decoded text: Hello! I am GRAC-LM v0.4, an AI language model.

GRAC tokenizer verification complete!


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class GRACCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        dim = config["embedding_dim"]
        self.num_heads = config["num_heads"]
        self.head_dim = dim // self.num_heads

        self.qkv = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(config["dropout"])

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.dropout(self.output(y))


class GRACFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        dim = config["embedding_dim"]

        self.network = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
            nn.Dropout(config["dropout"])
        )

    def forward(self, x):
        return self.network(x)


class GRACTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        dim = config["embedding_dim"]

        self.norm1 = nn.LayerNorm(dim)
        self.attention = GRACCausalSelfAttention(config)

        self.norm2 = nn.LayerNorm(dim)
        self.feed_forward = GRACFeedForward(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


class GRACLM(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        vocab = config["vocab_size"]
        dim = config["embedding_dim"]
        context = config["context_length"]

        self.token_embedding = nn.Embedding(vocab, dim)
        self.position_embedding = nn.Embedding(context, dim)

        self.blocks = nn.ModuleList([
            GRACTransformerBlock(config)
            for _ in range(config["num_layers"])
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab, bias=False)

        # Share input and output embedding weights.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape

        if T > self.config["context_length"]:
            raise ValueError("Sequence exceeds context length")

        positions = torch.arange(T, device=input_ids.device)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        return logits, loss


print("GRAC-LM v0.4 architecture defined successfully!")


GRAC-LM v0.4 architecture defined successfully!


In [ ]:

import torch

print("=" * 60)
print("GRAC-LM v0.4 — INITIALIZATION TEST")
print("=" * 60)

# 1. Select the GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "T4 GPU is required!"

# 2. Create the new model
torch.manual_seed(42)

model = GRACLM(GRAC_V04_CONFIG).to(device)

# 3. Count parameters
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Model: GRAC-LM v0.4")
print("Device:", torch.cuda.get_device_name(0))
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Parameters (millions): {total_params / 1e6:.2f}M")

# 4. Test a forward pass
model.eval()

test_ids = torch.randint(
    low=0,
    high=GRAC_V04_CONFIG["vocab_size"],
    size=(2, 32),
    device=device
)

with torch.no_grad():
    logits, loss = model(test_ids)

expected_shape = (
    2,
    32,
    GRAC_V04_CONFIG["vocab_size"]
)

assert logits.shape == expected_shape, (
    f"Unexpected output shape: {logits.shape}"
)

assert torch.isfinite(logits).all(), (
    "Model produced invalid values!"
)

print("\nForward-pass test:")
print("Input shape:", tuple(test_ids.shape))
print("Output shape:", tuple(logits.shape))
print("Finite outputs:", bool(torch.isfinite(logits).all()))
print("Loss:", loss)

# 5. Display GPU memory usage
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("\nGPU memory:")
print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")

print("\nGRAC-LM v0.4 INITIALIZATION SUCCESSFUL!")


GRAC-LM v0.4 — INITIALIZATION TEST
Model: GRAC-LM v0.4
Device: Tesla T4
Total parameters: 54,684,160
Trainable parameters: 54,684,160
Parameters (millions): 54.68M

Forward-pass test:
Input shape: (2, 32)
Output shape: (2, 32, 8000)
Finite outputs: True
Loss: None

GPU memory:
Allocated: 0.21 GB
Reserved: 0.24 GB

GRAC-LM v0.4 INITIALIZATION SUCCESSFUL!


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil
import json
import torch

# Mount Google Drive
drive.mount("/content/drive")

# Create permanent project folders
GRAC_ROOT = Path("/content/drive/MyDrive/GRAC")
V04_DIR = GRAC_ROOT / "GRAC-LM-v0.4"
CHECKPOINT_DIR = V04_DIR / "checkpoints"
DATA_DIR = V04_DIR / "data"
TOKENIZER_DIR = V04_DIR / "tokenizer"

for folder in [CHECKPOINT_DIR, DATA_DIR, TOKENIZER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Preserve tokenizer permanently
tokenizer_source = Path("/content/grac_tokenizer.json")
tokenizer_destination = TOKENIZER_DIR / "grac_tokenizer.json"

assert tokenizer_source.exists(), "Tokenizer file not found!"

shutil.copy2(tokenizer_source, tokenizer_destination)

# Save architecture configuration
config_path = V04_DIR / "config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(GRAC_V04_CONFIG, f, indent=2)

# Save the INITIAL untrained model separately
initial_path = CHECKPOINT_DIR / "GRAC-LM-v0.4-initial.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "config": GRAC_V04_CONFIG,
    "training_steps": 0,
    "status": "untrained"
}, initial_path)

print("=" * 60)
print("GRAC-LM v0.4 — PERMANENT STORAGE")
print("=" * 60)
print("Project:", V04_DIR)
print("Tokenizer:", tokenizer_destination)
print("Configuration:", config_path)
print("Initial checkpoint:", initial_path)
print("Checkpoint size:", round(initial_path.stat().st_size / 1024**2, 2), "MB")
print("\nGRAC-LM v0.4 files saved to Google Drive!")


Mounted at /content/drive
GRAC-LM v0.4 — PERMANENT STORAGE
Project: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4
Tokenizer: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/tokenizer/grac_tokenizer.json
Configuration: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/config.json
Initial checkpoint: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/checkpoints/GRAC-LM-v0.4-initial.pt
Checkpoint size: 208.65 MB

GRAC-LM v0.4 files saved to Google Drive!


In [ ]:

from datasets import load_dataset
import random

print("=" * 60)
print("GRAC-LM v0.4 — DATASET PREPARATION")
print("=" * 60)

# Load WikiText-103 (raw, not pre-tokenized)
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1"
)

print("\nDataset splits:")
for split in dataset:
    print(f"{split}: {len(dataset[split]):,} rows")

# Filter out empty lines
train_texts = [
    row["text"].strip()
    for row in dataset["train"]
    if row["text"].strip()
]

validation_texts = [
    row["text"].strip()
    for row in dataset["validation"]
    if row["text"].strip()
]

print("\nNon-empty text rows:")
print("Training:", len(train_texts))
print("Validation:", len(validation_texts))

# Inspect a few examples
rng = random.Random(42)

print("\nSAMPLE TRAINING TEXT")
print("-" * 60)

for i, sample in enumerate(rng.sample(train_texts, 3), 1):
    print(f"\nExample {i}:")
    print(sample[:500])

print("\nDataset inspection complete!")


GRAC-LM v0.4 — DATASET PREPARATION


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


Dataset splits:
test: 4,358 rows
train: 1,801,350 rows
validation: 3,760 rows

Non-empty text rows:
Training: 1165029
Validation: 2461

SAMPLE TRAINING TEXT
------------------------------------------------------------

Example 1:
The wings may be present in only one sex ( often the male ) in some groups such as velvet ants and Strepsiptera , or selectively lost in " workers " of social insects such as ants and termites . Rarely , the female is winged but the male not , as in fig wasps . In some cases , wings are produced only at particular times in the life cycle , such as in the dispersal phase of aphids . Beyond the mere presence / absence of wings , the structure and colouration will often vary with morphs , such as i

Example 2:
Reproduction is one of the few times the Sunda slow loris aggregates with conspecifics , as it is largely solitary . One study recorded that the most slow lorises ever seen together was six ; this appeared to be a female in estrus and five males following 

In [ ]:

print("=" * 60)
print("GRAC-LM v0.4 — TOKENIZATION TEST")
print("=" * 60)

sample_text = next(
    text for text in train_texts
    if len(text) > 100
)

encoding = tokenizer.encode(sample_text)

token_ids = encoding.ids

print("Original characters:", len(sample_text))
print("Token count:", len(token_ids))
print("First 30 token IDs:", token_ids[:30])

print("\nOriginal text:")
print(sample_text[:300])

print("\nDecoded text:")
print(tokenizer.decode(
    token_ids,
    skip_special_tokens=False
)[:300])

assert len(token_ids) > 0
assert all(
    0 <= token_id < GRAC_V04_CONFIG["vocab_size"]
    for token_id in token_ids
)

print("\nTokenization test passed!")


GRAC-LM v0.4 — TOKENIZATION TEST
Original characters: 703
Token count: 193
First 30 token IDs: [54, 224, 77, 4028, 875, 455, 1475, 92, 3371, 438, 487, 608, 6138, 431, 217, 7029, 3155, 322, 1986, 487, 172, 162, 181, 103, 161, 205, 116, 159, 174, 110]

Original text:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janua

Decoded text:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janua

Tokenization test passed!


In [ ]:

import numpy as np
from pathlib import Path
import time

print("=" * 60)
print("GRAC-LM v0.4 — DATASET TOKENIZATION")
print("=" * 60)

# Use the tokenizer loaded in Cell 4
tokenizer.no_padding()
tokenizer.no_truncation()

# Include non-empty rows, including section headings
def tokenize_split(texts, split_name):
    print(f"\nTokenizing {split_name}...")
    start = time.time()

    all_ids = []
    chunk_size = 1000

    eos_id = tokenizer.token_to_id("<eos>")

    for start_idx in range(0, len(texts), chunk_size):
        batch = texts[start_idx:start_idx + chunk_size]

        encodings = tokenizer.encode_batch(batch)

        for encoding in encodings:
            all_ids.extend(encoding.ids)

            # Separate documents when EOS exists
            if eos_id is not None:
                all_ids.append(eos_id)

        if start_idx % 20000 == 0:
            print(f"Processed {start_idx:,}/{len(texts):,} rows")

    result = np.asarray(all_ids, dtype=np.uint16)

    print(f"{split_name} tokens: {len(result):,}")
    print(f"Elapsed: {time.time() - start:.1f} seconds")

    return result

train_tokens = tokenize_split(train_texts, "Training")
val_tokens = tokenize_split(validation_texts, "Validation")

assert len(train_tokens) > 513
assert len(val_tokens) > 513
assert int(train_tokens.max()) < GRAC_V04_CONFIG["vocab_size"]
assert int(val_tokens.max()) < GRAC_V04_CONFIG["vocab_size"]

# Save to Colab's local disk for fast training access
LOCAL_DATA = Path("/content/grac_v04_data")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

np.save(LOCAL_DATA / "train_tokens.npy", train_tokens)
np.save(LOCAL_DATA / "val_tokens.npy", val_tokens)

print("\nTraining token file:",
      LOCAL_DATA / "train_tokens.npy")

print("Validation token file:",
      LOCAL_DATA / "val_tokens.npy")

print("\nGRAC-LM v0.4 tokenization completed!")


GRAC-LM v0.4 — DATASET TOKENIZATION

Tokenizing Training...
Processed 0/1,165,029 rows
Processed 20,000/1,165,029 rows
Processed 40,000/1,165,029 rows
Processed 60,000/1,165,029 rows
Processed 80,000/1,165,029 rows
Processed 100,000/1,165,029 rows
Processed 120,000/1,165,029 rows
Processed 140,000/1,165,029 rows
Processed 160,000/1,165,029 rows
Processed 180,000/1,165,029 rows
Processed 200,000/1,165,029 rows
Processed 220,000/1,165,029 rows
Processed 240,000/1,165,029 rows
Processed 260,000/1,165,029 rows
Processed 280,000/1,165,029 rows
Processed 300,000/1,165,029 rows
Processed 320,000/1,165,029 rows
Processed 340,000/1,165,029 rows
Processed 360,000/1,165,029 rows
Processed 380,000/1,165,029 rows
Processed 400,000/1,165,029 rows
Processed 420,000/1,165,029 rows
Processed 440,000/1,165,029 rows
Processed 460,000/1,165,029 rows
Processed 480,000/1,165,029 rows
Processed 500,000/1,165,029 rows
Processed 520,000/1,165,029 rows
Processed 540,000/1,165,029 rows
Processed 560,000/1,165,02

In [ ]:

import torch
import numpy as np

CONTEXT = GRAC_V04_CONFIG["context_length"]
BATCH_SIZE = GRAC_V04_CONFIG["micro_batch_size"]

def get_batch(split="train"):
    if split == "train":
        data = train_tokens
    elif split == "val":
        data = val_tokens
    else:
        raise ValueError("Split must be 'train' or 'val'")

    starts = torch.randint(
        low=0,
        high=len(data) - CONTEXT,
        size=(BATCH_SIZE,)
    ).tolist()

    x = torch.stack([
        torch.from_numpy(
            data[i:i + CONTEXT].astype(np.int64)
        )
        for i in starts
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1:i + CONTEXT + 1].astype(np.int64)
        )
        for i in starts
    ])

    return x.to(device), y.to(device)

# Verify the function
x, y = get_batch("train")

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Device:", x.device)

assert torch.equal(x[:, 1:], y[:, :-1])

print("GRAC-LM v0.4 batch generator READY!")


Input shape: torch.Size([4, 512])
Target shape: torch.Size([4, 512])
Device: cuda:0
GRAC-LM v0.4 batch generator READY!


In [ ]:

import torch
import time

print("=" * 60)
print("GRAC-LM v0.4 — FIRST TRAINING STEP")
print("=" * 60)

# Training configuration
MICRO_BATCH_SIZE = GRAC_V04_CONFIG["micro_batch_size"]
GRAD_ACCUM_STEPS = GRAC_V04_CONFIG["gradient_accumulation_steps"]
LEARNING_RATE = GRAC_V04_CONFIG["learning_rate"]

# Initialize optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=GRAC_V04_CONFIG["weight_decay"]
)

# Mixed precision for Tesla T4
scaler = torch.amp.GradScaler("cuda")

model.train()
optimizer.zero_grad(set_to_none=True)

start_time = time.time()
losses = []

# Accumulate gradients across micro-batches
for micro_step in range(GRAD_ACCUM_STEPS):
    x, y = get_batch("train")

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits, loss = model(x, targets=y)

    assert torch.isfinite(loss).item(), "Non-finite loss detected!"

    losses.append(loss.item())

    # Normalize loss for gradient accumulation
    scaled_loss = loss / GRAD_ACCUM_STEPS

    scaler.scale(scaled_loss).backward()

# Clip gradients to prevent exploding updates
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

assert torch.isfinite(grad_norm).item(), "Invalid gradient norm!"

# Update model weights
scale_before = scaler.get_scale()
scaler.step(optimizer)
scaler.update()
scale_after = scaler.get_scale()

if scale_after < scale_before:
    print("WARNING: Optimizer step may have been skipped due to overflow.")
else:
    print("Optimizer step completed without detected overflow.")

optimizer.zero_grad(set_to_none=True)

elapsed = time.time() - start_time

print("\nTraining results:")
print(f"Average loss: {sum(losses) / len(losses):.4f}")
print(f"Gradient norm: {grad_norm.item():.4f}")
print(f"Time: {elapsed:.2f} seconds")

print("\nGPU memory:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Peak allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

print("\nGRAC-LM v0.4 FIRST TRAINING TEST COMPLETE!")


GRAC-LM v0.4 — FIRST TRAINING STEP
Optimizer step completed without detected overflow.

Training results:
Average loss: 9.1162
Gradient norm: 3.9455
Time: 1.02 seconds

GPU memory:
Allocated: 0.67 GB
Peak allocated: 1.19 GB

GRAC-LM v0.4 FIRST TRAINING TEST COMPLETE!


In [ ]:

import math
import torch

# GRAC-LM v0.4 pretraining configuration
MAX_STEPS = 10000
WARMUP_STEPS = 300

MAX_LR = 3e-4
MIN_LR = 3e-5

EVAL_INTERVAL = 250
EVAL_BATCHES = 20

SAVE_INTERVAL = 500

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS

    if step >= MAX_STEPS:
        return MIN_LR

    progress = (step - WARMUP_STEPS) / (
        MAX_STEPS - WARMUP_STEPS
    )

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

    return MIN_LR + cosine * (MAX_LR - MIN_LR)


@torch.no_grad()
def estimate_loss():
    results = {}

    was_training = model.training
    model.eval()

    for split in ["train", "val"]:
        losses = []

        for _ in range(EVAL_BATCHES):
            x, y = get_batch(split)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                _, loss = model(x, targets=y)

            losses.append(loss.item())

        results[split] = sum(losses) / len(losses)

    model.train(was_training)
    return results


print("=" * 60)
print("GRAC-LM v0.4 — TRAINING SCHEDULE")
print("=" * 60)

print("Total optimizer steps:", MAX_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Maximum learning rate:", MAX_LR)
print("Minimum learning rate:", MIN_LR)
print("Evaluation interval:", EVAL_INTERVAL)
print("Checkpoint interval:", SAVE_INTERVAL)

print("\nLearning-rate samples:")
for step in [0, 100, 300, 1000, 5000, 9999]:
    print(f"Step {step:5d}: {get_lr(step):.8f}")

print("\nRunning initial validation...")
initial_losses = estimate_loss()

print(f"Training loss: {initial_losses['train']:.4f}")
print(f"Validation loss: {initial_losses['val']:.4f}")

print("\nGRAC-LM v0.4 validation system READY!")


GRAC-LM v0.4 — TRAINING SCHEDULE
Total optimizer steps: 10000
Warmup steps: 300
Maximum learning rate: 0.0003
Minimum learning rate: 3e-05
Evaluation interval: 250
Checkpoint interval: 500

Learning-rate samples:
Step     0: 0.00000100
Step   100: 0.00010100
Step   300: 0.00030000
Step  1000: 0.00029655
Step  5000: 0.00017156
Step  9999: 0.00003000

Running initial validation...
Training loss: 4.4470
Validation loss: 4.5059

GRAC-LM v0.4 validation system READY!


In [ ]:

import os
import time
import math
import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — PRETRAINING ENGINE")
print("=" * 65)

# -----------------------------------------
# 1. Check required files and configuration
# -----------------------------------------

checkpoint_dir = Path(CHECKPOINT_DIR)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

initial_path = checkpoint_dir / "GRAC-LM-v0.4-initial.pt"
latest_path = checkpoint_dir / "GRAC-LM-v0.4-latest.pt"
best_path = checkpoint_dir / "GRAC-LM-v0.4-best.pt"

assert initial_path.exists(), (
    "Initial checkpoint missing! Run Cell 7 first."
)

assert "train_tokens" in globals()
assert "val_tokens" in globals()

# -----------------------------------------
# 2. Training settings
# -----------------------------------------

TOTAL_STEPS = MAX_STEPS
RUN_UNTIL_STEP = 10000

GRAD_ACCUM = GRAC_V04_CONFIG[
    "gradient_accumulation_steps"
]

# -----------------------------------------
# 3. Initialize model and optimizer
# -----------------------------------------

model = GRACLM(GRAC_V04_CONFIG).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=GRAC_V04_CONFIG["weight_decay"]
)

scaler = torch.amp.GradScaler("cuda")

start_step = 0
best_val_loss = float("inf")

# -----------------------------------------
# 4. Resume if checkpoint exists
# -----------------------------------------

if latest_path.exists():
    print("Resuming existing training checkpoint...")

    ckpt = torch.load(
        latest_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_step = ckpt["step"]
    best_val_loss = ckpt["best_val_loss"]

    print("Resumed at step:", start_step)

else:
    print("Starting clean training from step 0...")

    ckpt = torch.load(
        initial_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(ckpt["model_state_dict"])

# -----------------------------------------
# 5. Checkpoint-saving function
# -----------------------------------------

def save_training_checkpoint(path, step, val_loss):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "config": GRAC_V04_CONFIG,
        "step": step,
        "best_val_loss": best_val_loss,
        "validation_loss": val_loss
    }

    temporary_path = path.with_suffix(".tmp")

    torch.save(checkpoint, temporary_path)

    os.replace(temporary_path, path)

# -----------------------------------------
# 6. Main training loop
# -----------------------------------------

print("\nTraining target:", RUN_UNTIL_STEP)
print("Starting from:", start_step)

training_start = time.time()

for step in range(start_step, min(RUN_UNTIL_STEP, TOTAL_STEPS)):

    model.train()

    current_lr = get_lr(step)

    for group in optimizer.param_groups:
        group["lr"] = current_lr

    optimizer.zero_grad(set_to_none=True)

    micro_losses = []

    for micro_step in range(GRAD_ACCUM):

        x, y = get_batch("train")

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            _, loss = model(x, targets=y)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {step}"
            )

        micro_losses.append(loss.item())

        scaler.scale(
            loss / GRAD_ACCUM
        ).backward()

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    scale_before = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < scale_before:
        print(
            f"Warning: skipped optimizer update at step {step}"
        )

    completed_step = step + 1

    # -------------------------------------
    # 7. Logging
    # -------------------------------------

    if completed_step % 25 == 0:
        avg_loss = sum(micro_losses) / len(micro_losses)

        elapsed = time.time() - training_start

        print(
            f"Step {completed_step:5d}/{TOTAL_STEPS} | "
            f"Loss {avg_loss:.4f} | "
            f"LR {current_lr:.7f} | "
            f"Elapsed {elapsed / 60:.1f} min"
        )

    # -------------------------------------
    # 8. Evaluation and checkpoints
    # -------------------------------------

    should_evaluate = (
        completed_step % EVAL_INTERVAL == 0
        or completed_step == RUN_UNTIL_STEP
    )

    if should_evaluate:

        losses = estimate_loss()
        val_loss = losses["val"]

        print("\n" + "-" * 60)
        print(
            f"VALIDATION — STEP {completed_step}"
        )
        print(f"Train loss: {losses['train']:.4f}")
        print(f"Validation loss: {val_loss:.4f}")
        print("-" * 60)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            save_training_checkpoint(
                best_path,
                completed_step,
                val_loss
            )

            print("NEW BEST MODEL SAVED!")

        save_training_checkpoint(
            latest_path,
            completed_step,
            val_loss
        )

        print("Resume checkpoint saved.")

print("\n" + "=" * 65)
print("GRAC-LM v0.4 TRAINING RUN COMPLETE")
print("Completed steps:", completed_step if start_step < RUN_UNTIL_STEP else start_step)
print("Best validation loss:", best_val_loss)
print("Checkpoint folder:", checkpoint_dir)
print("=" * 65)


GRAC-LM v0.4 — PRETRAINING ENGINE
Resuming existing training checkpoint...
Resumed at step: 2000

Training target: 10000
Starting from: 2000
Step  2025/10000 | Loss 4.5449 | LR 0.0002795 | Elapsed 0.1 min
Step  2050/10000 | Loss 4.6311 | LR 0.0002789 | Elapsed 0.3 min
Step  2075/10000 | Loss 4.5542 | LR 0.0002783 | Elapsed 0.4 min
Step  2100/10000 | Loss 4.5159 | LR 0.0002777 | Elapsed 0.5 min
Step  2125/10000 | Loss 4.5092 | LR 0.0002771 | Elapsed 0.6 min
Step  2150/10000 | Loss 4.4753 | LR 0.0002765 | Elapsed 0.8 min
Step  2175/10000 | Loss 4.3005 | LR 0.0002759 | Elapsed 0.9 min
Step  2200/10000 | Loss 4.4172 | LR 0.0002753 | Elapsed 1.0 min
Step  2225/10000 | Loss 4.3198 | LR 0.0002746 | Elapsed 1.2 min
Step  2250/10000 | Loss 4.4585 | LR 0.0002740 | Elapsed 1.3 min

------------------------------------------------------------
VALIDATION — STEP 2250
Train loss: 4.4892
Validation loss: 4.3983
------------------------------------------------------------
NEW BEST MODEL SAVED!
Resume c

In [ ]:

import torch
from pathlib import Path

print("=" * 60)
print("GRAC-LM v0.4 — FIRST GENERATION EXAM")
print("=" * 60)

best_path = Path(CHECKPOINT_DIR) / "GRAC-LM-v0.4-best.pt"

assert best_path.exists(), "Best checkpoint not found!"

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False
)

# Load the trained model, not the untrained initial model
generation_model = GRACLM(GRAC_V04_CONFIG).to(device)
generation_model.load_state_dict(checkpoint["model_state_dict"])
generation_model.eval()

print("Checkpoint step:", checkpoint["step"])
print("Best validation loss:", checkpoint["validation_loss"])

@torch.no_grad()
def generate_grac(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    encoded = tokenizer.encode(prompt)
    ids = encoded.ids

    if not ids:
        raise ValueError("Prompt produced no tokens")

    tokens = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    eos_id = tokenizer.token_to_id("<eos>")

    for _ in range(max_new_tokens):
        context = tokens[:, -GRAC_V04_CONFIG["context_length"]:]

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, _ = generation_model(context)

        next_logits = logits[:, -1, :].float()
        next_logits = next_logits / temperature

        if top_k is not None:
            values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            threshold = values[:, -1].unsqueeze(-1)
            next_logits[next_logits < threshold] = -float("inf")

        probabilities = torch.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(probabilities, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

        if eos_id is not None and next_token.item() == eos_id:
            break

    return tokenizer.decode(
        tokens[0].tolist(),
        skip_special_tokens=True
    )

test_prompts = [
    "The history of artificial intelligence",
    "Scientists discovered that",
    "The government announced",
    "In the future, computers will",
    "The city of London"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'=' * 60}")
    print(f"TEST {i}: {prompt}")
    print("-" * 60)

    result = generate_grac(prompt)
    print(result)

print("\nGRAC-LM v0.4 generation exam completed!")


GRAC-LM v0.4 — FIRST GENERATION EXAM
Checkpoint step: 8750
Best validation loss: 3.3355633139610292

TEST 1: The history of artificial intelligence
------------------------------------------------------------
The history of artificial intelligence , and for the eventuality of the " Friends of the Lion " and the use of the " Fellow " and " Friends of the Lion " . He also commented :

TEST 2: Scientists discovered that
------------------------------------------------------------
Scientists discovered that the binary structure was a part of the structure of the structure ( e.g. that was also known as the " binary " ) in the outer binary structure . At the same time , several elements of the binary system were added to the structure . This area also called the " binary " , which is the " binary " ( or " binary " , which is a binary ) , and the binary system is used by this area . The binary system

TEST 3: The government announced
-----------------------------------------------------------

In [ ]:

import torch
from pathlib import Path

latest_path = Path(CHECKPOINT_DIR) / "GRAC-LM-v0.4-latest.pt"

checkpoint = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=False
)

print("Latest completed step:", checkpoint["step"])
print("Latest validation loss:", checkpoint["validation_loss"])
print("Best validation loss:", checkpoint["best_val_loss"])


Latest completed step: 10000
Latest validation loss: 3.363109314441681
Best validation loss: 3.3355633139610292


In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — GENERATION DIAGNOSTIC EXAM")
print("=" * 65)

print("Model checkpoint step:", checkpoint["step"])

test_prompts = [
    "The city of London",
    "Scientists discovered that",
    "The government announced",
    "Artificial intelligence is"
]

settings = [
    {"temperature": 0.6, "top_k": 20},
    {"temperature": 0.8, "top_k": 40},
    {"temperature": 1.0, "top_k": 80}
]

for prompt in test_prompts:
    print("\n" + "=" * 65)
    print("PROMPT:", prompt)
    print("=" * 65)

    for setting in settings:
        print(
            f"\nTemperature: {setting['temperature']} | "
            f"Top-k: {setting['top_k']}"
        )

        result = generate_grac(
            prompt,
            max_new_tokens=80,
            temperature=setting["temperature"],
            top_k=setting["top_k"]
        )

        print(result)
        print("-" * 50)

print("\nDIAGNOSTIC EXAM COMPLETE")


GRAC-LM v0.4 — GENERATION DIAGNOSTIC EXAM
Model checkpoint step: 10000

PROMPT: The city of London

Temperature: 0.6 | Top-k: 20
The city of London was largely destroyed , and the town is now part of the town of Chester . The town is listed by the Metropolitan Borough Council . The town is a major residential centre , which is a Grade II listed building . The town is located on the southern side of the town centre , which is located in the town centre . The town is an area of the town , and the town is
--------------------------------------------------

Temperature: 0.8 | Top-k: 40
The city of London , and the main entrance to a new building was built next to the old building . It was built in 1938 as part of the construction of the city , the Great North , which was built to accommodate its first floor , and a new building was built at the end of the 20th century . In addition , the city of London , built in the early 14th century , had a capacity of
----------------------------------

In [ ]:

import torch

@torch.no_grad()
def generate_grac_v2(
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=40,
    repetition_penalty=1.15
):
    ids = tokenizer.encode(prompt).ids

    if not ids:
        raise ValueError("Prompt produced no tokens")

    tokens = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    eos_id = tokenizer.token_to_id("<eos>")

    generation_model.eval()

    for _ in range(max_new_tokens):
        context = tokens[:, -GRAC_V04_CONFIG["context_length"]:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits, _ = generation_model(context)

        next_logits = logits[:, -1, :].float()

        # Penalize tokens already present in recent context
        recent_tokens = torch.unique(tokens[0, -64:])

        scores = next_logits[:, recent_tokens]

        next_logits[:, recent_tokens] = torch.where(
            scores < 0,
            scores * repetition_penalty,
            scores / repetition_penalty
        )

        next_logits /= temperature

        if top_k is not None:
            values, _ = torch.topk(
                next_logits,
                min(top_k, next_logits.size(-1))
            )
            threshold = values[:, -1].unsqueeze(-1)
            next_logits[next_logits < threshold] = -float("inf")

        probabilities = torch.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        tokens = torch.cat([tokens, next_token], dim=1)

        if eos_id is not None and next_token.item() == eos_id:
            break

    return tokenizer.decode(
        tokens[0].tolist(),
        skip_special_tokens=True
    )


print("=" * 60)
print("GRAC-LM v0.4 — REPETITION CONTROL TEST")
print("=" * 60)

prompts = [
    "The city of London",
    "Scientists discovered that",
    "The government announced",
    "Artificial intelligence is"
]

for prompt in prompts:
    print("\nPROMPT:", prompt)
    print("-" * 60)

    result = generate_grac_v2(
        prompt,
        temperature=0.8,
        top_k=40,
        repetition_penalty=1.15
    )

    print(result)

print("\nTEST COMPLETE")


GRAC-LM v0.4 — REPETITION CONTROL TEST

PROMPT: The city of London
------------------------------------------------------------
The city of London in 1275 , and for the same time it was demolished and opened . The walls are divided into two main towers ( Gate Hill , and Walsall Road ) – which are similar to those in the castle ; this is a lodge of several chantry rooms :

PROMPT: Scientists discovered that
------------------------------------------------------------
Scientists discovered that the group 's theory of evolutionary structure has been compounded by the influence of the phenomenon in many other genes .

PROMPT: The government announced
------------------------------------------------------------
The government announced that the Human Rights Act should be passed by Congress to act in an election to an end of the year . After a three @-@ year session , Congress was split into two elections .

PROMPT: Artificial intelligence is
-------------------------------------------------

In [ ]:

import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — CHECKPOINT COMPARISON")
print("=" * 65)

checkpoint_dir = Path(CHECKPOINT_DIR)

checkpoints = {
    "BEST": checkpoint_dir / "GRAC-LM-v0.4-best.pt",
    "FINAL": checkpoint_dir / "GRAC-LM-v0.4-latest.pt"
}

prompts = [
    "The city of London",
    "Scientists discovered that",
    "The government announced",
    "Artificial intelligence is"
]

for name, path in checkpoints.items():

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    generation_model.load_state_dict(
        ckpt["model_state_dict"]
    )
    generation_model.eval()

    print("\n" + "=" * 65)
    print(name, "CHECKPOINT")
    print("Step:", ckpt["step"])
    print("Validation loss:", ckpt["validation_loss"])
    print("=" * 65)

    # Same random seed for reproducible sampling
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    for prompt in prompts:

        result = generate_grac_v2(
            prompt,
            max_new_tokens=100,
            temperature=0.8,
            top_k=40,
            repetition_penalty=1.15
        )

        print("\nPROMPT:", prompt)
        print("OUTPUT:", result)

print("\nCHECKPOINT COMPARISON COMPLETE")


GRAC-LM v0.4 — CHECKPOINT COMPARISON

BEST CHECKPOINT
Step: 8750
Validation loss: 3.3355633139610292

PROMPT: The city of London
OUTPUT: The city of London is a popular tourist destination , and has since reached an annual average speed of 4 @.@ 3 miles per hour ( 160 km / h ) . There was also a pinnacle of 109 people living in London , which is the oldest for any major airport in the world .

PROMPT: Scientists discovered that
OUTPUT: Scientists discovered that one of the most famous men in Europe was the Great Lionel 's Manufacty ( 1870 ) , who had made over the course of several years . It was also very important for him to maintain a new generation while he studied at a friendship with the New Yorkshire Language Company . The two children were taught by his sons that " their mothers are born in good condition " and some of them were " not the stern of man or who is now

PROMPT: The government announced
OUTPUT: The government announced that the construction of these buildings were c

In [ ]:

from pathlib import Path
import shutil
import json
import torch

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
checkpoint_dir = base_dir / "checkpoints"
archive_dir = base_dir / "archive"

archive_dir.mkdir(parents=True, exist_ok=True)

files = {
    "GRAC-LM-v0.4-best.pt": "GRAC-LM-v0.4-pretraining-best-8750.pt",
    "GRAC-LM-v0.4-latest.pt": "GRAC-LM-v0.4-pretraining-final-10000.pt"
}

for source_name, archive_name in files.items():
    source = checkpoint_dir / source_name
    destination = archive_dir / archive_name

    assert source.exists(), f"Missing checkpoint: {source}"

    if not destination.exists():
        shutil.copy2(source, destination)
        print("Archived:", archive_name)
    else:
        print("Already archived:", archive_name)

print("\nGRAC-LM v0.4 checkpoints preserved!")


Archived: GRAC-LM-v0.4-pretraining-best-8750.pt
Archived: GRAC-LM-v0.4-pretraining-final-10000.pt

GRAC-LM v0.4 checkpoints preserved!


In [ ]:

import numpy as np
import json
from pathlib import Path

benchmark_dir = base_dir / "benchmarks"
benchmark_dir.mkdir(parents=True, exist_ok=True)

benchmark_path = benchmark_dir / "fixed_validation_indices.npy"

CONTEXT_LENGTH = GRAC_V04_CONFIG["context_length"]
BENCHMARK_SAMPLES = 256

assert "val_tokens" in globals(), (
    "Validation tokens missing. Restore the validation dataset first."
)

assert len(val_tokens) > CONTEXT_LENGTH + 1

if benchmark_path.exists():
    fixed_indices = np.load(benchmark_path)
    print("Loaded existing fixed benchmark.")
else:
    rng = np.random.default_rng(2026)

    max_start = len(val_tokens) - CONTEXT_LENGTH - 1

    fixed_indices = rng.integers(
        0,
        max_start + 1,
        size=BENCHMARK_SAMPLES,
        dtype=np.int64
    )

    np.save(benchmark_path, fixed_indices)
    print("Created fixed benchmark.")

print("Benchmark samples:", len(fixed_indices))
print("Saved at:", benchmark_path)


Loaded existing fixed benchmark.
Benchmark samples: 256
Saved at: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/benchmarks/fixed_validation_indices.npy


In [ ]:

import torch
import numpy as np

@torch.no_grad()
def evaluate_fixed_checkpoint(checkpoint_path):

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    eval_model = GRACLM(GRAC_V04_CONFIG).to(device)
    eval_model.load_state_dict(ckpt["model_state_dict"])
    eval_model.eval()

    losses = []
    batch_size = 4

    for start in range(0, len(fixed_indices), batch_size):
        batch_indices = fixed_indices[start:start + batch_size]

        x = torch.stack([
            torch.tensor(
                val_tokens[i:i + CONTEXT_LENGTH],
                dtype=torch.long
            )
            for i in batch_indices
        ]).to(device)

        y = torch.stack([
            torch.tensor(
                val_tokens[i + 1:i + CONTEXT_LENGTH + 1],
                dtype=torch.long
            )
            for i in batch_indices
        ]).to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            _, loss = eval_model(x, targets=y)

        losses.append(
            (loss.item(), len(batch_indices))
        )

    total = sum(loss * count for loss, count in losses)
    count = sum(count for _, count in losses)

    del eval_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total / count


best_path = archive_dir / "GRAC-LM-v0.4-pretraining-best-8750.pt"
final_path = archive_dir / "GRAC-LM-v0.4-pretraining-final-10000.pt"

best_loss = evaluate_fixed_checkpoint(best_path)
final_loss = evaluate_fixed_checkpoint(final_path)

print("=" * 60)
print("GRAC-LM v0.4 — FIXED BENCHMARK RESULTS")
print("=" * 60)

print(f"Best checkpoint (8750):  {best_loss:.4f}")
print(f"Final checkpoint (10000): {final_loss:.4f}")

winner = "8750" if best_loss < final_loss else "10000"
print("Lower-loss checkpoint:", winner)


GRAC-LM v0.4 — FIXED BENCHMARK RESULTS
Best checkpoint (8750):  3.4165
Final checkpoint (10000): 3.3890
Lower-loss checkpoint: 10000


In [ ]:

from datasets import load_dataset
from collections import Counter

print("=" * 65)
print("GRAC-LM v0.4 — DATASET INSPECTION")
print("=" * 65)

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1"
)

train_texts = [
    row["text"]
    for row in dataset["train"]
    if row["text"].strip()
]

print("Nonempty training sections:", len(train_texts))

artifacts = [
    " @-@ ",
    " @.@ ",
    " @,@ ",
    " = = ",
    " = = = "
]

for artifact in artifacts:
    count = sum(text.count(artifact) for text in train_texts)
    print(f"{artifact!r}: {count:,}")

print("\nEXAMPLE SECTIONS:")
for text in train_texts[:5]:
    print(repr(text[:250]))
    print("-" * 50)


GRAC-LM v0.4 — DATASET INSPECTION
Nonempty training sections: 1165029
' @-@ ': 881,576
' @.@ ': 158,057
' @,@ ': 149,068
' = = ': 552,846
' = = = ': 270,523

EXAMPLE SECTIONS:
' = Valkyria Chronicles III = \n'
--------------------------------------------------
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Visi'
--------------------------------------------------
' The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving f'
--------------------------------------------------
' It met with positive sales in Japan , and was praised by both Japanese and western critics . After release , it received download

In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — TOKENIZER COMPATIBILITY TEST")
print("=" * 65)

samples = [
    "The city of London was founded centuries ago.",
    "Artificial intelligence is a field of computer science.",
    "The temperature was 4.3 degrees.",
    "The government announced a three-year programme."
]

unk_id = tokenizer.token_to_id("<unk>")

for sample in samples:
    encoded = tokenizer.encode(sample)
    ids = encoded.ids

    unknown_count = (
        ids.count(unk_id) if unk_id is not None else 0
    )

    print("\nTEXT:", sample)
    print("TOKEN IDS:", ids)
    print("TOKEN COUNT:", len(ids))
    print("UNKNOWN TOKENS:", unknown_count)
    print("DECODED:", tokenizer.decode(ids))

print("\nTOKENIZER TEST COMPLETE")


GRAC-LM v0.4 — TOKENIZER COMPATIBILITY TEST

TEXT: The city of London was founded centuries ago.
TOKEN IDS: [506, 1223, 232, 1690, 270, 4122, 5278, 6294, 17]
TOKEN COUNT: 9
UNKNOWN TOKENS: 0
DECODED: The city of London was founded centuries ago.

TEXT: Artificial intelligence is a field of computer science.
TOKEN IDS: [5219, 87, 828, 423, 6696, 324, 209, 1954, 232, 5424, 4017, 17]
TOKEN COUNT: 12
UNKNOWN TOKENS: 0
DECODED: Artificial intelligence is a field of computer science.

TEXT: The temperature was 4.3 degrees.
TOKEN IDS: [506, 5666, 270, 504, 17, 22, 3084, 7567, 17]
TOKEN COUNT: 9
UNKNOWN TOKENS: 0
DECODED: The temperature was 4.3 degrees.

TEXT: The government announced a three-year programme.
TOKEN IDS: [506, 1335, 1967, 209, 725, 16, 92, 391, 7009, 17]
TOKEN COUNT: 10
UNKNOWN TOKENS: 0
DECODED: The government announced a three-year programme.

TOKENIZER TEST COMPLETE


In [ ]:

import numpy as np
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
clean_file = base_dir / "data/cleaned/wikitext103-clean-train.txt"
token_file = base_dir / "data/cleaned/train_tokens_v2.npy"

assert clean_file.exists(), "Run Cell 23 first."

print("=" * 65)
print("GRAC-LM v0.4 — CLEANED DATA TOKENIZATION")
print("=" * 65)

if token_file.exists():
    print("Existing token file found; preserving it.")
else:
    all_ids = []
    batch = []
    batch_size = 256

    def process_batch(lines):
        encodings = tokenizer.encode_batch(lines)
        return [
            token_id
            for encoding in encodings
            for token_id in encoding.ids
        ]

    with open(clean_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            batch.append(line)

            if len(batch) >= batch_size:
                all_ids.extend(process_batch(batch))
                batch.clear()

        if batch:
            all_ids.extend(process_batch(batch))

    tokens = np.asarray(all_ids, dtype=np.uint16)

    assert len(tokens) > GRAC_V04_CONFIG["context_length"] + 1
    assert int(tokens.max()) < GRAC_V04_CONFIG["vocab_size"]

    np.save(token_file, tokens)

    print("Saved cleaned training tokens.")

clean_tokens = np.load(token_file, mmap_mode="r")

print("Total tokens:", f"{len(clean_tokens):,}")
print("Token dtype:", clean_tokens.dtype)
print("Saved to:", token_file)


AssertionError: Run Cell 23 first.

In [ ]:

import re
import json
from pathlib import Path

clean_dir = Path(
    "/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/cleaned"
)
clean_dir.mkdir(parents=True, exist_ok=True)

def clean_wikitext(text):
    # Restore common WikiText punctuation artifacts
    text = text.replace(" @-@ ", "-")
    text = text.replace(" @.@ ", ".")
    text = text.replace(" @,@ ", ",")

    # Normalize excessive whitespace
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

clean_train = [
    clean_wikitext(text)
    for text in train_texts
]

clean_train = [
    text for text in clean_train if text
]

output_path = clean_dir / "wikitext103-clean-train.txt"

if output_path.exists():
    print("Cleaned dataset already exists; not overwriting.")
else:
    with open(output_path, "w", encoding="utf-8") as f:
        for text in clean_train:
            f.write(text + "\n")

print("Cleaned sections:", len(clean_train))
print("Saved to:", output_path)

print("\nBEFORE:")
print(train_texts[0][:300])

print("\nAFTER:")
print(clean_train[0][:300])


Cleaned sections: 1165029
Saved to: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/cleaned/wikitext103-clean-train.txt

BEFORE:
 = Valkyria Chronicles III = 


AFTER:
= Valkyria Chronicles III =


In [ ]:

import numpy as np
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
clean_file = base_dir / "data/cleaned/wikitext103-clean-train.txt"
token_file = base_dir / "data/cleaned/train_tokens_v2.npy"

assert clean_file.exists(), "Run Cell 23 first."

print("=" * 65)
print("GRAC-LM v0.4 — CLEANED DATA TOKENIZATION")
print("=" * 65)

if token_file.exists():
    print("Existing token file found; preserving it.")
else:
    all_ids = []
    batch = []
    batch_size = 256

    def process_batch(lines):
        encodings = tokenizer.encode_batch(lines)
        return [
            token_id
            for encoding in encodings
            for token_id in encoding.ids
        ]

    with open(clean_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            batch.append(line)

            if len(batch) >= batch_size:
                all_ids.extend(process_batch(batch))
                batch.clear()

        if batch:
            all_ids.extend(process_batch(batch))

    tokens = np.asarray(all_ids, dtype=np.uint16)

    assert len(tokens) > GRAC_V04_CONFIG["context_length"] + 1
    assert int(tokens.max()) < GRAC_V04_CONFIG["vocab_size"]

    np.save(token_file, tokens)

    print("Saved cleaned training tokens.")

clean_tokens = np.load(token_file, mmap_mode="r")

print("Total tokens:", f"{len(clean_tokens):,}")
print("Token dtype:", clean_tokens.dtype)
print("Saved to:", token_file)


GRAC-LM v0.4 — CLEANED DATA TOKENIZATION
Saved cleaned training tokens.
Total tokens: 135,251,975
Token dtype: uint16
Saved to: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/cleaned/train_tokens_v2.npy


In [ ]:

import numpy as np
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
token_file = base_dir / "data/cleaned/train_tokens_v2.npy"

assert token_file.exists(), "Cell 25 must finish first."

clean_tokens = np.load(token_file, mmap_mode="r")

vocab_size = GRAC_V04_CONFIG["vocab_size"]
context_length = GRAC_V04_CONFIG["context_length"]

assert len(clean_tokens) > context_length + 1
assert int(clean_tokens.min()) >= 0
assert int(clean_tokens.max()) < vocab_size

print("=" * 60)
print("GRAC-LM v0.4 — CLEANED DATA VERIFICATION")
print("=" * 60)

print("Total tokens:", f"{len(clean_tokens):,}")
print("Minimum token ID:", int(clean_tokens.min()))
print("Maximum token ID:", int(clean_tokens.max()))
print("Token dtype:", clean_tokens.dtype)

print("\nDECODED SAMPLE:")
print(tokenizer.decode(clean_tokens[:150].astype(int).tolist()))

print("\nVERIFICATION PASSED")


GRAC-LM v0.4 — CLEANED DATA VERIFICATION
Total tokens: 135,251,975
Minimum token ID: 4
Maximum token ID: 7999
Token dtype: uint16

DECODED SAMPLE:
= Valkyria Chronicles III =Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story

VERIFICATION PASSED


In [ ]:

from pathlib import Path
import json

phase2_dir = base_dir / "phase2"
phase2_dir.mkdir(parents=True, exist_ok=True)

PHASE2_CONFIG = {
    "starting_checkpoint": "best-8750",
    "training_steps": 500,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5,
    "weight_decay": 0.1,
    "eval_every": 100,
    "context_length": 512
}

config_path = phase2_dir / "phase2_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(PHASE2_CONFIG, f, indent=2)

print("GRAC-LM v0.4 — PHASE 2 CONFIGURATION")
print(json.dumps(PHASE2_CONFIG, indent=2))


GRAC-LM v0.4 — PHASE 2 CONFIGURATION
{
  "starting_checkpoint": "best-8750",
  "training_steps": 500,
  "micro_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 1e-05,
  "weight_decay": 0.1,
  "eval_every": 100,
  "context_length": 512
}


In [ ]:

import torch

best_checkpoint = (
    base_dir
    / "archive"
    / "GRAC-LM-v0.4-pretraining-best-8750.pt"
)

assert best_checkpoint.exists(), "Archived best checkpoint missing."

checkpoint = torch.load(
    best_checkpoint,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["step"] == 8750

phase2_model = GRACLM(GRAC_V04_CONFIG).to(device)

phase2_model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer2 = torch.optim.AdamW(
    phase2_model.parameters(),
    lr=PHASE2_CONFIG["learning_rate"],
    weight_decay=PHASE2_CONFIG["weight_decay"]
)

print("GRAC-LM v0.4 — PHASE 2 MODEL LOADED")
print("Starting checkpoint:", checkpoint["step"])
print("Parameters:", f"{sum(p.numel() for p in phase2_model.parameters()):,}")
print("Device:", device)


GRAC-LM v0.4 — PHASE 2 MODEL LOADED
Starting checkpoint: 8750
Parameters: 54,684,160
Device: cuda


In [ ]:

import numpy as np
import torch

rng_phase2 = np.random.default_rng(2026)

def get_clean_batch():
    batch_size = PHASE2_CONFIG["micro_batch_size"]
    context = PHASE2_CONFIG["context_length"]

    max_start = len(clean_tokens) - context - 1

    starts = rng_phase2.integers(
        0,
        max_start + 1,
        size=batch_size
    )

    x = np.stack([
        clean_tokens[i:i + context]
        for i in starts
    ])

    y = np.stack([
        clean_tokens[i + 1:i + context + 1]
        for i in starts
    ])

    x = torch.tensor(x.astype(np.int64), device=device)
    y = torch.tensor(y.astype(np.int64), device=device)

    return x, y

x, y = get_clean_batch()

print("Input batch:", x.shape)
print("Target batch:", y.shape)
print("Batch preparation successful!")


Input batch: torch.Size([4, 512])
Target batch: torch.Size([4, 512])
Batch preparation successful!


In [ ]:

import torch
import numpy as np
from pathlib import Path

assert "fixed_indices" in globals()
assert "val_tokens" in globals()

phase2_model.train()

use_amp = torch.cuda.is_available()
scaler2 = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

steps = PHASE2_CONFIG["training_steps"]
accum = PHASE2_CONFIG["gradient_accumulation_steps"]

def phase2_validation_loss():
    phase2_model.eval()
    weighted_loss = 0.0
    sample_count = 0

    with torch.no_grad():
        for start in range(0, len(fixed_indices), 4):
            indices = fixed_indices[start:start + 4]

            x = torch.stack([
                torch.tensor(
                    val_tokens[int(i):int(i) + 512],
                    dtype=torch.long
                )
                for i in indices
            ]).to(device)

            y = torch.stack([
                torch.tensor(
                    val_tokens[int(i) + 1:int(i) + 513],
                    dtype=torch.long
                )
                for i in indices
            ]).to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                _, loss = phase2_model(x, targets=y)

            weighted_loss += loss.item() * len(indices)
            sample_count += len(indices)

    phase2_model.train()
    return weighted_loss / sample_count

best_phase2_loss = float("inf")

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 2 TRAINING")
print("=" * 65)

for step in range(1, steps + 1):
    optimizer2.zero_grad(set_to_none=True)

    for micro in range(accum):
        x, y = get_clean_batch()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            _, loss = phase2_model(x, targets=y)
            loss = loss / accum

        scaler2.scale(loss).backward()

    scaler2.unscale_(optimizer2)

    torch.nn.utils.clip_grad_norm_(
        phase2_model.parameters(),
        max_norm=1.0
    )

    scaler2.step(optimizer2)
    scaler2.update()

    if step % PHASE2_CONFIG["eval_every"] == 0:
        val_loss = phase2_validation_loss()

        print(
            f"Step {step}/{steps} | "
            f"Validation loss: {val_loss:.4f}",
            flush=True
        )

        if val_loss < best_phase2_loss:
            best_phase2_loss = val_loss

            torch.save(
                {
                    "model_state_dict": phase2_model.state_dict(),
                    "step": step,
                    "validation_loss": val_loss,
                    "config": GRAC_V04_CONFIG
                },
                phase2_dir / "GRAC-LM-v0.4-phase2-best.pt"
            )

torch.save(
    {
        "model_state_dict": phase2_model.state_dict(),
        "step": steps,
        "validation_loss": val_loss,
        "config": GRAC_V04_CONFIG
    },
    phase2_dir / "GRAC-LM-v0.4-phase2-final.pt"
)

print("\nPHASE 2 TRAINING COMPLETE!")
print("Best validation loss:", best_phase2_loss)


GRAC-LM v0.4 — PHASE 2 TRAINING
Step 100/500 | Validation loss: 3.4447
Step 200/500 | Validation loss: 3.4518
Step 300/500 | Validation loss: 3.4548
Step 400/500 | Validation loss: 3.4571
Step 500/500 | Validation loss: 3.4586

PHASE 2 TRAINING COMPLETE!
Best validation loss: 3.444709897041321


In [ ]:

import torch

phase2_best = phase2_dir / "GRAC-LM-v0.4-phase2-best.pt"

original_loss = evaluate_fixed_checkpoint(best_checkpoint)
new_loss = evaluate_fixed_checkpoint(phase2_best)

print("=" * 60)
print("GRAC-LM v0.4 — PHASE 2 COMPARISON")
print("=" * 60)

print(f"Original model: {original_loss:.4f}")
print(f"Phase 2 model:  {new_loss:.4f}")

if new_loss < original_loss:
    print("\nPhase 2 improved fixed validation loss!")
else:
    print("\nNo fixed-validation improvement yet.")

print("Original checkpoint remains preserved.")


GRAC-LM v0.4 — PHASE 2 COMPARISON
Original model: 3.4165
Phase 2 model:  3.4447

No fixed-validation improvement yet.
Original checkpoint remains preserved.


In [ ]:

import torch

models_to_test = {
    "ORIGINAL": best_checkpoint,
    "PHASE 2": phase2_best
}

test_prompts = [
    "The city of London",
    "Scientists discovered that",
    "The government announced",
    "Artificial intelligence is"
]

for name, path in models_to_test.items():

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    generation_model.load_state_dict(
        ckpt["model_state_dict"]
    )
    generation_model.eval()

    print("\n" + "=" * 65)
    print("GRAC-LM v0.4 —", name)
    print("=" * 65)

    torch.manual_seed(42)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    for prompt in test_prompts:
        print("\nPROMPT:", prompt)

        print(
            generate_grac_v2(
                prompt,
                max_new_tokens=100,
                temperature=0.8,
                top_k=40,
                repetition_penalty=1.15
            )
        )

print("\nGENERATION COMPARISON COMPLETE")



GRAC-LM v0.4 — ORIGINAL

PROMPT: The city of London
The city of London is a popular tourist destination , and has since reached an annual average speed of 4 @.@ 3 miles per hour ( 160 km / h ) . There was also a pinnacle of 109 people living in London , which is the oldest for any major airport in the world .

PROMPT: Scientists discovered that
Scientists discovered that one of the most famous men in Europe was the Great Lionel 's Manufacty ( 1870 ) , who had made over the course of several years . It was also very important for him to maintain a new generation while he studied at a friendship with the New Yorkshire Language Company . The two children were taught by his sons that " their mothers are born in good condition " and some of them were " not the stern of man or who is now

PROMPT: The government announced
The government announced that the construction of these buildings were completed by February 2009 .

PROMPT: Artificial intelligence is
Artificial intelligence is a part of

In [ ]:

from pathlib import Path
from collections import Counter
import re

clean_file = (
    base_dir / "data/cleaned/wikitext103-clean-train.txt"
)

patterns = {
    "Hyphen artifacts": r"@\s*-\s*@",
    "Decimal artifacts": r"@\s*\.\s*@",
    "Comma artifacts": r"@\s*,\s*@",
    "Wiki heading markers": r"=\s*=\s*[^=\n]+\s*=\s*=",
    "Repeated whitespace": r" {3,}"
}

print("=" * 65)
print("GRAC-LM v0.4 — CLEANED DATA QUALITY AUDIT")
print("=" * 65)

counts = Counter()
line_count = 0
examples = {}

with open(clean_file, encoding="utf-8") as f:
    for line in f:
        line_count += 1

        for name, pattern in patterns.items():
            matches = list(re.finditer(pattern, line))

            counts[name] += len(matches)

            if matches and name not in examples:
                examples[name] = line[:200].strip()

print("Total lines:", f"{line_count:,}")

for name in patterns:
    print(f"\n{name}: {counts[name]:,}")

    if name in examples:
        print("Example:", examples[name])

print("\nDATA QUALITY AUDIT COMPLETE")


GRAC-LM v0.4 — CLEANED DATA QUALITY AUDIT
Total lines: 1,165,029

Hyphen artifacts: 0

Decimal artifacts: 0

Comma artifacts: 0

Wiki heading markers: 291,028
Example: = = Gameplay = =

Repeated whitespace: 0

DATA QUALITY AUDIT COMPLETE


In [ ]:

import re
import json
from pathlib import Path
from datasets import load_dataset

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
v2_dir = base_dir / "data" / "v2"
v2_dir.mkdir(parents=True, exist_ok=True)

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1"
)

def clean_line(text):
    text = text.strip()

    # Restore WikiText punctuation
    text = re.sub(r"\s*@-@\s*", "-", text)
    text = re.sub(r"\s*@\.@\s*", ".", text)
    text = re.sub(r"\s*@,@\s*", ",", text)

    # Remove heading-only lines
    if re.fullmatch(r"\s*=+\s*[^=]+\s*=+\s*", text):
        return ""

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

def extract_articles(split):
    articles = []
    current = []

    for row in dataset[split]:
        raw = row["text"].strip()

        # Major article heading marks a new document
        if re.fullmatch(r"=\s*[^=]+\s*=", raw):
            if current:
                articles.append("\n".join(current))
                current = []
            continue

        cleaned = clean_line(raw)

        if cleaned:
            current.append(cleaned)

    if current:
        articles.append("\n".join(current))

    return articles

train_articles = extract_articles("train")
val_articles = extract_articles("validation")

print("Training articles:", len(train_articles))
print("Validation articles:", len(val_articles))

print("\nSAMPLE:")
print(train_articles[0][:600])


Training articles: 29443
Validation articles: 60

SAMPLE:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War wh


In [ ]:

import json

def save_jsonl(articles, path):
    with open(path, "w", encoding="utf-8") as f:
        for article in articles:
            f.write(
                json.dumps(
                    {"text": article},
                    ensure_ascii=False
                ) + "\n"
            )

train_path = v2_dir / "train_articles.jsonl"
val_path = v2_dir / "validation_articles.jsonl"

save_jsonl(train_articles, train_path)
save_jsonl(val_articles, val_path)

print("Saved:", train_path)
print("Saved:", val_path)
print("Training articles:", len(train_articles))
print("Validation articles:", len(val_articles))


Saved: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/v2/train_articles.jsonl
Saved: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/v2/validation_articles.jsonl
Training articles: 29443
Validation articles: 60


In [ ]:

from collections import Counter
import re

patterns = {
    "Hyphen artifacts": r"@\s*-\s*@",
    "Decimal artifacts": r"@\s*\.\s*@",
    "Comma artifacts": r"@\s*,\s*@",
    "Wiki headings": r"(?m)^\s*=+.*=+\s*$"
}

counts = Counter()

for article in train_articles:
    for name, pattern in patterns.items():
        counts[name] += len(re.findall(pattern, article))

print("=" * 60)
print("GRAC-LM v0.4 — V2 CORPUS AUDIT")
print("=" * 60)

for name, count in counts.items():
    print(f"{name}: {count:,}")

print("\nFirst article:")
print(train_articles[0][:500])

print("\nAudit complete.")


GRAC-LM v0.4 — V2 CORPUS AUDIT
Hyphen artifacts: 0
Decimal artifacts: 0
Comma artifacts: 0
Wiki headings: 275,631

First article:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows th

Audit complete.


In [ ]:

import numpy as np
import json
from pathlib import Path

train_output = v2_dir / "train_tokens_v2.npy"
eos_id = tokenizer.token_to_id("<eos>")

assert eos_id is not None, "Tokenizer has no <eos> token."

print("Starting document-aware tokenization...", flush=True)

if train_output.exists():
    print("Existing output preserved:", train_output)
else:
    temp_dir = Path("/content/grac_v2_chunks")
    temp_dir.mkdir(parents=True, exist_ok=True)

    chunk_paths = []
    token_buffer = []
    document_count = 0

    def flush_chunk():
        global token_buffer

        if not token_buffer:
            return

        path = temp_dir / f"tokens_{len(chunk_paths):05d}.npy"
        np.save(path, np.asarray(token_buffer, dtype=np.uint16))
        chunk_paths.append(path)
        token_buffer = []

    with open(train_path, encoding="utf-8") as f:
        for line in f:
            article = json.loads(line)["text"]
            ids = tokenizer.encode(article).ids

            token_buffer.extend(ids)
            token_buffer.append(eos_id)

            document_count += 1

            if len(token_buffer) >= 250_000:
                flush_chunk()

            if document_count % 100 == 0:
                print(
                    f"Tokenized {document_count:,} articles",
                    flush=True
                )

    flush_chunk()

    total_tokens = sum(
        len(np.load(p, mmap_mode="r"))
        for p in chunk_paths
    )

    assert total_tokens > 513

    output = np.lib.format.open_memmap(
        train_output,
        mode="w+",
        dtype=np.uint16,
        shape=(total_tokens,)
    )

    offset = 0

    for path in chunk_paths:
        chunk = np.load(path, mmap_mode="r")
        output[offset:offset + len(chunk)] = chunk
        offset += len(chunk)

    output.flush()
    del output

    print("Tokenization complete.")

tokens_v2 = np.load(train_output, mmap_mode="r")

print("Total tokens:", f"{len(tokens_v2):,}")
print("Output:", train_output)


Starting document-aware tokenization...
Tokenized 100 articles
Tokenized 200 articles
Tokenized 300 articles
Tokenized 400 articles
Tokenized 500 articles
Tokenized 600 articles
Tokenized 700 articles
Tokenized 800 articles
Tokenized 900 articles
Tokenized 1,000 articles
Tokenized 1,100 articles
Tokenized 1,200 articles
Tokenized 1,300 articles
Tokenized 1,400 articles
Tokenized 1,500 articles
Tokenized 1,600 articles
Tokenized 1,700 articles
Tokenized 1,800 articles
Tokenized 1,900 articles
Tokenized 2,000 articles
Tokenized 2,100 articles
Tokenized 2,200 articles
Tokenized 2,300 articles
Tokenized 2,400 articles
Tokenized 2,500 articles
Tokenized 2,600 articles
Tokenized 2,700 articles
Tokenized 2,800 articles
Tokenized 2,900 articles
Tokenized 3,000 articles
Tokenized 3,100 articles
Tokenized 3,200 articles
Tokenized 3,300 articles
Tokenized 3,400 articles
Tokenized 3,500 articles
Tokenized 3,600 articles
Tokenized 3,700 articles
Tokenized 3,800 articles
Tokenized 3,900 articles
Tok

In [ ]:

import numpy as np
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — DOCUMENT-AWARE TOKEN VERIFICATION")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
token_path = base_dir / "data/v2/train_tokens_v2.npy"

assert token_path.exists(), "Cell 37 must finish first!"

tokens_v2 = np.load(token_path, mmap_mode="r")

vocab_size = GRAC_V04_CONFIG["vocab_size"]
eos_id = tokenizer.token_to_id("<eos>")

assert eos_id is not None, "EOS token missing!"
assert len(tokens_v2) > 513, "Dataset too small!"
assert int(tokens_v2.min()) >= 0
assert int(tokens_v2.max()) < vocab_size

print("Total tokens:", f"{len(tokens_v2):,}")
print("Token dtype:", tokens_v2.dtype)
print("Minimum token ID:", int(tokens_v2.min()))
print("Maximum token ID:", int(tokens_v2.max()))

print("\nDECODED SAMPLE:")
sample = tokens_v2[:200].astype(int).tolist()
print(tokenizer.decode(sample, skip_special_tokens=False))

eos_count = int(np.count_nonzero(tokens_v2 == eos_id))

print("\nEnd-of-document tokens:", f"{eos_count:,}")

print("\n" + "=" * 65)
print("VERIFICATION PASSED!")
print("=" * 65)


GRAC-LM v0.4 — DOCUMENT-AWARE TOKEN VERIFICATION
Total tokens: 136,126,238
Token dtype: uint16
Minimum token ID: 1
Maximum token ID: 7999

DECODED SAMPLE:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .<unk>The game began development in

End-of-document tokens: 29,443

VERIFICATION PASSED!


In [ ]:

import json
import numpy as np
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — DATASET INTEGRITY AUDIT")
print("=" * 65)

v2_dir = Path(
    "/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/v2"
)

train_path = v2_dir / "train_articles.jsonl"
token_path = v2_dir / "train_tokens_v2.npy"

tokens_v2 = np.load(token_path, mmap_mode="r")

eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")

with open(train_path, encoding="utf-8") as f:
    article_count = sum(1 for line in f if line.strip())

eos_count = int(np.count_nonzero(tokens_v2 == eos_id))

unk_count = (
    int(np.count_nonzero(tokens_v2 == unk_id))
    if unk_id is not None
    else None
)

print("Article records:", f"{article_count:,}")
print("EOS tokens:", f"{eos_count:,}")
print("Unknown tokens:", unk_count)
print("Total tokens:", f"{len(tokens_v2):,}")

if unk_count is not None:
    print(
        "Unknown-token percentage:",
        f"{100 * unk_count / len(tokens_v2):.4f}%"
    )

print("\nTOKENIZER SPECIAL TOKEN TEST:")

for sample in [
    "Hello world.",
    "Hello\nworld.",
    "Hello\n\nworld."
]:
    encoded = tokenizer.encode(sample)
    print("\nInput:", repr(sample))
    print("Token IDs:", encoded.ids)
    print(
        "Decoded:",
        repr(tokenizer.decode(
            encoded.ids,
            skip_special_tokens=False
        ))
    )

print("\nEOS count equals article count:",
      eos_count == article_count)

print("\nAUDIT COMPLETE")


GRAC-LM v0.4 — DATASET INTEGRITY AUDIT
Article records: 29,443
EOS tokens: 29,443
Unknown tokens: 1106143
Total tokens: 136,126,238
Unknown-token percentage: 0.8126%

TOKENIZER SPECIAL TOKEN TEST:

Input: 'Hello world.'
Token IDs: [43, 425, 82, 1245, 17]
Decoded: 'Hello world.'

Input: 'Hello\nworld.'
Token IDs: [43, 425, 82, 1, 90, 735, 17]
Decoded: 'Hello<unk>world.'

Input: 'Hello\n\nworld.'
Token IDs: [43, 425, 82, 1, 1, 90, 735, 17]
Decoded: 'Hello<unk><unk>world.'

EOS count equals article count: True

AUDIT COMPLETE


In [ ]:

import json
from pathlib import Path

train_path = Path(
    "/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/v2/train_articles.jsonl"
)

print("=" * 60)
print("GRAC-LM v0.4 — FAST NEWLINE INVESTIGATION")
print("=" * 60)

article_count = 0
newline_count = 0

with open(train_path, encoding="utf-8") as f:
    for line in f:
        article = json.loads(line)["text"]

        article_count += 1
        newline_count += article.count("\n")

        if article_count % 5000 == 0:
            print(
                f"Processed {article_count:,} articles",
                flush=True
            )

print("\nTotal articles:", f"{article_count:,}")
print("Total newlines:", f"{newline_count:,}")
print("Previously measured unknown tokens: 1,106,143")

print("\nINVESTIGATION COMPLETE")


GRAC-LM v0.4 — FAST NEWLINE INVESTIGATION
Processed 5,000 articles
Processed 10,000 articles
Processed 15,000 articles
Processed 20,000 articles
Processed 25,000 articles

Total articles: 29,443
Total newlines: 1,106,143
Previously measured unknown tokens: 1,106,143

INVESTIGATION COMPLETE


In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — NEWLINE FIX TEST")
print("=" * 65)

samples = [
    "Hello\nworld.",
    "Hello\n\nworld.",
    "Artificial intelligence\nis a field of computer science.",
    "The government\nannounced a new policy.",
    "Scientists\n\nmade an important discovery."
]

unk_id = tokenizer.token_to_id("<unk>")

for original in samples:

    # Replace newlines and repeated whitespace with spaces
    cleaned = " ".join(original.split())

    before = tokenizer.encode(original).ids
    after = tokenizer.encode(cleaned).ids

    before_unknown = before.count(unk_id)
    after_unknown = after.count(unk_id)

    print("\n" + "-" * 55)
    print("ORIGINAL:", repr(original))
    print("CLEANED:", repr(cleaned))

    print("Unknown tokens BEFORE:", before_unknown)
    print("Unknown tokens AFTER:", after_unknown)

    print("Decoded result:", tokenizer.decode(after))

print("\n" + "=" * 65)
print("CELL 41 TEST COMPLETE!")
print("=" * 65)


GRAC-LM v0.4 — NEWLINE FIX TEST

-------------------------------------------------------
ORIGINAL: 'Hello\nworld.'
CLEANED: 'Hello world.'
Unknown tokens BEFORE: 1
Unknown tokens AFTER: 0
Decoded result: Hello world.

-------------------------------------------------------
ORIGINAL: 'Hello\n\nworld.'
CLEANED: 'Hello world.'
Unknown tokens BEFORE: 2
Unknown tokens AFTER: 0
Decoded result: Hello world.

-------------------------------------------------------
ORIGINAL: 'Artificial intelligence\nis a field of computer science.'
CLEANED: 'Artificial intelligence is a field of computer science.'
Unknown tokens BEFORE: 1
Unknown tokens AFTER: 0
Decoded result: Artificial intelligence is a field of computer science.

-------------------------------------------------------
ORIGINAL: 'The government\nannounced a new policy.'
CLEANED: 'The government announced a new policy.'
Unknown tokens BEFORE: 1
Unknown tokens AFTER: 0
Decoded result: The government announced a new policy.

------------------

In [ ]:

import json
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
v2_dir = base_dir / "data/v2"

train_path = v2_dir / "train_articles.jsonl"
output_path = v2_dir / "train_tokens_v3_fixed.npy"

assert train_path.exists()

eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")
vocab_size = GRAC_V04_CONFIG["vocab_size"]

assert eos_id is not None
assert unk_id is not None

print("=" * 65)
print("GRAC-LM v0.4 — CORRECTED TOKEN DATASET")
print("=" * 65, flush=True)

def normalized_articles():
    with open(train_path, encoding="utf-8") as f:
        for line in f:
            article = json.loads(line)["text"]
            yield " ".join(article.split())

if output_path.exists():
    print("Existing corrected dataset found; not overwriting.")
else:
    # First pass: count tokens without retaining them
    total_tokens = 0
    article_count = 0

    for article in normalized_articles():
        ids = tokenizer.encode(article).ids
        total_tokens += len(ids) + 1
        article_count += 1

        if article_count % 1000 == 0:
            print(
                f"Counting: {article_count:,} articles",
                flush=True
            )

    print("Total expected tokens:", f"{total_tokens:,}")

    # Write locally first to avoid leaving a partial
    # final file in Google Drive.
    with TemporaryDirectory(dir="/content") as temp:
        temp_path = Path(temp) / "tokens.npy"

        output = np.lib.format.open_memmap(
            temp_path,
            mode="w+",
            dtype=np.uint16,
            shape=(total_tokens,)
        )

        offset = 0
        unknown_count = 0
        written_articles = 0

        for article in normalized_articles():
            ids = tokenizer.encode(article).ids

            assert all(0 <= i < vocab_size for i in ids)

            length = len(ids)

            output[offset:offset + length] = ids
            offset += length

            output[offset] = eos_id
            offset += 1

            unknown_count += ids.count(unk_id)
            written_articles += 1

            if written_articles % 1000 == 0:
                print(
                    f"Writing: {written_articles:,} articles",
                    flush=True
                )

        assert offset == total_tokens
        assert written_articles == article_count

        output.flush()
        del output

        import shutil
        shutil.copy2(temp_path, output_path)

    print("\nCORRECTED DATASET SAVED!")
    print("Articles:", f"{article_count:,}")
    print("Total tokens:", f"{total_tokens:,}")
    print("Unknown tokens:", f"{unknown_count:,}")
    print(
        "Unknown percentage:",
        f"{100 * unknown_count / total_tokens:.4f}%"
    )

fixed_tokens = np.load(output_path, mmap_mode="r")

print("\nSaved to:", output_path)
print("Token count:", f"{len(fixed_tokens):,}")
print("CELL 42 COMPLETE!")


GRAC-LM v0.4 — CORRECTED TOKEN DATASET
Counting: 1,000 articles
Counting: 2,000 articles
Counting: 3,000 articles
Counting: 4,000 articles
Counting: 5,000 articles
Counting: 6,000 articles
Counting: 7,000 articles
Counting: 8,000 articles
Counting: 9,000 articles
Counting: 10,000 articles
Counting: 11,000 articles
Counting: 12,000 articles
Counting: 13,000 articles
Counting: 14,000 articles
Counting: 15,000 articles
Counting: 16,000 articles
Counting: 17,000 articles
Counting: 18,000 articles
Counting: 19,000 articles
Counting: 20,000 articles
Counting: 21,000 articles
Counting: 22,000 articles
Counting: 23,000 articles
Counting: 24,000 articles
Counting: 25,000 articles
Counting: 26,000 articles
Counting: 27,000 articles
Counting: 28,000 articles
Counting: 29,000 articles
Total expected tokens: 134,676,072
Writing: 1,000 articles
Writing: 2,000 articles
Writing: 3,000 articles
Writing: 4,000 articles
Writing: 5,000 articles
Writing: 6,000 articles
Writing: 7,000 articles
Writing: 8,00

In [ ]:

import numpy as np

print("=" * 65)
print("GRAC-LM v0.4 — FINAL DATASET VERIFICATION")
print("=" * 65)

fixed_tokens = np.load(
    v2_dir / "train_tokens_v3_fixed.npy",
    mmap_mode="r"
)

eos_count = int(np.count_nonzero(fixed_tokens == eos_id))
unk_count = int(np.count_nonzero(fixed_tokens == unk_id))

print("Total tokens:", f"{len(fixed_tokens):,}")
print("EOS tokens:", f"{eos_count:,}")
print("Unknown tokens:", f"{unk_count:,}")
print(
    "Unknown-token percentage:",
    f"{100 * unk_count / len(fixed_tokens):.4f}%"
)

print("\nDECODED SAMPLE:")
print(
    tokenizer.decode(
        fixed_tokens[:200].astype(int).tolist(),
        skip_special_tokens=False
    )
)

assert eos_count == 29443, "EOS count mismatch!"
assert int(fixed_tokens.max()) < 8000

print("\nFINAL VERIFICATION PASSED!")


GRAC-LM v0.4 — FINAL DATASET VERIFICATION
Total tokens: 134,676,072
EOS tokens: 29,443
Unknown tokens: 0
Unknown-token percentage: 0.0000%

DECODED SAMPLE:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . The game began development in 2010

FINAL VERIFICATION PASSED!


In [ ]:

import numpy as np
import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — DOCUMENT-AWARE BATCH GENERATOR")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")

token_path = (
    base_dir / "data/v2/train_tokens_v3_fixed.npy"
)

assert token_path.exists(), "Run Cells 42 and 43 first."

fixed_tokens = np.load(token_path, mmap_mode="r")

eos_id = tokenizer.token_to_id("<eos>")
context_length = GRAC_V04_CONFIG["context_length"]

assert eos_id is not None

# Find all document boundaries
eos_positions = np.flatnonzero(fixed_tokens == eos_id)

# Document starts: first token, or token after previous EOS
document_starts = np.concatenate([
    np.array([0], dtype=np.int64),
    eos_positions[:-1] + 1
])

# Document ends are inclusive EOS positions
document_ends = eos_positions

# Each training sample needs 512 inputs + 512 targets,
# shifted by one token: 513 tokens in total.
valid_lengths = document_ends - document_starts + 1

valid_mask = valid_lengths >= context_length + 1

valid_starts = document_starts[valid_mask]
valid_ends = document_ends[valid_mask]

# Number of possible windows per document
window_counts = (
    valid_ends - valid_starts - context_length + 1
).astype(np.int64)

cumulative_windows = np.cumsum(window_counts)

total_windows = int(cumulative_windows[-1])

rng_doc = np.random.default_rng(2026)

print("Total documents:", len(document_starts))
print("Documents usable for 512-token training:", len(valid_starts))
print("Possible training windows:", f"{total_windows:,}")

def get_document_batch(batch_size=4):
    # Uniformly sample all valid 513-token windows.
    draws = rng_doc.integers(
        0,
        total_windows,
        size=batch_size,
        dtype=np.int64
    )

    doc_indices = np.searchsorted(
        cumulative_windows,
        draws,
        side="right"
    )

    previous_totals = np.where(
        doc_indices == 0,
        0,
        cumulative_windows[
            np.maximum(doc_indices - 1, 0)
        ]
    )

    offsets = draws - previous_totals

    starts = valid_starts[doc_indices] + offsets

    x = np.stack([
        fixed_tokens[int(s):int(s) + context_length]
        for s in starts
    ]).astype(np.int64)

    y = np.stack([
        fixed_tokens[int(s) + 1:int(s) + context_length + 1]
        for s in starts
    ]).astype(np.int64)

    return (
        torch.from_numpy(x).to(device),
        torch.from_numpy(y).to(device)
    )

# Test one batch
x, y = get_document_batch(batch_size=4)

print("\nInput shape:", tuple(x.shape))
print("Target shape:", tuple(y.shape))

assert x.shape == (4, context_length)
assert y.shape == (4, context_length)

assert torch.equal(x[:, 1:], y[:, :-1]), (
    "Input/target shift mismatch!"
)

print("\nDECODED TRAINING EXAMPLE:")
print(
    tokenizer.decode(
        x[0, :100].cpu().tolist(),
        skip_special_tokens=False
    )
)

print("\nCELL 44 PASSED — DOCUMENT-AWARE BATCHES READY!")


GRAC-LM v0.4 — DOCUMENT-AWARE BATCH GENERATOR
Total documents: 29443
Documents usable for 512-token training: 28700
Possible training windows: 119,894,059

Input shape: (4, 512)
Target shape: (4, 512)

DECODED TRAINING EXAMPLE:
rated as Bishop of Worcester in 961 . In 972 , Oswald was promoted to the see of York , although he continued to hold Worcester also . As bishop and archbishop , Oswald was a supporter and one of the leading promoters ( together with Æthelwold ) of Dunstan 's reforms of the church , including monastic reforms . Oswald founded a number of monasteries , including Ram

CELL 44 PASSED — DOCUMENT-AWARE BATCHES READY!


In [ ]:

import json
import torch
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase3_dir = base_dir / "phase3"
phase3_dir.mkdir(parents=True, exist_ok=True)

PHASE3_CONFIG = {
    "version": "0.4-phase3",
    "starting_checkpoint": "pretraining-best-8750",
    "training_steps": 500,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5,
    "weight_decay": 0.1,
    "eval_every": 100,
    "context_length": 512,
    "gradient_clip": 1.0
}

assert torch.cuda.is_available(), (
    "Enable a GPU runtime before Phase 3 training."
)

with open(phase3_dir / "config.json", "w") as f:
    json.dump(PHASE3_CONFIG, f, indent=2)

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 3 CONFIGURATION")
print("=" * 65)
print(json.dumps(PHASE3_CONFIG, indent=2))
print("GPU:", torch.cuda.get_device_name(0))


GRAC-LM v0.4 — PHASE 3 CONFIGURATION
{
  "version": "0.4-phase3",
  "starting_checkpoint": "pretraining-best-8750",
  "training_steps": 500,
  "micro_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 1e-05,
  "weight_decay": 0.1,
  "eval_every": 100,
  "context_length": 512,
  "gradient_clip": 1.0
}
GPU: Tesla T4


In [ ]:

import torch

original_checkpoint_path = (
    base_dir
    / "archive"
    / "GRAC-LM-v0.4-pretraining-best-8750.pt"
)

assert original_checkpoint_path.exists(), (
    "Original archived checkpoint missing."
)

ckpt = torch.load(
    original_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert ckpt["step"] == 8750

phase3_model = GRACLM(GRAC_V04_CONFIG).to(device)
phase3_model.load_state_dict(ckpt["model_state_dict"])
phase3_model.train()

optimizer3 = torch.optim.AdamW(
    phase3_model.parameters(),
    lr=PHASE3_CONFIG["learning_rate"],
    weight_decay=PHASE3_CONFIG["weight_decay"]
)

scaler3 = torch.amp.GradScaler("cuda")

print("GRAC-LM v0.4 — PHASE 3 MODEL READY")
print("Starting step:", ckpt["step"])
print(
    "Parameters:",
    f"{sum(p.numel() for p in phase3_model.parameters()):,}"
)
print("Device:", device)


GRAC-LM v0.4 — PHASE 3 MODEL READY
Starting step: 8750
Parameters: 54,684,160
Device: cuda


In [ ]:

assert "fixed_indices" in globals(), (
    "Run Cell 20 to restore the fixed benchmark."
)
assert "val_tokens" in globals(), (
    "Restore the original validation tokens first."
)
assert "evaluate_fixed_checkpoint" in globals(), (
    "Run Cell 21 to define the evaluation function."
)

original_fixed_loss = evaluate_fixed_checkpoint(
    original_checkpoint_path
)

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 3 BASELINE")
print("=" * 65)

print(f"Original fixed validation loss: {original_fixed_loss:.4f}")
print("Starting checkpoint: 8750")
print("Ready for controlled training.")


GRAC-LM v0.4 — PHASE 3 BASELINE
Original fixed validation loss: 3.4165
Starting checkpoint: 8750
Ready for controlled training.


In [ ]:

import torch
import math

steps = PHASE3_CONFIG["training_steps"]
accum = PHASE3_CONFIG["gradient_accumulation_steps"]
eval_every = PHASE3_CONFIG["eval_every"]

best_phase3_loss = original_fixed_loss

@torch.no_grad()
def evaluate_phase3():
    phase3_model.eval()

    total_loss = 0.0
    total_samples = 0

    for start in range(0, len(fixed_indices), 4):
        indices = fixed_indices[start:start + 4]

        x = torch.stack([
            torch.tensor(
                val_tokens[int(i):int(i) + 512],
                dtype=torch.long
            )
            for i in indices
        ]).to(device)

        y = torch.stack([
            torch.tensor(
                val_tokens[int(i) + 1:int(i) + 513],
                dtype=torch.long
            )
            for i in indices
        ]).to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            _, loss = phase3_model(x, targets=y)

        total_loss += loss.item() * len(indices)
        total_samples += len(indices)

    phase3_model.train()
    return total_loss / total_samples

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 3 TRAINING")
print("=" * 65)

for step in range(1, steps + 1):
    phase3_model.train()
    optimizer3.zero_grad(set_to_none=True)

    running_loss = 0.0

    for micro in range(accum):
        x, y = get_document_batch(
            batch_size=PHASE3_CONFIG["micro_batch_size"]
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            _, loss = phase3_model(x, targets=y)
            running_loss += loss.item()
            loss = loss / accum

        scaler3.scale(loss).backward()

    scaler3.unscale_(optimizer3)

    torch.nn.utils.clip_grad_norm_(
        phase3_model.parameters(),
        PHASE3_CONFIG["gradient_clip"]
    )

    scaler3.step(optimizer3)
    scaler3.update()

    if step % 25 == 0:
        print(
            f"Step {step}/{steps} | "
            f"Train loss: {running_loss / accum:.4f}",
            flush=True
        )

    if step % eval_every == 0:
        val_loss = evaluate_phase3()

        print(
            f"\nCHECKPOINT {step} | "
            f"Validation loss: {val_loss:.4f}",
            flush=True
        )

        state = {
            "model_state_dict": phase3_model.state_dict(),
            "optimizer_state_dict": optimizer3.state_dict(),
            "scaler_state_dict": scaler3.state_dict(),
            "step": step,
            "validation_loss": val_loss,
            "config": GRAC_V04_CONFIG
        }

        torch.save(
            state,
            phase3_dir / f"GRAC-LM-v0.4-phase3-step{step}.pt"
        )

        if val_loss < best_phase3_loss:
            best_phase3_loss = val_loss

            torch.save(
                state,
                phase3_dir / "GRAC-LM-v0.4-phase3-best.pt"
            )

print("\nPHASE 3 TRAINING COMPLETE!")
print("Original loss:", original_fixed_loss)
print("Best Phase 3 loss:", best_phase3_loss)


GRAC-LM v0.4 — PHASE 3 TRAINING
Step 25/500 | Train loss: 3.5094
Step 50/500 | Train loss: 3.5362
Step 75/500 | Train loss: 3.5719
Step 100/500 | Train loss: 3.6429

CHECKPOINT 100 | Validation loss: 3.4377
Step 125/500 | Train loss: 3.5559
Step 150/500 | Train loss: 3.5413
Step 175/500 | Train loss: 3.5550
Step 200/500 | Train loss: 3.5563

CHECKPOINT 200 | Validation loss: 3.4424
Step 225/500 | Train loss: 3.6464
Step 250/500 | Train loss: 3.4961
Step 275/500 | Train loss: 3.5924
Step 300/500 | Train loss: 3.5913

CHECKPOINT 300 | Validation loss: 3.4480
Step 325/500 | Train loss: 3.6355
Step 350/500 | Train loss: 3.6505
Step 375/500 | Train loss: 3.6812
Step 400/500 | Train loss: 3.5853

CHECKPOINT 400 | Validation loss: 3.4502
Step 425/500 | Train loss: 3.5946
Step 450/500 | Train loss: 3.5688
Step 475/500 | Train loss: 3.4562
Step 500/500 | Train loss: 3.5329

CHECKPOINT 500 | Validation loss: 3.4561

PHASE 3 TRAINING COMPLETE!
Original loss: 3.4164867103099823
Best Phase 3 loss: 

In [ ]:

from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 3 FINAL COMPARISON")
print("=" * 65)

original_loss = evaluate_fixed_checkpoint(
    original_checkpoint_path
)

phase3_best_path = (
    phase3_dir / "GRAC-LM-v0.4-phase3-best.pt"
)

if phase3_best_path.exists():
    phase3_loss = evaluate_fixed_checkpoint(
        phase3_best_path
    )

    print(f"Original loss: {original_loss:.4f}")
    print(f"Phase 3 loss:  {phase3_loss:.4f}")

    if phase3_loss < original_loss:
        print("\nPhase 3 improved fixed validation loss!")
    else:
        print("\nPhase 3 did not improve fixed validation loss.")
else:
    print(f"Original loss: {original_loss:.4f}")
    print("No Phase 3 checkpoint beat the original baseline.")

print("\nOriginal checkpoints remain safe.")


GRAC-LM v0.4 — PHASE 3 FINAL COMPARISON
Original loss: 3.4165
No Phase 3 checkpoint beat the original baseline.

Original checkpoints remain safe.


In [ ]:

import torch
import numpy as np
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 3 DIAGNOSTIC REPORT")
print("=" * 65)

results = []

for step in [100, 200, 300, 400, 500]:
    path = phase3_dir / f"GRAC-LM-v0.4-phase3-step{step}.pt"

    if not path.exists():
        print(f"Step {step}: checkpoint missing")
        continue

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    loss = checkpoint["validation_loss"]
    results.append((step, loss))

    print(
        f"Step {step:3d} | "
        f"Validation loss: {loss:.4f} | "
        f"Change: {loss - original_fixed_loss:+.4f}"
    )

print("\nORIGINAL BASELINE:", f"{original_fixed_loss:.4f}")

if results:
    best_step, best_loss = min(results, key=lambda x: x[1])

    print("BEST PHASE 3 STEP:", best_step)
    print("BEST PHASE 3 LOSS:", f"{best_loss:.4f}")

    if best_loss < original_fixed_loss:
        print("RESULT: Phase 3 improved.")
    else:
        print("RESULT: Original checkpoint remains better.")

print("\nDATASET INTEGRITY")

old_path = base_dir / "data/v2/train_tokens_v2.npy"
new_path = base_dir / "data/v2/train_tokens_v3_fixed.npy"

unk_id = tokenizer.token_to_id("<unk>")

for label, path in [
    ("Previous dataset", old_path),
    ("Corrected dataset", new_path)
]:
    if path.exists():
        tokens = np.load(path, mmap_mode="r")
        unknown = 0

        for start in range(0, len(tokens), 2_000_000):
            chunk = tokens[start:start + 2_000_000]
            unknown += int(np.count_nonzero(chunk == unk_id))

        percentage = 100 * unknown / len(tokens)

        print(
            f"{label}: "
            f"{len(tokens):,} tokens | "
            f"{unknown:,} unknown | "
            f"{percentage:.4f}% unknown"
        )

print("\nCELL 50 COMPLETE!")


GRAC-LM v0.4 — PHASE 3 DIAGNOSTIC REPORT
Step 100 | Validation loss: 3.4377 | Change: +0.0212
Step 200 | Validation loss: 3.4424 | Change: +0.0259
Step 300 | Validation loss: 3.4480 | Change: +0.0315
Step 400 | Validation loss: 3.4502 | Change: +0.0337
Step 500 | Validation loss: 3.4561 | Change: +0.0396

ORIGINAL BASELINE: 3.4165
BEST PHASE 3 STEP: 100
BEST PHASE 3 LOSS: 3.4377
RESULT: Original checkpoint remains better.

DATASET INTEGRITY
Previous dataset: 136,126,238 tokens | 1,106,143 unknown | 0.8126% unknown
Corrected dataset: 134,676,072 tokens | 0 unknown | 0.0000% unknown

CELL 50 COMPLETE!


In [ ]:

import json
import numpy as np
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — CLEAN VALIDATION DATASET")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
v2_dir = base_dir / "data/v2"

source_path = v2_dir / "validation_articles.jsonl"
output_path = v2_dir / "validation_tokens_v3_fixed.npy"

assert source_path.exists(), "Validation article file missing."

eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")

total_tokens = 0
article_count = 0

# Pass 1: count required tokens
with open(source_path, encoding="utf-8") as f:
    for line in f:
        article = json.loads(line)["text"]
        article = " ".join(article.split())

        total_tokens += len(tokenizer.encode(article).ids) + 1
        article_count += 1

print("Validation articles:", article_count)
print("Expected tokens:", f"{total_tokens:,}")

# Pass 2: create the validation file
# Use a temporary filename until writing is complete.
temp_path = output_path.with_name(
    "validation_tokens_v3_fixed.partial.npy"
)

tokens = np.lib.format.open_memmap(
    temp_path,
    mode="w+",
    dtype=np.uint16,
    shape=(total_tokens,)
)

offset = 0
unknown_count = 0

with open(source_path, encoding="utf-8") as f:
    for line in f:
        article = json.loads(line)["text"]
        article = " ".join(article.split())

        ids = tokenizer.encode(article).ids
        length = len(ids)

        tokens[offset:offset + length] = ids
        offset += length

        tokens[offset] = eos_id
        offset += 1

        unknown_count += ids.count(unk_id)

assert offset == total_tokens

tokens.flush()
del tokens

temp_path.replace(output_path)

print("\nSaved:", output_path)
print("Total tokens:", f"{total_tokens:,}")
print("Unknown tokens:", f"{unknown_count:,}")
print(
    "Unknown percentage:",
    f"{100 * unknown_count / total_tokens:.4f}%"
)

print("\nCELL 51 COMPLETE!")


GRAC-LM v0.4 — CLEAN VALIDATION DATASET
Validation articles: 60
Expected tokens: 281,564

Saved: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/data/v2/validation_tokens_v3_fixed.npy
Total tokens: 281,564
Unknown tokens: 0
Unknown percentage: 0.0000%

CELL 51 COMPLETE!


In [ ]:

import torch
import numpy as np
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — CLEAN VALIDATION BENCHMARK")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase3_dir = base_dir / "phase3"

clean_val_path = (
    base_dir / "data/v2/validation_tokens_v3_fixed.npy"
)

assert clean_val_path.exists(), "Run Cell 51 first."

clean_val_tokens = np.load(
    clean_val_path,
    mmap_mode="r"
)

context_length = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")

# Identify document boundaries in validation data.
eos_positions = np.flatnonzero(clean_val_tokens == eos_id)

document_starts = np.concatenate([
    np.array([0], dtype=np.int64),
    eos_positions[:-1] + 1
])

document_ends = eos_positions

valid_lengths = document_ends - document_starts + 1
valid_mask = valid_lengths >= context_length + 1

valid_starts = document_starts[valid_mask]
valid_ends = document_ends[valid_mask]

assert len(valid_starts) > 0, (
    "No validation documents are long enough."
)

window_counts = (
    valid_ends - valid_starts - context_length + 1
).astype(np.int64)

cumulative_windows = np.cumsum(window_counts)
total_windows = int(cumulative_windows[-1])

# Deterministic validation windows: same for every checkpoint.
rng_val = np.random.default_rng(2026)

sample_count = min(256, total_windows)

draws = rng_val.choice(
    total_windows,
    size=sample_count,
    replace=False
)

doc_indices = np.searchsorted(
    cumulative_windows,
    draws,
    side="right"
)

previous_totals = np.where(
    doc_indices == 0,
    0,
    cumulative_windows[np.maximum(doc_indices - 1, 0)]
)

offsets = draws - previous_totals
benchmark_starts = valid_starts[doc_indices] + offsets

print("Clean validation tokens:", f"{len(clean_val_tokens):,}")
print("Eligible documents:", len(valid_starts))
print("Benchmark windows:", len(benchmark_starts))

@torch.no_grad()
def evaluate_clean_checkpoint(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = GRACLM(GRAC_V04_CONFIG).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    weighted_loss = 0.0
    sample_total = 0

    for batch_start in range(0, len(benchmark_starts), 4):
        positions = benchmark_starts[batch_start:batch_start + 4]

        x = np.stack([
            clean_val_tokens[int(p):int(p) + context_length]
            for p in positions
        ]).astype(np.int64)

        y = np.stack([
            clean_val_tokens[int(p) + 1:int(p) + context_length + 1]
            for p in positions
        ]).astype(np.int64)

        x = torch.from_numpy(x).to(device)
        y = torch.from_numpy(y).to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(str(device).startswith("cuda"))
        ):
            _, loss = model(x, targets=y)

        weighted_loss += loss.item() * len(positions)
        sample_total += len(positions)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return weighted_loss / sample_total


checkpoints_to_test = [
    (
        "Original 8750",
        base_dir / "archive/GRAC-LM-v0.4-pretraining-best-8750.pt"
    )
]

for step in [100, 200, 300, 400, 500]:
    path = (
        phase3_dir / f"GRAC-LM-v0.4-phase3-step{step}.pt"
    )

    if path.exists():
        checkpoints_to_test.append((f"Phase 3 step {step}", path))

results_clean = []

print("\nEVALUATING CHECKPOINTS\n")

for name, path in checkpoints_to_test:
    print(f"Testing: {name}...", flush=True)

    loss = evaluate_clean_checkpoint(path)
    results_clean.append((name, loss))

    print(f"  Clean validation loss: {loss:.4f}", flush=True)

print("\n" + "=" * 65)
print("FINAL CLEAN BENCHMARK RESULTS")
print("=" * 65)

for name, loss in results_clean:
    print(f"{name:22s} | Loss: {loss:.4f}")

winner = min(results_clean, key=lambda result: result[1])

print("\nBEST CHECKPOINT:", winner[0])
print("BEST CLEAN LOSS:", f"{winner[1]:.4f}")

original_clean_loss = results_clean[0][1]

for name, loss in results_clean[1:]:
    print(
        f"{name}: "
        f"{loss - original_clean_loss:+.4f} vs original"
    )

print("\nCELL 52 COMPLETE!")


GRAC-LM v0.4 — CLEAN VALIDATION BENCHMARK
Clean validation tokens: 281,564
Eligible documents: 60
Benchmark windows: 256

EVALUATING CHECKPOINTS

Testing: Original 8750...
  Clean validation loss: 3.6244
Testing: Phase 3 step 100...
  Clean validation loss: 3.5035
Testing: Phase 3 step 200...
  Clean validation loss: 3.4840
Testing: Phase 3 step 300...
  Clean validation loss: 3.4734
Testing: Phase 3 step 400...
  Clean validation loss: 3.4659
Testing: Phase 3 step 500...
  Clean validation loss: 3.4621

FINAL CLEAN BENCHMARK RESULTS
Original 8750          | Loss: 3.6244
Phase 3 step 100       | Loss: 3.5035
Phase 3 step 200       | Loss: 3.4840
Phase 3 step 300       | Loss: 3.4734
Phase 3 step 400       | Loss: 3.4659
Phase 3 step 500       | Loss: 3.4621

BEST CHECKPOINT: Phase 3 step 500
BEST CLEAN LOSS: 3.4621
Phase 3 step 100: -0.1209 vs original
Phase 3 step 200: -0.1404 vs original
Phase 3 step 300: -0.1509 vs original
Phase 3 step 400: -0.1584 vs original
Phase 3 step 500: -0.

In [ ]:

import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — GENERATION SHOWDOWN")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")

original_path = (
    base_dir / "archive/GRAC-LM-v0.4-pretraining-best-8750.pt"
)

phase3_path = (
    base_dir / "phase3/GRAC-LM-v0.4-phase3-step500.pt"
)

assert original_path.exists(), "Original checkpoint missing!"
assert phase3_path.exists(), "Phase 3 checkpoint missing!"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def load_grac_model(path):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model = GRACLM(GRAC_V04_CONFIG).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model

original_model = load_grac_model(original_path)
phase3_model = load_grac_model(phase3_path)

print("Both GRAC models loaded successfully!")
print("Device:", device)


@torch.inference_mode()
def generate_grac_test(
    model,
    prompt,
    max_new_tokens=120,
    temperature=0.8,
    top_k=40,
    repetition_penalty=1.15,
    seed=42
):
    torch.manual_seed(seed)

    input_ids = tokenizer.encode(prompt).ids

    if not input_ids:
        raise ValueError("Prompt produced no tokens.")

    tokens = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )

    eos_id = tokenizer.token_to_id("<eos>")
    context_length = GRAC_V04_CONFIG["context_length"]

    for _ in range(max_new_tokens):
        context = tokens[:, -context_length:]

        # Compatible with GRACLM's forward interface
        logits, _ = model(context)

        next_logits = logits[:, -1, :].float().clone()

        # Penalize recently repeated tokens
        recent_tokens = tokens[0, -80:].tolist()

        for token_id in set(recent_tokens):
            if token_id == eos_id:
                continue

            score = next_logits[0, token_id]

            if score > 0:
                next_logits[0, token_id] = (
                    score / repetition_penalty
                )
            else:
                next_logits[0, token_id] = (
                    score * repetition_penalty
                )

        next_logits /= temperature

        k = min(top_k, next_logits.shape[-1])

        top_values, top_indices = torch.topk(
            next_logits,
            k=k,
            dim=-1
        )

        probabilities = torch.softmax(
            top_values,
            dim=-1
        )

        sampled_index = torch.multinomial(
            probabilities,
            num_samples=1
        )

        next_token = top_indices.gather(
            -1,
            sampled_index
        )

        tokens = torch.cat(
            [tokens, next_token],
            dim=1
        )

        if (
            eos_id is not None
            and next_token.item() == eos_id
        ):
            break

    return tokenizer.decode(
        tokens[0].tolist(),
        skip_special_tokens=True
    )


test_prompts = [
    "Artificial intelligence is",
    "The government announced",
    "Scientists discovered",
    "The history of India",
    "The future of technology"
]

for i, prompt in enumerate(test_prompts, 1):
    print("\n" + "=" * 65)
    print(f"TEST {i}: {prompt}")
    print("=" * 65)

    print("\nORIGINAL GRAC:")
    print(
        generate_grac_test(
            original_model,
            prompt,
            seed=100 + i
        )
    )

    print("\nPHASE 3 GRAC:")
    print(
        generate_grac_test(
            phase3_model,
            prompt,
            seed=100 + i
        )
    )

print("\n" + "=" * 65)
print("CELL 53 COMPLETE!")
print("=" * 65)


GRAC-LM v0.4 — GENERATION SHOWDOWN
Both GRAC models loaded successfully!
Device: cuda

TEST 1: Artificial intelligence is

ORIGINAL GRAC:
Artificial intelligence is often inferior to what he called " Sons of Man " or " Covenant " . In the game , it moves directly to the " Locket Fat " mode known as " X " and then the protagonist with VHS ; it appears that all his characters are the same as that of HBO : Chains of Man " The Possom of Man " . It also uses a variety of weapons to defeat the player 's power , such as a grid map ( STP ) or an alignment ( SCP @-@ RFM

PHASE 3 GRAC:
Artificial intelligence is often not justified by an enemy . The two are the subjects in the game , but have been called " Cosmody " and " The Flying " because they may be responsible for the game 's events , and the first three dozen missions are named after him . = = Casting = = While shooting for Sakurai ( 1993 ) , a former student who has played basketball for Sakurai is cast as an actor who appears in his own

In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — INTERACTIVE TEST")
print("=" * 65)

while True:
    prompt = input("\nYOU (or type exit): ").strip()

    if prompt.lower() in ["exit", "quit", "stop"]:
        print("GRAC testing ended.")
        break

    if not prompt:
        continue

    response = generate_grac_test(
        phase3_model,
        prompt,
        max_new_tokens=150,
        temperature=0.75,
        top_k=40,
        repetition_penalty=1.15
    )

    print("\nGRAC:")
    print(response)


GRAC-LM v0.4 — INTERACTIVE TEST

YOU (or type exit): hello

GRAC:
hello , the " Happy Talker " for the episode . During the finale of the show 's premiere , a writer from the show was given to Fringe : The Doctor Who episode of Star Trek on February 4 , 1992 at the ABC network in Los Angeles . In August 1994 a group of staff members were made up of The Doctor Who and The Sleepers , The Ride to Earth ( now the Season One ) , where they were cast as episodes of Parents and the X-980 networks of their own networks . In October 1995 he was signed by the show 's creator for Star Trek : The Unfortunately Star Trek : The Wizard Show . He initially wanted to

YOU (or type exit): do u know me?

GRAC:
do u know me? " . However , he stated his father had told him that the word was derived from his genealogical words . He said that it is probably not afraid to mean that " he has an old man , when they are born and have a strange sense of nature " with which one described as " a man " in a similar 

In [ ]:

import json
import random
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 4 INSTRUCTION DATASET")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
data_dir = phase4_dir / "data"

data_dir.mkdir(parents=True, exist_ok=True)

# Hand-authored examples: no external dataset or license dependency.
examples = [
    ("Hello!", "Hello! How can I help you today?"),
    ("Hi!", "Hi! What would you like to talk about?"),
    ("Good morning.", "Good morning! I hope you have a good day."),
    ("How are you?", "I'm ready to help. What can I do for you?"),
    ("What is your name?", "My name is GRAC."),
    ("Who created you?", "I am GRAC, an experimental AI model developed by GRC."),
    ("Are you a human?", "No. I am an artificial intelligence model."),
    ("What can you do?", "I can generate text and help with simple questions, although I am still learning."),
    ("Do you know me?", "I only know what you tell me in our conversation."),
    ("Thank you.", "You're welcome!"),
    ("Goodbye.", "Goodbye! Have a great day."),
    ("What is artificial intelligence?", "Artificial intelligence is technology that enables computers to perform tasks associated with human intelligence."),
    ("What is machine learning?", "Machine learning is a field of AI where systems learn patterns from data."),
    ("What is a language model?", "A language model predicts and generates sequences of text."),
    ("What is a computer?", "A computer is an electronic device that processes data according to instructions."),
    ("What is a CPU?", "A CPU is the central processing unit that executes instructions."),
    ("What is RAM?", "RAM is temporary memory used by a computer while programs are running."),
    ("What is Python?", "Python is a programming language used for automation, software development, and artificial intelligence."),
    ("What is electricity?", "Electricity involves the presence and movement of electric charge."),
    ("State Ohm's law.", "Ohm's law states that current through a conductor is directly proportional to voltage across it, provided temperature and other physical conditions remain constant."),
    ("What is photosynthesis?", "Photosynthesis is the process by which green plants use sunlight to make glucose from carbon dioxide and water."),
    ("What is gravity?", "Gravity is the attractive interaction between objects with mass."),
    ("What is puberty?", "Puberty is the stage of development when a person's body undergoes physical and hormonal changes toward sexual maturity."),
    ("What is the capital of India?", "New Delhi is the capital of India."),
    ("What is the capital of France?", "Paris is the capital of France."),
    ("What is 2 + 2?", "2 + 2 = 4."),
    ("What is 10 multiplied by 5?", "10 multiplied by 5 is 50."),
    ("What is a prime number?", "A prime number is a whole number greater than 1 with exactly two positive factors: 1 and itself."),
    ("Explain simply.", "Of course. Tell me which topic you would like explained."),
    ("I do not understand.", "Tell me which part is confusing, and I will try to explain it more clearly."),
    ("Can you help me study?", "Yes. Tell me your subject and topic, and we can practise together."),
    ("Tell me a joke.", "Why did the computer go to the doctor? It had a virus!"),
    ("What should you do if you are unsure?", "I should say that I am unsure rather than invent an answer."),
    ("Can you make mistakes?", "Yes. My responses may contain mistakes, so important information should be checked."),
    ("What does AI stand for?", "AI stands for artificial intelligence."),
    ("What is the Sun?", "The Sun is the star at the centre of our solar system."),
    ("What is water made of?", "Water molecules contain two hydrogen atoms and one oxygen atom."),
    ("What is a robot?", "A robot is a machine designed to carry out physical tasks automatically or under human control."),
    ("What is a Raspberry Pi?", "A Raspberry Pi is a small single-board computer used for computing, electronics, and educational projects."),
    ("What is the difference between hardware and software?", "Hardware is the physical equipment of a computer. Software consists of programs and instructions."),
]

random.Random(2026).shuffle(examples)

split = int(len(examples) * 0.8)

train_examples = examples[:split]
val_examples = examples[split:]

def format_example(question, answer):
    return {
        "instruction": question,
        "response": answer,
        "text": (
            "User: " + question + "\n"
            "Assistant: " + answer
        )
    }

for filename, rows in [
    ("instruction_train.jsonl", train_examples),
    ("instruction_validation.jsonl", val_examples)
]:
    with open(data_dir / filename, "w", encoding="utf-8") as f:
        for question, answer in rows:
            f.write(
                json.dumps(
                    format_example(question, answer),
                    ensure_ascii=False
                ) + "\n"
            )

print("Total examples:", len(examples))
print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))

print("\nEXAMPLE:")
print(json.dumps(
    format_example(*train_examples[0]),
    indent=2
))

print("\nSaved to:", data_dir)
print("CELL 55 COMPLETE!")


GRAC-LM v0.4 — PHASE 4 INSTRUCTION DATASET
Total examples: 40
Training examples: 32
Validation examples: 8

EXAMPLE:
{
  "instruction": "What is the capital of France?",
  "response": "Paris is the capital of France.",
  "text": "User: What is the capital of France?\nAssistant: Paris is the capital of France."
}

Saved to: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4/data
CELL 55 COMPLETE!


In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — INSTRUCTION TOKENIZER TEST")
print("=" * 65)

unk_id = tokenizer.token_to_id("<unk>")

def format_grac_conversation(question, answer):
    return (
        "User: " + " ".join(question.split())
        + " Assistant: " + " ".join(answer.split())
    )

test_pairs = [
    ("Hello!", "Hello! How can I help you?"),
    ("What is puberty?", "Puberty is a stage of human development."),
    ("What is 2 + 2?", "2 + 2 = 4.")
]

for question, answer in test_pairs:
    text = format_grac_conversation(question, answer)
    ids = tokenizer.encode(text).ids

    print("\nTEXT:", text)
    print("TOKENS:", len(ids))
    print("UNKNOWN TOKENS:", ids.count(unk_id))
    print("DECODED:", tokenizer.decode(ids))

    assert ids.count(unk_id) == 0, (
        "Unexpected unknown token in conversation."
    )

print("\nCELL 56 PASSED!")


GRAC-LM v0.4 — INSTRUCTION TOKENIZER TEST

TEXT: User: Hello! Assistant: Hello! How can I help you?
TOKENS: 20
UNKNOWN TOKENS: 0
DECODED: User: Hello! Assistant: Hello! How can I help you?

TEXT: User: What is puberty? Assistant: Puberty is a stage of human development.
TOKENS: 25
UNKNOWN TOKENS: 0
DECODED: User: What is puberty? Assistant: Puberty is a stage of human development.

TEXT: User: What is 2 + 2? Assistant: 2 + 2 = 4.
TOKENS: 20
UNKNOWN TOKENS: 0
DECODED: User: What is 2 + 2? Assistant: 2 + 2 = 4.

CELL 56 PASSED!


In [ ]:

import json
import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — RESPONSE-ONLY DATA PREPARATION")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
data_dir = phase4_dir / "data"

context_length = GRAC_V04_CONFIG["context_length"]

eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")

assert eos_id is not None
assert unk_id is not None

def prepare_instruction_example(question, answer):
    question = " ".join(question.split())
    answer = " ".join(answer.split())

    prefix = f"User: {question} Assistant:"
    full_text = prefix + " " + answer

    # Encode prefix and full sequence separately.
    prefix_ids = tokenizer.encode(prefix).ids
    full_ids = tokenizer.encode(full_text).ids

    # Tokenizers may merge across the prefix/answer boundary.
    # Only mask the prefix if it is an exact token prefix.
    if full_ids[:len(prefix_ids)] != prefix_ids:
        # Re-encode with an explicit, consistently tokenized
        # boundary to preserve the exact response mask.
        prefix = f"User: {question} Assistant: "
        answer_ids = tokenizer.encode(answer).ids
        prefix_ids = tokenizer.encode(prefix).ids
        full_ids = prefix_ids + answer_ids

    full_ids = full_ids + [eos_id]

    if unk_id in full_ids:
        raise ValueError("Unknown token in instruction example.")

    if len(full_ids) > context_length + 1:
        raise ValueError("Instruction example exceeds context.")

    # Next-token prediction:
    # x[t] predicts y[t] = full_ids[t+1].
    x = full_ids[:-1]
    y = full_ids[1:]

    # Ignore targets belonging to the user/prefix.
    # The target at position len(prefix_ids)-1 is the
    # first answer token, so retain it.
    ignore_count = max(0, len(prefix_ids) - 1)
    y[:ignore_count] = [-100] * ignore_count

    assert any(token != -100 for token in y)

    return {
        "input_ids": x,
        "labels": y
    }

def load_instruction_examples(path):
    prepared = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)

            prepared.append(
                prepare_instruction_example(
                    item["instruction"],
                    item["response"]
                )
            )

    return prepared

train_instruction_data = load_instruction_examples(
    data_dir / "instruction_train.jsonl"
)

val_instruction_data = load_instruction_examples(
    data_dir / "instruction_validation.jsonl"
)

print("Training examples:", len(train_instruction_data))
print("Validation examples:", len(val_instruction_data))

sample = train_instruction_data[0]

print("\nINPUT TOKENS:", len(sample["input_ids"]))
print(
    "TRAINED TARGET TOKENS:",
    sum(t != -100 for t in sample["labels"])
)

print("\nDECODED INPUT:")
print(tokenizer.decode(sample["input_ids"]))

print("\nCELL 57 COMPLETE!")


GRAC-LM v0.4 — RESPONSE-ONLY DATA PREPARATION
Training examples: 32
Validation examples: 8

INPUT TOKENS: 22
TRAINED TARGET TOKENS: 8

DECODED INPUT:
User: What is the capital of France? Assistant: Paris is the capital of France.

CELL 57 COMPLETE!


In [ ]:

import random
import torch
import torch.nn.functional as F

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 4 INSTRUCTION FINE-TUNING")
print("=" * 65)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable GPU runtime."

phase3_checkpoint = (
    base_dir / "phase3/GRAC-LM-v0.4-phase3-step500.pt"
)

assert phase3_checkpoint.exists()

checkpoint = torch.load(
    phase3_checkpoint,
    map_location="cpu",
    weights_only=False
)

instruction_model = GRACLM(GRAC_V04_CONFIG).to(device)
instruction_model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer4 = torch.optim.AdamW(
    instruction_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler4 = torch.amp.GradScaler("cuda")

pad_id = eos_id

def make_instruction_batch(dataset, batch_size=4):
    samples = random.choices(dataset, k=batch_size)

    max_length = max(len(s["input_ids"]) for s in samples)

    x_batch = []
    y_batch = []

    for sample in samples:
        x = sample["input_ids"]
        y = sample["labels"]

        pad_length = max_length - len(x)

        x_batch.append(x + [pad_id] * pad_length)
        y_batch.append(y + [-100] * pad_length)

    return (
        torch.tensor(x_batch, dtype=torch.long, device=device),
        torch.tensor(y_batch, dtype=torch.long, device=device)
    )

def instruction_loss(model, x, y):
    # Use raw logits and compute cross-entropy ourselves
    # to support labels masked with -100.
    logits, _ = model(x)

    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)).float(),
        y.reshape(-1),
        ignore_index=-100
    )

@torch.no_grad()
def evaluate_instruction_model():
    instruction_model.eval()

    total_loss = 0.0
    total_tokens = 0

    for sample in val_instruction_data:
        x = torch.tensor(
            [sample["input_ids"]],
            dtype=torch.long,
            device=device
        )
        y = torch.tensor(
            [sample["labels"]],
            dtype=torch.long,
            device=device
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss = instruction_loss(instruction_model, x, y)

        count = int((y != -100).sum().item())
        total_loss += loss.item() * count
        total_tokens += count

    instruction_model.train()
    return total_loss / total_tokens

phase4_dir.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")

steps = 200
batch_size = 4

print("Training examples:", len(train_instruction_data))
print("Validation examples:", len(val_instruction_data))
print("Training steps:", steps)

for step in range(1, steps + 1):
    instruction_model.train()

    x, y = make_instruction_batch(
        train_instruction_data,
        batch_size=batch_size
    )

    optimizer4.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        loss = instruction_loss(instruction_model, x, y)

    scaler4.scale(loss).backward()
    scaler4.unscale_(optimizer4)

    torch.nn.utils.clip_grad_norm_(
        instruction_model.parameters(),
        1.0
    )

    scaler4.step(optimizer4)
    scaler4.update()

    if step % 20 == 0:
        val_loss = evaluate_instruction_model()

        print(
            f"Step {step:3d}/{steps} | "
            f"Train loss: {loss.item():.4f} | "
            f"Validation loss: {val_loss:.4f}",
            flush=True
        )

        state = {
            "model_state_dict": instruction_model.state_dict(),
            "step": step,
            "validation_loss": val_loss,
            "source_checkpoint": str(phase3_checkpoint),
            "config": GRAC_V04_CONFIG
        }

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                state,
                phase4_dir / "GRAC-LM-v0.4-phase4-best.pt"
            )

        if step == steps:
            torch.save(
                state,
                phase4_dir / "GRAC-LM-v0.4-phase4-final.pt"
            )

print("\nPHASE 4 TRAINING COMPLETE!")
print("Best instruction validation loss:", best_val_loss)
print("Original checkpoints remain untouched.")


GRAC-LM v0.4 — PHASE 4 INSTRUCTION FINE-TUNING
Training examples: 32
Validation examples: 8
Training steps: 200
Step  20/200 | Train loss: 3.0722 | Validation loss: 3.7171
Step  40/200 | Train loss: 2.5047 | Validation loss: 3.5131
Step  60/200 | Train loss: 1.1653 | Validation loss: 3.4961
Step  80/200 | Train loss: 0.8666 | Validation loss: 3.5468
Step 100/200 | Train loss: 0.7502 | Validation loss: 3.6165
Step 120/200 | Train loss: 0.4471 | Validation loss: 3.6499
Step 140/200 | Train loss: 0.1824 | Validation loss: 3.7023
Step 160/200 | Train loss: 0.2437 | Validation loss: 3.7408
Step 180/200 | Train loss: 0.0873 | Validation loss: 3.8164
Step 200/200 | Train loss: 0.0677 | Validation loss: 3.8955

PHASE 4 TRAINING COMPLETE!
Best instruction validation loss: 3.4960991470781093
Original checkpoints remain untouched.


In [ ]:

print("=" * 65)
print("GRAC-LM v0.4 — FIRST INSTRUCTION TEST")
print("=" * 65)

best_path = (
    phase4_dir / "GRAC-LM-v0.4-phase4-best.pt"
)

assert best_path.exists()

instruction_test_model = load_grac_model(best_path)

questions = [
    "Hello!",
    "What is your name?",
    "What is artificial intelligence?",
    "What is puberty?",
    "What is 2 + 2?",
    "Do you know me?"
]

for question in questions:
    prompt = f"User: {question} Assistant:"

    output = generate_grac_test(
        instruction_test_model,
        prompt,
        max_new_tokens=80,
        temperature=0.5,
        top_k=20,
        repetition_penalty=1.1,
        seed=42
    )

    print("\n" + "-" * 55)
    print("USER:", question)
    print("GRAC:", output[len(prompt):].strip())

print("\nCELL 59 COMPLETE!")


GRAC-LM v0.4 — FIRST INSTRUCTION TEST

-------------------------------------------------------
USER: Hello!
GRAC: Hello A.

-------------------------------------------------------
USER: What is your name?
GRAC: My name is the word " cognitionary " meaning " cognitionary " is a word used for reference purposes.

-------------------------------------------------------
USER: What is artificial intelligence?
GRAC: Artificial intelligence.

-------------------------------------------------------
USER: What is puberty?
GRAC: puberty has a central processing unit that is a system that allows the ability of activated computing, and educational programs to help ensure healthy education.

-------------------------------------------------------
USER: What is 2 + 2?
GRAC: 2 + 3 = 4.

-------------------------------------------------------
USER: Do you know me?
GRAC: I am aforated by what you do with your own message.

CELL 59 COMPLETE!


In [ ]:

!pip -q install datasets

from datasets import load_dataset
from collections import Counter

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 4 DATASET EXPANSION")
print("=" * 65)

dolly = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

print("Total examples:", len(dolly))
print("Columns:", dolly.column_names)

categories = Counter(dolly["category"])

print("\nCATEGORIES:")
for category, count in categories.most_common():
    print(f"{category:30s} {count:,}")

print("\nFIRST EXAMPLE:")
example = dolly[0]

print("Instruction:", example["instruction"])
print("Response:", example["response"])
print("Context:", example["context"])

print("\nCELL 60 COMPLETE!")


GRAC-LM v0.4 — PHASE 4 DATASET EXPANSION


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Total examples: 15011
Columns: ['instruction', 'context', 'response', 'category']

CATEGORIES:
open_qa                        3,742
general_qa                     2,191
classification                 2,136
closed_qa                      1,773
brainstorming                  1,766
information_extraction         1,506
summarization                  1,188
creative_writing               709

FIRST EXAMPLE:
Instruction: When did Virgin Australia start operating?
Response: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.
Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has s

In [ ]:

from pathlib import Path

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
expanded_dir = phase4_dir / "expanded_data"

print("=" * 65)
print("GRAC-LM v0.4 — DIRECTORY REPAIR")
print("=" * 65)

print("Expanded directory:", expanded_dir)

train_file = expanded_dir / "train.jsonl"
val_file = expanded_dir / "validation.jsonl"

print("Training file exists:", train_file.exists())
print("Validation file exists:", val_file.exists())

if train_file.exists() and val_file.exists():
    print("\nREPAIR SUCCESSFUL! Rerun Cell 62.")
else:
    print("\nFILES MISSING! Run Cell 61 first.")


GRAC-LM v0.4 — DIRECTORY REPAIR
Expanded directory: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4/expanded_data
Training file exists: False
Validation file exists: False

FILES MISSING! Run Cell 61 first.


In [ ]:

import json
import random
from pathlib import Path
from collections import Counter

print("=" * 65)
print("GRAC-LM v0.4 — INSTRUCTION DATA CLEANING")
print("=" * 65)

# Google Drive directories
base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
expanded_dir = phase4_dir / "expanded_data"

expanded_dir.mkdir(parents=True, exist_ok=True)

# Model settings
max_context = GRAC_V04_CONFIG["context_length"]
unk_id = tokenizer.token_to_id("<unk>")
eos_id = tokenizer.token_to_id("<eos>")

assert eos_id is not None
assert unk_id is not None
assert "dolly" in globals(), "Run Cell 60 first!"

def normalize_text(text):
    return " ".join(text.split()).strip()

accepted = []
rejected = Counter()

for item in dolly:
    instruction = normalize_text(item["instruction"])
    response = normalize_text(item["response"])
    context = normalize_text(item["context"] or "")

    if not instruction or not response:
        rejected["empty"] += 1
        continue

    # Skip examples requiring additional context
    if context:
        rejected["requires_context"] += 1
        continue

    # Match the format used for GRAC training
    prefix = f"User: {instruction} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    response_ids = tokenizer.encode(response).ids

    if unk_id in prefix_ids or unk_id in response_ids:
        rejected["unknown_token"] += 1
        continue

    # Ensure the complete sequence fits the model
    if len(prefix_ids) + len(response_ids) + 1 > max_context:
        rejected["too_long"] += 1
        continue

    accepted.append({
        "instruction": instruction,
        "response": response,
        "category": item["category"]
    })

# Remove duplicate question-answer pairs
unique = {}

for item in accepted:
    key = (
        item["instruction"].casefold(),
        item["response"].casefold()
    )
    unique[key] = item

accepted = list(unique.values())

# Deterministic train/validation split
random.Random(2026).shuffle(accepted)

split = int(len(accepted) * 0.9)

train_rows = accepted[:split]
val_rows = accepted[split:]

# Save datasets to Google Drive
for filename, rows in [
    ("train.jsonl", train_rows),
    ("validation.jsonl", val_rows)
]:
    output_path = expanded_dir / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for item in rows:
            f.write(
                json.dumps(item, ensure_ascii=False) + "\n"
            )

print("\nDATASET PREPARATION COMPLETE!")
print("Original Dolly examples:", len(dolly))
print("Accepted examples:", len(accepted))
print("Training examples:", len(train_rows))
print("Validation examples:", len(val_rows))

print("\nREJECTION REPORT:")
for reason, count in rejected.items():
    print(f"{reason}: {count:,}")

print("\nSAVED FILES:")
print(expanded_dir / "train.jsonl")
print(expanded_dir / "validation.jsonl")

print("\nCELL 61 COMPLETE!")


GRAC-LM v0.4 — INSTRUCTION DATA CLEANING

DATASET PREPARATION COMPLETE!
Original Dolly examples: 15011
Accepted examples: 10307
Training examples: 9276
Validation examples: 1031

REJECTION REPORT:
requires_context: 4,467
too_long: 227

SAVED FILES:
/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4/expanded_data/train.jsonl
/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4/expanded_data/validation.jsonl

CELL 61 COMPLETE!


In [ ]:

import json
import torch
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — EXPANDED INSTRUCTION PREPARATION")
print("=" * 65)

# Restore paths so this cell works independently
base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
expanded_dir = phase4_dir / "expanded_data"

train_path = expanded_dir / "train.jsonl"
val_path = expanded_dir / "validation.jsonl"

assert train_path.exists(), "Run Cell 61 first!"
assert val_path.exists(), "Validation file missing!"

# Model and tokenizer settings
max_context = GRAC_V04_CONFIG["context_length"]

eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")

assert eos_id is not None
assert unk_id is not None

def prepare_expanded_example(item):
    instruction = " ".join(item["instruction"].split())
    response = " ".join(item["response"].split())

    # Consistent conversation format
    prefix = f"User: {instruction} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(response).ids

    # Concatenate independently encoded segments
    full_ids = prefix_ids + answer_ids + [eos_id]

    if len(full_ids) > max_context:
        raise ValueError("Example exceeds context limit.")

    if unk_id in full_ids:
        raise ValueError("Unknown token detected.")

    # Input and next-token targets
    x = full_ids[:-1]
    y = full_ids[1:]

    # Mask all question/prefix targets
    # Keep the first answer token and EOS trainable
    mask_length = len(prefix_ids) - 1

    y[:mask_length] = [-100] * mask_length

    assert len(x) == len(y)
    assert any(token != -100 for token in y)

    return {
        "input_ids": x,
        "labels": y
    }

def load_expanded_examples(path):
    prepared = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            item = json.loads(line)

            try:
                sample = prepare_expanded_example(item)
                prepared.append(sample)
            except ValueError as error:
                print(
                    f"Skipping line {line_number}: {error}"
                )

    return prepared

# Load training and validation data
expanded_train = load_expanded_examples(train_path)
expanded_val = load_expanded_examples(val_path)

print("\nDATASET LOADED SUCCESSFULLY")
print("Training examples:", len(expanded_train))
print("Validation examples:", len(expanded_val))

assert len(expanded_train) > 0
assert len(expanded_val) > 0

# Inspect one example
sample = expanded_train[0]

trained_targets = [
    token
    for token in sample["labels"]
    if token != -100
]

print("\nSAMPLE INPUT:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )
)

print("\nTRAINED ANSWER TOKENS:")
print(
    tokenizer.decode(
        trained_targets,
        skip_special_tokens=False
    )
)

print("\nTOKEN STATISTICS:")
print("Input length:", len(sample["input_ids"]))
print("Answer target tokens:", len(trained_targets))
print("Ignored targets:", sample["labels"].count(-100))

assert trained_targets[-1] == eos_id, (
    "Answer must end with EOS."
)

print("\nCELL 62 COMPLETE — READY FOR FINE-TUNING!")


GRAC-LM v0.4 — EXPANDED INSTRUCTION PREPARATION

DATASET LOADED SUCCESSFULLY
Training examples: 9276
Validation examples: 1031

SAMPLE INPUT:
User: What does ROYGBIV stand for? Assistant: ROYGBIV stands for the colors of the rainbow: Red, Orange, Yellow, Green, Blue, Indigo, Violet

TRAINED ANSWER TOKENS:
ROYGBIV stands for the colors of the rainbow: Red, Orange, Yellow, Green, Blue, Indigo, Violet<eos>

TOKEN STATISTICS:
Input length: 56
Answer target tokens: 37
Ignored targets: 19

CELL 62 COMPLETE — READY FOR FINE-TUNING!


In [ ]:

import torch
import torch.nn.functional as F
import numpy as np
import random
import json
import math
import gc
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — PHASE 4 EXPANDED FINE-TUNING")
print("=" * 65)

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4_dir = base_dir / "phase4"
phase4_dir.mkdir(parents=True, exist_ok=True)

source_checkpoint = (
    base_dir / "phase3/GRAC-LM-v0.4-phase3-step500.pt"
)

assert source_checkpoint.exists()
assert len(expanded_train) > 0
assert len(expanded_val) > 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable Colab GPU runtime!"

FT_CONFIG = {
    "max_steps": 1000,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5,
    "weight_decay": 0.01,
    "eval_every": 100,
    "log_every": 25,
    "max_grad_norm": 1.0,
    "patience": 4,
    "seed": 2026
}

random.seed(FT_CONFIG["seed"])
np.random.seed(FT_CONFIG["seed"])
torch.manual_seed(FT_CONFIG["seed"])
torch.cuda.manual_seed_all(FT_CONFIG["seed"])

print("Training examples:", len(expanded_train))
print("Validation examples:", len(expanded_val))
print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 2. LOAD PHASE 3 MODEL
# --------------------------------------------------

checkpoint = torch.load(
    source_checkpoint,
    map_location="cpu",
    weights_only=False
)

ft_model = GRACLM(GRAC_V04_CONFIG).to(device)
ft_model.load_state_dict(checkpoint["model_state_dict"])

del checkpoint
gc.collect()

ft_model.train()

optimizer_ft = torch.optim.AdamW(
    ft_model.parameters(),
    lr=FT_CONFIG["learning_rate"],
    weight_decay=FT_CONFIG["weight_decay"]
)

scaler_ft = torch.amp.GradScaler("cuda")

print(
    "Parameters:",
    f"{sum(p.numel() for p in ft_model.parameters()):,}"
)

# --------------------------------------------------
# 3. BATCHING WITH RESPONSE-ONLY MASKS
# --------------------------------------------------

pad_id = tokenizer.token_to_id("<eos>")

def make_ft_batch(dataset, batch_size, rng):
    samples = rng.choices(dataset, k=batch_size)

    max_len = max(len(s["input_ids"]) for s in samples)

    x_rows = []
    y_rows = []

    for sample in samples:
        x = sample["input_ids"]
        y = sample["labels"]

        padding = max_len - len(x)

        x_rows.append(x + [pad_id] * padding)
        y_rows.append(y + [-100] * padding)

    x_tensor = torch.tensor(
        x_rows, dtype=torch.long, device=device
    )
    y_tensor = torch.tensor(
        y_rows, dtype=torch.long, device=device
    )

    return x_tensor, y_tensor

def ft_loss(model, x, y):
    logits, _ = model(x)

    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]).float(),
        y.reshape(-1),
        ignore_index=-100
    )

train_rng = random.Random(2026)

# --------------------------------------------------
# 4. VALIDATION
# --------------------------------------------------

@torch.no_grad()
def evaluate_ft_model():
    ft_model.eval()

    total_loss = 0.0
    total_tokens = 0

    for start in range(0, len(expanded_val), 4):
        batch_samples = expanded_val[start:start + 4]

        max_len = max(
            len(s["input_ids"]) for s in batch_samples
        )

        x_rows = []
        y_rows = []

        for sample in batch_samples:
            pad = max_len - len(sample["input_ids"])

            x_rows.append(
                sample["input_ids"] + [pad_id] * pad
            )
            y_rows.append(
                sample["labels"] + [-100] * pad
            )

        x = torch.tensor(
            x_rows, dtype=torch.long, device=device
        )
        y = torch.tensor(
            y_rows, dtype=torch.long, device=device
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss = ft_loss(ft_model, x, y)

        count = int((y != -100).sum().item())

        total_loss += loss.item() * count
        total_tokens += count

    ft_model.train()

    return total_loss / total_tokens

# --------------------------------------------------
# 5. TRAINING
# --------------------------------------------------

best_val_loss = float("inf")
bad_evaluations = 0

history = []

print("\nTRAINING STARTED!\n")

for step in range(1, FT_CONFIG["max_steps"] + 1):
    ft_model.train()
    optimizer_ft.zero_grad(set_to_none=True)

    running_loss = 0.0

    for _ in range(
        FT_CONFIG["gradient_accumulation_steps"]
    ):
        x, y = make_ft_batch(
            expanded_train,
            FT_CONFIG["micro_batch_size"],
            train_rng
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss = ft_loss(ft_model, x, y)

        running_loss += loss.item()

        scaler_ft.scale(
            loss / FT_CONFIG["gradient_accumulation_steps"]
        ).backward()

    scaler_ft.unscale_(optimizer_ft)

    torch.nn.utils.clip_grad_norm_(
        ft_model.parameters(),
        FT_CONFIG["max_grad_norm"]
    )

    scaler_ft.step(optimizer_ft)
    scaler_ft.update()

    if step % FT_CONFIG["log_every"] == 0:
        print(
            f"Step {step:4d}/{FT_CONFIG['max_steps']} | "
            f"Train loss: "
            f"{running_loss / FT_CONFIG['gradient_accumulation_steps']:.4f}",
            flush=True
        )

    if step % FT_CONFIG["eval_every"] == 0:
        val_loss = evaluate_ft_model()

        history.append({
            "step": step,
            "validation_loss": val_loss
        })

        print(
            f"\nVALIDATION — STEP {step} | "
            f"Loss: {val_loss:.4f}",
            flush=True
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            bad_evaluations = 0

            torch.save(
                {
                    "model_state_dict": ft_model.state_dict(),
                    "step": step,
                    "validation_loss": val_loss,
                    "config": GRAC_V04_CONFIG,
                    "source_checkpoint": str(source_checkpoint)
                },
                phase4_dir / "GRAC-LM-v0.4-expanded-best.pt"
            )

            print("NEW BEST CHECKPOINT SAVED!")

        else:
            bad_evaluations += 1

        with open(
            phase4_dir / "expanded_training_history.json",
            "w"
        ) as f:
            json.dump(history, f, indent=2)

        if bad_evaluations >= FT_CONFIG["patience"]:
            print("\nEARLY STOPPING TRIGGERED!")
            break

# --------------------------------------------------
# 6. SAVE FINAL CHECKPOINT
# --------------------------------------------------

torch.save(
    {
        "model_state_dict": ft_model.state_dict(),
        "step": step,
        "best_validation_loss": best_val_loss,
        "config": GRAC_V04_CONFIG
    },
    phase4_dir / "GRAC-LM-v0.4-expanded-final.pt"
)

print("\n" + "=" * 65)
print("GRAC-LM v0.4 — EXPANDED TRAINING COMPLETE")
print("=" * 65)

print("Final step:", step)
print("Best validation loss:", f"{best_val_loss:.4f}")
print("Best checkpoint saved in:", phase4_dir)
print("Original and Phase 3 checkpoints remain untouched.")


GRAC-LM v0.4 — PHASE 4 EXPANDED FINE-TUNING
Training examples: 9276
Validation examples: 1031
GPU: Tesla T4
Parameters: 54,684,160

TRAINING STARTED!

Step   25/1000 | Train loss: 4.6891
Step   50/1000 | Train loss: 4.3670
Step   75/1000 | Train loss: 4.3032
Step  100/1000 | Train loss: 4.2981

VALIDATION — STEP 100 | Loss: 4.0826
NEW BEST CHECKPOINT SAVED!
Step  125/1000 | Train loss: 4.0514
Step  150/1000 | Train loss: 4.2245
Step  175/1000 | Train loss: 3.9991
Step  200/1000 | Train loss: 4.1926

VALIDATION — STEP 200 | Loss: 3.9535
NEW BEST CHECKPOINT SAVED!
Step  225/1000 | Train loss: 3.6572
Step  250/1000 | Train loss: 3.9993
Step  275/1000 | Train loss: 3.3478
Step  300/1000 | Train loss: 4.0141

VALIDATION — STEP 300 | Loss: 3.8825
NEW BEST CHECKPOINT SAVED!
Step  325/1000 | Train loss: 3.9686
Step  350/1000 | Train loss: 3.4545
Step  375/1000 | Train loss: 3.4897
Step  400/1000 | Train loss: 3.5544

VALIDATION — STEP 400 | Loss: 3.8317
NEW BEST CHECKPOINT SAVED!
Step  425/100

In [ ]:

import json
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — EXPANDED TRAINING RESULTS")
print("=" * 65)

history_path = (
    phase4_dir / "expanded_training_history.json"
)

assert history_path.exists(), "Run Cell 63 first."

with open(history_path, "r") as f:
    history = json.load(f)

for entry in history:
    print(
        f"Step {entry['step']:4d} | "
        f"Validation loss: {entry['validation_loss']:.4f}"
    )

best = min(
    history,
    key=lambda item: item["validation_loss"]
)

print("\nBEST STEP:", best["step"])
print("BEST LOSS:", f"{best['validation_loss']:.4f}")

print("\nCELL 64 COMPLETE!")


GRAC-LM v0.4 — EXPANDED TRAINING RESULTS
Step  100 | Validation loss: 4.0826
Step  200 | Validation loss: 3.9535
Step  300 | Validation loss: 3.8825
Step  400 | Validation loss: 3.8317
Step  500 | Validation loss: 3.7953
Step  600 | Validation loss: 3.7693
Step  700 | Validation loss: 3.7448
Step  800 | Validation loss: 3.7277
Step  900 | Validation loss: 3.7099
Step 1000 | Validation loss: 3.6920

BEST STEP: 1000
BEST LOSS: 3.6920

CELL 64 COMPLETE!


In [ ]:

import torch
import random
from pathlib import Path

print("=" * 65)
print("GRAC-LM v0.4 — CONVERSATION TEST V2")
print("=" * 65)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
best_path = base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"

assert best_path.exists(), "Best checkpoint not found!"
assert "tokenizer" in globals(), "Load your tokenizer first!"
assert "GRACLM" in globals(), "Run the model definition cell first!"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Reuse the training model to avoid allocating
# a second 54.7M-parameter model on the GPU.
if "ft_model" in globals():
    chat_model = ft_model
else:
    chat_model = GRACLM(GRAC_V04_CONFIG).to(device)

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False
)

chat_model.load_state_dict(
    checkpoint["model_state_dict"]
)
chat_model.to(device)
chat_model.eval()

eos_id = tokenizer.token_to_id("<eos>")
context_limit = GRAC_V04_CONFIG["context_length"]

print("Loaded checkpoint:", best_path.name)
print("Best step:", checkpoint.get("step", "unknown"))
print(
    "Validation loss:",
    checkpoint.get("validation_loss", "unknown")
)

@torch.inference_mode()
def ask_grac(
    question,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40
):
    question = " ".join(question.split()).strip()

    # Exactly the same prefix format as training.
    prefix = f"User: {question} Assistant: "
    prompt_ids = tokenizer.encode(prefix).ids

    if len(prompt_ids) >= context_limit:
        return "[Question exceeds context limit]"

    tokens = list(prompt_ids)
    generated = []

    for _ in range(max_new_tokens):
        if len(tokens) >= context_limit:
            break

        x = torch.tensor(
            [tokens],
            dtype=torch.long,
            device=device
        )

        logits, _ = chat_model(x)

        next_logits = logits[0, -1].float()

        # Temperature scaling
        next_logits = next_logits / max(
            temperature, 1e-5
        )

        # Top-k sampling
        k = min(top_k, next_logits.numel())

        values, indices = torch.topk(
            next_logits, k=k
        )

        probs = torch.softmax(values, dim=-1)

        sampled_position = torch.multinomial(
            probs, num_samples=1
        )

        next_id = indices[
            sampled_position
        ].item()

        if next_id == eos_id:
            break

        tokens.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# Fixed seed for reproducible tests
random.seed(2026)
torch.manual_seed(2026)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)

questions = [
    "Hello!",
    "What is your name?",
    "What is artificial intelligence?",
    "What is 2 + 2?",
    "Why is the sky blue?",
    "What is a computer?",
    "What is photosynthesis?",
    "Explain gravity in simple words.",
    "What is the capital of India?",
    "How can I study better?"
]

print("\n" + "=" * 65)
print("GRAC-LM v0.4 — ACTUAL RESPONSES")
print("=" * 65)

for i, question in enumerate(questions, start=1):
    print(f"\n[{i}] USER: {question}")

    answer = ask_grac(question)

    print(
        "GRAC:",
        answer if answer else "[No response]"
    )

print("\n" + "=" * 65)
print("CELL 65 COMPLETE!")
print("=" * 65)


GRAC-LM v0.4 — CONVERSATION TEST V2
Loaded checkpoint: GRAC-LM-v0.4-expanded-best.pt
Best step: 1000
Validation loss: 3.6919780671036886

GRAC-LM v0.4 — ACTUAL RESPONSES

[1] USER: Hello!
GRAC: People are an eponymous group of scientists in the world, like scientists. The group is the first group of scientists to find the first of the genome.

[2] USER: What is your name?
GRAC: If your name is a word that is derived from the word """'s name is derived from the term "" of """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""

[3] USER: What is artificial intelligence?
GRAC: A artificial intelligence is a type of artificial intelligence with a range of organic intelligence and artificial intelligence. The technology of artificial intelligence is based on the nature and mechanical intelligence.

[4] USER: What is 2 + 2?
GRAC: Super 2 is a key step-grammatical device that is made by a series of data that provides data to the application and data from the Internet. The d

In [ ]:

import torch
import torch.nn.functional as F
import math

print("=" * 70)
print("GRAC-LM v0.4 — CELL 66: ANSWER PROBABILITY DIAGNOSTIC")
print("=" * 70)

assert "chat_model" in globals(), "Run Cell 65 first."
assert "tokenizer" in globals(), "Tokenizer is missing."

chat_model.eval()

device = next(chat_model.parameters()).device
context_limit = GRAC_V04_CONFIG["context_length"]

eos_id = tokenizer.token_to_id("<eos>")

# --------------------------------------------------
# 1. DIAGNOSTIC QUESTIONS
# --------------------------------------------------

diagnostic_questions = [
    {
        "question": "What is 2 + 2?",
        "answers": [
            "4.",
            "5.",
            "10."
        ],
        "correct": 0
    },
    {
        "question": "What is the capital of India?",
        "answers": [
            "New Delhi.",
            "Mumbai.",
            "Paris."
        ],
        "correct": 0
    },
    {
        "question": "What is your name?",
        "answers": [
            "My name is GRAC.",
            "My name is Google.",
            "My name is Wikipedia."
        ],
        "correct": 0
    },
    {
        "question": "What is photosynthesis?",
        "answers": [
            "Plants use sunlight to make food.",
            "Plants eat rocks to make food.",
            "Plants produce electricity from metal."
        ],
        "correct": 0
    },
    {
        "question": "Why is the sky blue?",
        "answers": [
            "Because the atmosphere scatters blue light.",
            "Because the sky is made of blue paint.",
            "Because the Sun is blue."
        ],
        "correct": 0
    },
    {
        "question": "What is a computer?",
        "answers": [
            "An electronic device that processes data.",
            "A living animal that eats plants.",
            "A type of natural mountain."
        ],
        "correct": 0
    },
    {
        "question": "What is gravity?",
        "answers": [
            "A force that attracts objects with mass.",
            "A chemical used to clean water.",
            "A language spoken in Europe."
        ],
        "correct": 0
    },
    {
        "question": "What is artificial intelligence?",
        "answers": [
            "Technology that performs tasks associated with human intelligence.",
            "A type of rock found in oceans.",
            "A machine that turns water into gold."
        ],
        "correct": 0
    }
]

# --------------------------------------------------
# 2. SCORE ANSWERS USING TEACHER FORCING
# --------------------------------------------------

@torch.inference_mode()
def score_candidate_answer(question, answer):
    question = " ".join(question.split()).strip()
    answer = " ".join(answer.split()).strip()

    prefix = f"User: {question} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(answer).ids

    if not answer_ids:
        raise ValueError("Empty answer tokenization.")

    # Match the training format.
    full_ids = prefix_ids + answer_ids + [eos_id]

    if len(full_ids) > context_limit:
        raise ValueError("Sequence exceeds context limit.")

    x = torch.tensor(
        [full_ids[:-1]],
        dtype=torch.long,
        device=device
    )

    target = torch.tensor(
        full_ids[1:],
        dtype=torch.long,
        device=device
    )

    logits, _ = chat_model(x)

    log_probs = F.log_softmax(
        logits[0].float(),
        dim=-1
    )

    token_log_probs = log_probs.gather(
        1,
        target[0].unsqueeze(1)
    ).squeeze(1)

    # Ignore question tokens.
    # Score only answer tokens + EOS.
    answer_start = len(prefix_ids) - 1

    answer_log_probs = token_log_probs[
        answer_start:
    ]

    total_log_prob = answer_log_probs.sum().item()

    average_log_prob = answer_log_probs.mean().item()

    return {
        "total_log_prob": total_log_prob,
        "average_log_prob": average_log_prob,
        "token_count": len(answer_log_probs)
    }

# --------------------------------------------------
# 3. RUN DIAGNOSTICS
# --------------------------------------------------

correct_count = 0
diagnostic_results = []

for number, item in enumerate(diagnostic_questions, start=1):
    print("\n" + "-" * 70)
    print(f"QUESTION {number}: {item['question']}")
    print("-" * 70)

    scores = []

    for index, answer in enumerate(item["answers"]):
        result = score_candidate_answer(
            item["question"],
            answer
        )

        scores.append(result["average_log_prob"])

        print(
            f"{chr(65 + index)}. {answer}\n"
            f"   Average log probability: "
            f"{result['average_log_prob']:.4f}\n"
            f"   Answer tokens: {result['token_count']}"
        )

    predicted_index = max(
        range(len(scores)),
        key=lambda i: scores[i]
    )

    correct_index = item["correct"]

    is_correct = predicted_index == correct_index

    if is_correct:
        correct_count += 1

    print(
        "\nGRAC PREFERRED:",
        chr(65 + predicted_index)
    )

    print(
        "CORRECT ANSWER:",
        chr(65 + correct_index)
    )

    print(
        "RESULT:",
        "CORRECT" if is_correct else "INCORRECT"
    )

    diagnostic_results.append({
        "question": item["question"],
        "predicted": predicted_index,
        "correct": correct_index,
        "is_correct": is_correct,
        "scores": scores
    })

# --------------------------------------------------
# 4. FINAL REPORT
# --------------------------------------------------

total_questions = len(diagnostic_questions)

accuracy = 100 * correct_count / total_questions

print("\n" + "=" * 70)
print("GRAC-LM v0.4 — DIAGNOSTIC REPORT")
print("=" * 70)

print("Questions tested:", total_questions)
print("Correct preferences:", correct_count)
print(f"Accuracy: {accuracy:.1f}%")

print("\nINTERPRETATION:")

if accuracy >= 75:
    print(
        "GRAC often prefers the correct answer when "
        "given choices, despite poor free generation."
    )
elif accuracy >= 40:
    print(
        "GRAC shows some ability to rank answers, "
        "but its preferences are unreliable."
    )
else:
    print(
        "GRAC struggles even when correct answers "
        "are supplied. More foundational learning "
        "may be necessary."
    )

print(
    "\nNOTE: This is a small diagnostic, not a "
    "reliable general benchmark."
)

print("\nCELL 66 COMPLETE!")


GRAC-LM v0.4 — CELL 66: ANSWER PROBABILITY DIAGNOSTIC

----------------------------------------------------------------------
QUESTION 1: What is 2 + 2?
----------------------------------------------------------------------


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [ ]:

import torch
import torch.nn.functional as F

@torch.inference_mode()
def score_candidate_answer(question, answer):
    question = " ".join(question.split()).strip()
    answer = " ".join(answer.split()).strip()

    prefix = f"User: {question} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(answer).ids

    if not answer_ids:
        raise ValueError("Answer tokenization is empty.")

    full_ids = prefix_ids + answer_ids + [eos_id]

    if len(full_ids) > context_limit:
        raise ValueError("Sequence exceeds context limit.")

    input_ids = torch.tensor(
        [full_ids[:-1]],
        dtype=torch.long,
        device=device
    )

    targets = torch.tensor(
        full_ids[1:],
        dtype=torch.long,
        device=device
    )

    logits, _ = chat_model(input_ids)

    # logits: [1, sequence_length, vocab_size]
    # targets: [sequence_length]
    log_probs = F.log_softmax(
        logits[0].float(),
        dim=-1
    )

    token_log_probs = log_probs.gather(
        dim=1,
        index=targets.unsqueeze(1)
    ).squeeze(1)

    # Ignore question/prompt tokens
    answer_start = len(prefix_ids) - 1

    answer_log_probs = token_log_probs[answer_start:]

    return {
        "total_log_prob": answer_log_probs.sum().item(),
        "average_log_prob": answer_log_probs.mean().item(),
        "token_count": len(answer_log_probs)
    }

print("CELL 66A COMPLETE — SCORING FUNCTION REPAIRED!")


CELL 66A COMPLETE — SCORING FUNCTION REPAIRED!


In [ ]:

correct_count = 0
diagnostic_results = []

print("=" * 70)
print("GRAC-LM v0.4 — CORRECTED DIAGNOSTIC")
print("=" * 70)

for number, item in enumerate(diagnostic_questions, start=1):
    print(f"\nQUESTION {number}: {item['question']}")

    scores = []

    for i, answer in enumerate(item["answers"]):
        result = score_candidate_answer(
            item["question"], answer
        )

        score = result["average_log_prob"]
        scores.append(score)

        print(
            f"  {chr(65+i)}. {answer} "
            f"| Score: {score:.4f}"
        )

    predicted = max(
        range(len(scores)),
        key=lambda i: scores[i]
    )

    correct = item["correct"]
    passed = predicted == correct

    correct_count += int(passed)

    print(
        "GRAC chose:",
        chr(65 + predicted),
        "|",
        "CORRECT" if passed else "INCORRECT"
    )

    diagnostic_results.append({
        "question": item["question"],
        "predicted": predicted,
        "correct": correct,
        "is_correct": passed,
        "scores": scores
    })

accuracy = 100 * correct_count / len(diagnostic_questions)

print("\n" + "=" * 70)
print("FINAL DIAGNOSTIC REPORT")
print("=" * 70)
print(f"Correct: {correct_count}/{len(diagnostic_questions)}")
print(f"Accuracy: {accuracy:.1f}%")
print("CELL 66B COMPLETE!")


GRAC-LM v0.4 — CORRECTED DIAGNOSTIC

QUESTION 1: What is 2 + 2?


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [ ]:

import torch
import torch.nn.functional as F

print("=" * 70)
print("GRAC-LM v0.4 — CELL 66C: FULL DIAGNOSTIC REPAIR")
print("=" * 70)

assert "chat_model" in globals(), "Run Cell 65 first."
assert "diagnostic_questions" in globals(), "Run Cell 66 setup first."

chat_model.eval()

device = next(chat_model.parameters()).device
context_limit = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")

# Completely replace the old broken function
@torch.inference_mode()
def score_candidate_answer(question, answer):

    question = " ".join(question.split()).strip()
    answer = " ".join(answer.split()).strip()

    prefix = f"User: {question} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(answer).ids

    full_ids = prefix_ids + answer_ids + [eos_id]

    if len(full_ids) > context_limit:
        raise ValueError("Sequence exceeds context limit.")

    # Input shape: [1, sequence_length]
    x = torch.tensor(
        [full_ids[:-1]],
        dtype=torch.long,
        device=device
    )

    # Target shape: [sequence_length]
    targets = torch.tensor(
        full_ids[1:],
        dtype=torch.long,
        device=device
    )

    logits, _ = chat_model(x)

    # Shape: [sequence_length, vocabulary_size]
    log_probs = F.log_softmax(
        logits[0].float(),
        dim=-1
    )

    # Both tensors now have compatible dimensions.
    selected_log_probs = log_probs.gather(
        dim=1,
        index=targets[:, None]
    ).squeeze(1)

    # Score answer and EOS only
    answer_start = len(prefix_ids) - 1
    answer_scores = selected_log_probs[answer_start:]

    return {
        "average_log_prob": answer_scores.mean().item(),
        "total_log_prob": answer_scores.sum().item(),
        "token_count": answer_scores.numel()
    }

print("Corrected scoring function installed.")
print("Running all diagnostic questions...\n")

correct_count = 0
diagnostic_results = []

for number, item in enumerate(diagnostic_questions, start=1):

    print("-" * 70)
    print(f"QUESTION {number}: {item['question']}")

    scores = []

    for i, answer in enumerate(item["answers"]):

        result = score_candidate_answer(
            item["question"],
            answer
        )

        scores.append(result["average_log_prob"])

        print(
            f"{chr(65+i)}. {answer} "
            f"| Score: {result['average_log_prob']:.4f}"
        )

    predicted = max(
        range(len(scores)),
        key=lambda i: scores[i]
    )

    correct = item["correct"]
    passed = predicted == correct

    correct_count += int(passed)

    print(
        "GRAC preferred:",
        chr(65 + predicted),
        "|",
        "CORRECT" if passed else "INCORRECT"
    )

    diagnostic_results.append({
        "question": item["question"],
        "predicted": predicted,
        "correct": correct,
        "is_correct": passed,
        "scores": scores
    })

accuracy = 100 * correct_count / len(diagnostic_questions)

print("\n" + "=" * 70)
print("GRAC-LM v0.4 — FINAL DIAGNOSTIC REPORT")
print("=" * 70)

print(f"Correct: {correct_count}/{len(diagnostic_questions)}")
print(f"Accuracy: {accuracy:.1f}%")

print("\nCELL 66C COMPLETE!")


GRAC-LM v0.4 — CELL 66C: FULL DIAGNOSTIC REPAIR
Corrected scoring function installed.
Running all diagnostic questions...

----------------------------------------------------------------------
QUESTION 1: What is 2 + 2?
A. 4. | Score: -3.3615
B. 5. | Score: -3.5714
C. 10. | Score: -3.5656
GRAC preferred: A | CORRECT
----------------------------------------------------------------------
QUESTION 2: What is the capital of India?
A. New Delhi. | Score: -3.2897
B. Mumbai. | Score: -3.5785
C. Paris. | Score: -4.2839
GRAC preferred: A | CORRECT
----------------------------------------------------------------------
QUESTION 3: What is your name?
A. My name is GRAC. | Score: -3.3361
B. My name is Google. | Score: -2.3885
C. My name is Wikipedia. | Score: -2.4042
GRAC preferred: B | INCORRECT
----------------------------------------------------------------------
QUESTION 4: What is photosynthesis?
A. Plants use sunlight to make food. | Score: -3.6363
B. Plants eat rocks to make food. | Score: 

In [ ]:

import torch
import random

print("=" * 68)
print("GRAC-LM v0.4 — CELL 67: QUESTION SENSITIVITY TEST")
print("=" * 68)

# Uses the repaired score_candidate_answer from Cell 66C.
assert "score_candidate_answer" in globals()

tests = [
    {
        "questions": [
            ("What is the capital of India?", "New Delhi."),
            ("What is the capital of France?", "Paris.")
        ],
        "candidates": ["New Delhi.", "Paris.", "Mumbai."]
    },
    {
        "questions": [
            ("What is 2 + 2?", "4."),
            ("What is 2 + 3?", "5.")
        ],
        "candidates": ["4.", "5.", "10."]
    },
    {
        "questions": [
            ("What is photosynthesis?",
             "Plants use sunlight to make food."),
            ("What is gravity?",
             "A force that attracts objects with mass.")
        ],
        "candidates": [
            "Plants use sunlight to make food.",
            "A force that attracts objects with mass.",
            "A computer processes data."
        ]
    }
]

correct = 0
total = 0

for test_number, test in enumerate(tests, 1):
    print(f"\n{'-' * 68}")
    print(f"TEST GROUP {test_number}")
    print("-" * 68)

    for question, expected in test["questions"]:
        scores = [
            score_candidate_answer(
                question, candidate
            )["average_log_prob"]
            for candidate in test["candidates"]
        ]

        best_index = max(
            range(len(scores)),
            key=lambda i: scores[i]
        )

        predicted = test["candidates"][best_index]
        passed = predicted == expected

        correct += int(passed)
        total += 1

        print(f"\nQUESTION: {question}")

        for candidate, score in zip(
            test["candidates"], scores
        ):
            print(f"  {score:8.4f} | {candidate}")

        print("GRAC PREFERRED:", predicted)
        print("EXPECTED:", expected)
        print("RESULT:", "CORRECT" if passed else "INCORRECT")

print("\n" + "=" * 68)
print("QUESTION SENSITIVITY REPORT")
print("=" * 68)
print(f"Correct preferences: {correct}/{total}")
print(f"Accuracy: {100 * correct / total:.1f}%")
print("\nCELL 67 COMPLETE!")


GRAC-LM v0.4 — CELL 67: QUESTION SENSITIVITY TEST

--------------------------------------------------------------------
TEST GROUP 1
--------------------------------------------------------------------

QUESTION: What is the capital of India?
   -3.2897 | New Delhi.
   -4.2839 | Paris.
   -3.5785 | Mumbai.
GRAC PREFERRED: New Delhi.
EXPECTED: New Delhi.
RESULT: CORRECT

QUESTION: What is the capital of France?
   -3.6427 | New Delhi.
   -4.4156 | Paris.
   -4.1202 | Mumbai.
GRAC PREFERRED: New Delhi.
EXPECTED: Paris.
RESULT: INCORRECT

--------------------------------------------------------------------
TEST GROUP 2
--------------------------------------------------------------------

QUESTION: What is 2 + 2?
   -3.3615 | 4.
   -3.5714 | 5.
   -3.5656 | 10.
GRAC PREFERRED: 4.
EXPECTED: 4.
RESULT: CORRECT

QUESTION: What is 2 + 3?
   -3.2583 | 4.
   -3.4798 | 5.
   -3.5483 | 10.
GRAC PREFERRED: 4.
EXPECTED: 5.
RESULT: INCORRECT

----------------------------------------------------------

In [ ]:

import json
import random
from pathlib import Path
from collections import Counter

print("=" * 70)
print("GRAC-LM v0.4 — CELL 68: CONTRASTIVE QA DATASET")
print("=" * 70)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4b_dir = base_dir / "phase4b"
data_dir = phase4b_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# Each group contains related questions with different answers.
# Training and test phrasings are kept separate.

qa_facts = [
    {
        "id": "capital_india",
        "answer": "New Delhi.",
        "train": [
            "What is the capital of India?",
            "Name the capital city of India.",
            "Which city is India's capital?"
        ],
        "test": [
            "What city serves as the capital of India?"
        ]
    },
    {
        "id": "capital_france",
        "answer": "Paris.",
        "train": [
            "What is the capital of France?",
            "Name the capital city of France.",
            "Which city is France's capital?"
        ],
        "test": [
            "What city serves as the capital of France?"
        ]
    },
    {
        "id": "capital_japan",
        "answer": "Tokyo.",
        "train": [
            "What is the capital of Japan?",
            "Name the capital city of Japan.",
            "Which city is Japan's capital?"
        ],
        "test": [
            "What city serves as the capital of Japan?"
        ]
    },
    {
        "id": "math_2_2",
        "answer": "4.",
        "train": [
            "What is 2 + 2?",
            "Calculate 2 plus 2.",
            "What is the sum of 2 and 2?"
        ],
        "test": [
            "If you add 2 and 2, what do you get?"
        ]
    },
    {
        "id": "math_2_3",
        "answer": "5.",
        "train": [
            "What is 2 + 3?",
            "Calculate 2 plus 3.",
            "What is the sum of 2 and 3?"
        ],
        "test": [
            "If you add 2 and 3, what do you get?"
        ]
    },
    {
        "id": "math_3_3",
        "answer": "6.",
        "train": [
            "What is 3 + 3?",
            "Calculate 3 plus 3.",
            "What is the sum of 3 and 3?"
        ],
        "test": [
            "If you add 3 and 3, what do you get?"
        ]
    },
    {
        "id": "photosynthesis",
        "answer": "Plants use sunlight to make food.",
        "train": [
            "What is photosynthesis?",
            "Explain photosynthesis.",
            "How do plants make food using sunlight?"
        ],
        "test": [
            "Describe how plants produce food with sunlight."
        ]
    },
    {
        "id": "gravity",
        "answer": "Gravity is the attraction between objects with mass.",
        "train": [
            "What is gravity?",
            "Explain gravity.",
            "What force attracts objects with mass?"
        ],
        "test": [
            "Describe the force that pulls masses together."
        ]
    },
    {
        "id": "computer",
        "answer": "A computer is an electronic device that processes data.",
        "train": [
            "What is a computer?",
            "Define a computer.",
            "What kind of device processes data electronically?"
        ],
        "test": [
            "Describe an electronic device used to process information."
        ]
    },
    {
        "id": "ai",
        "answer": "Artificial intelligence is technology that performs tasks associated with human intelligence.",
        "train": [
            "What is artificial intelligence?",
            "Define artificial intelligence.",
            "What does AI mean in computing?"
        ],
        "test": [
            "Explain what artificial intelligence means."
        ]
    },
    {
        "id": "sky_blue",
        "answer": "The sky appears blue because air scatters blue light more strongly.",
        "train": [
            "Why is the sky blue?",
            "Explain why the sky looks blue.",
            "What causes the blue color of the sky?"
        ],
        "test": [
            "Why does the daytime sky appear blue?"
        ]
    },
    {
        "id": "model_name",
        "answer": "My name is GRAC.",
        "train": [
            "What is your name?",
            "Tell me your name.",
            "What should I call you?"
        ],
        "test": [
            "How should I address you?"
        ]
    }
]

train_rows = []
test_rows = []

for fact in qa_facts:
    for question in fact["train"]:
        train_rows.append({
            "id": fact["id"],
            "instruction": question,
            "response": fact["answer"]
        })

    for question in fact["test"]:
        test_rows.append({
            "id": fact["id"],
            "instruction": question,
            "response": fact["answer"]
        })

random.Random(2026).shuffle(train_rows)

# Check tokenization before saving.
max_context = GRAC_V04_CONFIG["context_length"]
unk_id = tokenizer.token_to_id("<unk>")
eos_id = tokenizer.token_to_id("<eos>")

def check_example(row):
    prefix = f"User: {row['instruction']} Assistant: "
    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(row["response"]).ids
    full_ids = prefix_ids + answer_ids + [eos_id]

    return (
        len(full_ids) <= max_context
        and unk_id not in full_ids
        and len(answer_ids) > 0
    )

for row in train_rows + test_rows:
    assert check_example(row), (
        f"Tokenizer rejected: {row['id']}"
    )

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(data_dir / "train.jsonl", train_rows)
save_jsonl(data_dir / "test.jsonl", test_rows)

print("\nDATASET CREATED")
print("Fact groups:", len(qa_facts))
print("Training examples:", len(train_rows))
print("Held-out paraphrases:", len(test_rows))

print("\nTRAINING GROUPS:")
for key, count in sorted(
    Counter(row["id"] for row in train_rows).items()
):
    print(f"  {key}: {count}")

print("\nSAVED:")
print(data_dir / "train.jsonl")
print(data_dir / "test.jsonl")

print("\nCELL 68 COMPLETE!")


GRAC-LM v0.4 — CELL 68: CONTRASTIVE QA DATASET

DATASET CREATED
Fact groups: 12
Training examples: 36
Held-out paraphrases: 12

TRAINING GROUPS:
  ai: 3
  capital_france: 3
  capital_india: 3
  capital_japan: 3
  computer: 3
  gravity: 3
  math_2_2: 3
  math_2_3: 3
  math_3_3: 3
  model_name: 3
  photosynthesis: 3
  sky_blue: 3

SAVED:
/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4b/data/train.jsonl
/content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4b/data/test.jsonl

CELL 68 COMPLETE!


In [ ]:

import json
import torch

print("=" * 70)
print("GRAC-LM v0.4 — CELL 69: PHASE 4B TOKENIZATION")
print("=" * 70)

def prepare_phase4b_example(row):
    prefix = (
        f"User: {row['instruction']} Assistant: "
    )

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(row["response"]).ids

    full_ids = prefix_ids + answer_ids + [eos_id]

    x = full_ids[:-1]
    y = full_ids[1:]

    # Only the assistant answer contributes to loss.
    y[:len(prefix_ids) - 1] = (
        [-100] * (len(prefix_ids) - 1)
    )

    return {
        "input_ids": x,
        "labels": y,
        "id": row["id"],
        "question": row["instruction"],
        "answer": row["response"]
    }

def read_phase4b_data(filename):
    results = []

    with open(data_dir / filename, encoding="utf-8") as f:
        for line in f:
            results.append(
                prepare_phase4b_example(json.loads(line))
            )

    return results

phase4b_train = read_phase4b_data("train.jsonl")
phase4b_test = read_phase4b_data("test.jsonl")

print("Training examples:", len(phase4b_train))
print("Held-out paraphrases:", len(phase4b_test))

sample = phase4b_train[0]

print("\nEXAMPLE QUESTION:")
print(sample["question"])

print("\nEXPECTED ANSWER:")
print(sample["answer"])

print("\nTRAINABLE TARGET TOKENS:")
target_ids = [
    token for token in sample["labels"]
    if token != -100
]
print(tokenizer.decode(target_ids))

print("\nCELL 69 COMPLETE!")


GRAC-LM v0.4 — CELL 69: PHASE 4B TOKENIZATION
Training examples: 36
Held-out paraphrases: 12

EXAMPLE QUESTION:
Explain why the sky looks blue.

EXPECTED ANSWER:
The sky appears blue because air scatters blue light more strongly.

TRAINABLE TARGET TOKENS:
The sky appears blue because air scatters blue light more strongly.

CELL 69 COMPLETE!


In [ ]:

import torch
import torch.nn.functional as F
import random
import json
import math
import gc
from pathlib import Path

print("=" * 70)
print("GRAC-LM v0.4 — CELL 70: CONTROLLED QA FINE-TUNING")
print("=" * 70)

# --------------------------------------------------
# 1. PATHS AND CONFIGURATION
# --------------------------------------------------

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4b_dir = base_dir / "phase4b"
phase4b_dir.mkdir(parents=True, exist_ok=True)

source_checkpoint = (
    base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"
)

output_checkpoint = (
    phase4b_dir / "GRAC-LM-v0.4-phase4b-step300.pt"
)

history_path = (
    phase4b_dir / "training_history.json"
)

assert source_checkpoint.exists(), "Phase 4 checkpoint missing!"
assert len(phase4b_train) == 36, "Unexpected training dataset!"
assert len(phase4b_test) == 12, "Unexpected test dataset!"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable Colab GPU runtime!"

FT4B = {
    "max_steps": 300,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "log_every": 25,
    "seed": 2026
}

random.seed(FT4B["seed"])
torch.manual_seed(FT4B["seed"])
torch.cuda.manual_seed_all(FT4B["seed"])

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(phase4b_train))
print("Held-out examples (not used):", len(phase4b_test))
print("Optimizer steps:", FT4B["max_steps"])

# --------------------------------------------------
# 2. LOAD MODEL FROM PHASE 4
# --------------------------------------------------

checkpoint = torch.load(
    source_checkpoint,
    map_location="cpu",
    weights_only=False
)

# Reuse the existing model if available
if "ft_model" in globals():
    model_4b = ft_model
elif "chat_model" in globals():
    model_4b = chat_model
else:
    model_4b = GRACLM(GRAC_V04_CONFIG).to(device)

model_4b.load_state_dict(
    checkpoint["model_state_dict"]
)

model_4b.to(device)
model_4b.train()

del checkpoint
gc.collect()

optimizer_4b = torch.optim.AdamW(
    model_4b.parameters(),
    lr=FT4B["learning_rate"],
    weight_decay=FT4B["weight_decay"]
)

scaler_4b = torch.amp.GradScaler("cuda")

pad_id = tokenizer.token_to_id("<eos>")

# --------------------------------------------------
# 3. CREATE TRAINING BATCHES
# --------------------------------------------------

rng = random.Random(2026)

def make_4b_batch(batch_size):
    samples = rng.choices(
        phase4b_train,
        k=batch_size
    )

    max_len = max(
        len(sample["input_ids"])
        for sample in samples
    )

    inputs = []
    targets = []

    for sample in samples:
        x = sample["input_ids"]
        y = sample["labels"]

        padding = max_len - len(x)

        inputs.append(
            x + [pad_id] * padding
        )

        targets.append(
            y + [-100] * padding
        )

    return (
        torch.tensor(
            inputs,
            dtype=torch.long,
            device=device
        ),
        torch.tensor(
            targets,
            dtype=torch.long,
            device=device
        )
    )

# --------------------------------------------------
# 4. LOSS FUNCTION
# --------------------------------------------------

def phase4b_loss(x, y):
    logits, _ = model_4b(x)

    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)).float(),
        y.reshape(-1),
        ignore_index=-100
    )

# --------------------------------------------------
# 5. TRAINING LOOP
# --------------------------------------------------

history_4b = []

print("\nTRAINING STARTED!\n")

for step in range(1, FT4B["max_steps"] + 1):
    model_4b.train()
    optimizer_4b.zero_grad(set_to_none=True)

    accumulated_loss = 0.0

    for _ in range(
        FT4B["gradient_accumulation_steps"]
    ):
        x, y = make_4b_batch(
            FT4B["micro_batch_size"]
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss = phase4b_loss(x, y)

        accumulated_loss += loss.item()

        scaler_4b.scale(
            loss / FT4B["gradient_accumulation_steps"]
        ).backward()

    scaler_4b.unscale_(optimizer_4b)

    torch.nn.utils.clip_grad_norm_(
        model_4b.parameters(),
        1.0
    )

    scaler_4b.step(optimizer_4b)
    scaler_4b.update()

    if step % FT4B["log_every"] == 0:
        average_loss = (
            accumulated_loss /
            FT4B["gradient_accumulation_steps"]
        )

        history_4b.append({
            "step": step,
            "training_loss": average_loss
        })

        print(
            f"Step {step:3d}/{FT4B['max_steps']} | "
            f"Training loss: {average_loss:.4f}",
            flush=True
        )

# --------------------------------------------------
# 6. SAVE CHECKPOINT
# --------------------------------------------------

torch.save(
    {
        "model_state_dict": model_4b.state_dict(),
        "step": FT4B["max_steps"],
        "config": GRAC_V04_CONFIG,
        "source_checkpoint": str(source_checkpoint),
        "training_config": FT4B
    },
    output_checkpoint
)

with open(history_path, "w") as f:
    json.dump(history_4b, f, indent=2)

print("\n" + "=" * 70)
print("GRAC-LM v0.4 — PHASE 4B TRAINING COMPLETE")
print("=" * 70)

print("Final step:", FT4B["max_steps"])
print("Checkpoint:", output_checkpoint)
print("History:", history_path)
print("Held-out questions remain unused.")

print("\nCELL 70 COMPLETE!")


GRAC-LM v0.4 — CELL 70: CONTROLLED QA FINE-TUNING
GPU: Tesla T4
Training examples: 36
Held-out examples (not used): 12
Optimizer steps: 300

TRAINING STARTED!

Step  25/300 | Training loss: 0.5398
Step  50/300 | Training loss: 0.0551
Step  75/300 | Training loss: 0.0305
Step 100/300 | Training loss: 0.0186
Step 125/300 | Training loss: 0.0066
Step 150/300 | Training loss: 0.0058
Step 175/300 | Training loss: 0.0030
Step 200/300 | Training loss: 0.0028
Step 225/300 | Training loss: 0.0020
Step 250/300 | Training loss: 0.0013
Step 275/300 | Training loss: 0.0013
Step 300/300 | Training loss: 0.0013

GRAC-LM v0.4 — PHASE 4B TRAINING COMPLETE
Final step: 300
Checkpoint: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4b/GRAC-LM-v0.4-phase4b-step300.pt
History: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4b/training_history.json
Held-out questions remain unused.

CELL 70 COMPLETE!


In [ ]:

import torch
import torch.nn.functional as F
import json
import random
from pathlib import Path

print("=" * 72)
print("GRAC-LM v0.4 — CELL 71: GENERALIZATION EVALUATION")
print("=" * 72)

# --------------------------------------------------
# 1. PATHS AND CONFIGURATION
# --------------------------------------------------

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")

old_path = (
    base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"
)

new_path = (
    base_dir / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt"
)

assert old_path.exists(), "Original checkpoint missing!"
assert new_path.exists(), "Phase 4B checkpoint missing!"
assert len(phase4b_test) == 12, "Held-out dataset missing!"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

eos_id = tokenizer.token_to_id("<eos>")
context_limit = GRAC_V04_CONFIG["context_length"]

# Reuse a single model to conserve GPU memory.
if "model_4b" in globals():
    eval_model = model_4b
elif "chat_model" in globals():
    eval_model = chat_model
else:
    eval_model = GRACLM(GRAC_V04_CONFIG).to(device)

eval_model.to(device)
eval_model.eval()

# --------------------------------------------------
# 2. LOAD CHECKPOINT
# --------------------------------------------------

def load_eval_checkpoint(path):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    eval_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    del checkpoint
    eval_model.eval()

# --------------------------------------------------
# 3. ANSWER SCORING
# --------------------------------------------------

@torch.inference_mode()
def score_answer(question, answer):
    question = " ".join(question.split()).strip()
    answer = " ".join(answer.split()).strip()

    prefix = f"User: {question} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(answer).ids

    full_ids = prefix_ids + answer_ids + [eos_id]

    if len(full_ids) > context_limit:
        raise ValueError("Sequence exceeds context length.")

    x = torch.tensor(
        [full_ids[:-1]],
        dtype=torch.long,
        device=device
    )

    targets = torch.tensor(
        full_ids[1:],
        dtype=torch.long,
        device=device
    )

    logits, _ = eval_model(x)

    log_probs = F.log_softmax(
        logits[0].float(),
        dim=-1
    )

    selected = log_probs.gather(
        dim=1,
        index=targets[:, None]
    ).squeeze(1)

    answer_start = len(prefix_ids) - 1
    answer_log_probs = selected[answer_start:]

    return answer_log_probs.mean().item()

# --------------------------------------------------
# 4. GREEDY GENERATION
# --------------------------------------------------

@torch.inference_mode()
def generate_answer(question, max_new_tokens=60):
    question = " ".join(question.split()).strip()

    prefix = f"User: {question} Assistant: "
    tokens = tokenizer.encode(prefix).ids

    generated = []

    for _ in range(max_new_tokens):
        if len(tokens) >= context_limit:
            break

        x = torch.tensor(
            [tokens],
            dtype=torch.long,
            device=device
        )

        logits, _ = eval_model(x)

        next_id = int(
            torch.argmax(
                logits[0, -1]
            ).item()
        )

        if next_id == eos_id:
            break

        tokens.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# --------------------------------------------------
# 5. TEST BOTH MODELS
# --------------------------------------------------

# Every fact competes against all other fact answers.
# This makes the test harder than choosing from 3 options.
candidates = [
    fact["answer"]
    for fact in qa_facts
]

results = {}

for label, path in [
    ("PHASE 4 EXPANDED", old_path),
    ("PHASE 4B CONTROLLED", new_path)
]:
    print("\n" + "=" * 72)
    print("EVALUATING:", label)
    print("=" * 72)

    load_eval_checkpoint(path)

    correct_rankings = 0
    correct_generations = 0
    records = []

    for i, sample in enumerate(phase4b_test, start=1):
        question = sample["question"]
        expected = sample["answer"]

        scores = [
            score_answer(question, candidate)
            for candidate in candidates
        ]

        best_index = max(
            range(len(scores)),
            key=lambda index: scores[index]
        )

        predicted = candidates[best_index]

        generated = generate_answer(question)

        ranking_correct = predicted == expected

        # Exact-match generation is deliberately strict.
        generation_correct = (
            " ".join(generated.split()).casefold()
            ==
            " ".join(expected.split()).casefold()
        )

        correct_rankings += int(ranking_correct)
        correct_generations += int(generation_correct)

        print(f"\n[{i}/12] QUESTION: {question}")
        print("EXPECTED:", expected)
        print("PREFERRED:", predicted)
        print(
            "RANKING:",
            "CORRECT" if ranking_correct else "INCORRECT"
        )
        print("GENERATED:", generated or "[Empty]")
        print(
            "GENERATION:",
            "EXACT MATCH" if generation_correct else "NO MATCH"
        )

        records.append({
            "question": question,
            "expected": expected,
            "preferred": predicted,
            "generated": generated,
            "ranking_correct": ranking_correct,
            "generation_correct": generation_correct
        })

    results[label] = {
        "ranking_correct": correct_rankings,
        "generation_correct": correct_generations,
        "records": records
    }

    print("\n" + "-" * 72)
    print(
        "RANKING ACCURACY:",
        f"{correct_rankings}/12",
        f"({100 * correct_rankings / 12:.1f}%)"
    )
    print(
        "EXACT GENERATION:",
        f"{correct_generations}/12",
        f"({100 * correct_generations / 12:.1f}%)"
    )

# --------------------------------------------------
# 6. COMPARISON REPORT
# --------------------------------------------------

print("\n" + "=" * 72)
print("GRAC-LM v0.4 — PHASE 4 VS PHASE 4B")
print("=" * 72)

old = results["PHASE 4 EXPANDED"]
new = results["PHASE 4B CONTROLLED"]

print(
    "Ranking:",
    f"{old['ranking_correct']}/12",
    "->",
    f"{new['ranking_correct']}/12"
)

print(
    "Exact generation:",
    f"{old['generation_correct']}/12",
    "->",
    f"{new['generation_correct']}/12"
)

# Save evaluation results
output_path = (
    base_dir / "phase4b/generalization_results.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("\nSaved evaluation report:", output_path)
print("CELL 71 COMPLETE!")

# Restore the Phase 4B checkpoint for subsequent experiments.
load_eval_checkpoint(new_path)


GRAC-LM v0.4 — CELL 71: GENERALIZATION EVALUATION

EVALUATING: PHASE 4 EXPANDED

[1/12] QUESTION: What city serves as the capital of India?
EXPECTED: New Delhi.
PREFERRED: New Delhi.
RANKING: CORRECT
GENERATED: The capital of India is the capital of India.
GENERATION: NO MATCH

[2/12] QUESTION: What city serves as the capital of France?
EXPECTED: Paris.
PREFERRED: New Delhi.
RANKING: INCORRECT
GENERATED: The capital of France is the capital of France.
GENERATION: NO MATCH

[3/12] QUESTION: What city serves as the capital of Japan?
EXPECTED: Tokyo.
PREFERRED: New Delhi.
RANKING: INCORRECT
GENERATED: The capital of Japan is the capital of Japan.
GENERATION: NO MATCH

[4/12] QUESTION: If you add 2 and 2, what do you get?
EXPECTED: 4.
PREFERRED: A computer is an electronic device that processes data.
RANKING: INCORRECT
GENERATED: These are the same as you are, you are not able to get yourself. If you are able to get yourself, you can get yourself to get yourself to yourself. If you are abl

In [ ]:

import torch
import torch.nn.functional as F
import json
from pathlib import Path

print("=" * 70)
print("GRAC-LM v0.4 — CELL 71: PHASE 4B GENERALIZATION TEST")
print("=" * 70)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")

old_path = base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"
new_path = base_dir / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt"

assert old_path.exists(), "Phase 4 checkpoint missing!"
assert new_path.exists(), "Phase 4B checkpoint missing!"
assert len(phase4b_test) == 12
assert len(qa_facts) == 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eos_id = tokenizer.token_to_id("<eos>")
context_limit = GRAC_V04_CONFIG["context_length"]

# Use the existing model to save GPU memory.
if "model_4b" in globals():
    eval_model = model_4b
else:
    eval_model = GRACLM(GRAC_V04_CONFIG).to(device)

eval_model.to(device)
eval_model.eval()

def load_model(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    eval_model.load_state_dict(ckpt["model_state_dict"])
    eval_model.eval()
    del ckpt

@torch.inference_mode()
def answer_score(question, answer):
    prefix = f"User: {' '.join(question.split())} Assistant: "

    p = tokenizer.encode(prefix).ids
    a = tokenizer.encode(" ".join(answer.split())).ids

    ids = p + a + [eos_id]

    assert len(ids) <= context_limit

    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    targets = torch.tensor(ids[1:], dtype=torch.long, device=device)

    logits, _ = eval_model(x)

    log_probs = F.log_softmax(logits[0].float(), dim=-1)

    selected = log_probs.gather(
        1, targets.unsqueeze(1)
    ).squeeze(1)

    answer_log_probs = selected[len(p)-1:]

    return float(answer_log_probs.mean().item())

@torch.inference_mode()
def greedy_answer(question, max_new_tokens=80):
    prefix = f"User: {' '.join(question.split())} Assistant: "
    tokens = tokenizer.encode(prefix).ids
    generated = []

    for _ in range(max_new_tokens):
        if len(tokens) >= context_limit:
            break

        x = torch.tensor([tokens], dtype=torch.long, device=device)
        logits, _ = eval_model(x)

        next_id = int(torch.argmax(logits[0, -1]).item())

        if next_id == eos_id:
            break

        tokens.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

# Use the same 12 candidate answers for both models.
candidates = [fact["answer"] for fact in qa_facts]

results = {}

for model_name, path in [
    ("PHASE 4 EXPANDED", old_path),
    ("PHASE 4B CONTROLLED", new_path)
]:
    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    load_model(path)

    rank_correct = 0
    gen_correct = 0
    records = []

    for index, sample in enumerate(phase4b_test, 1):
        question = sample["question"]
        expected = sample["answer"]

        scores = [
            answer_score(question, candidate)
            for candidate in candidates
        ]

        predicted = candidates[max(
            range(len(scores)),
            key=lambda i: scores[i]
        )]

        generated = greedy_answer(question)

        rank_ok = predicted == expected
        gen_ok = (
            " ".join(generated.split()).casefold()
            == " ".join(expected.split()).casefold()
        )

        rank_correct += int(rank_ok)
        gen_correct += int(gen_ok)

        print(f"\n[{index}/12] {question}")
        print("Expected:", expected)
        print("Preferred:", predicted)
        print("Generated:", generated or "[Empty]")
        print(
            "Ranking:", "CORRECT" if rank_ok else "INCORRECT",
            "| Generation:", "MATCH" if gen_ok else "NO MATCH"
        )

        records.append({
            "question": question,
            "expected": expected,
            "preferred": predicted,
            "generated": generated,
            "ranking_correct": rank_ok,
            "generation_correct": gen_ok
        })

    results[model_name] = {
        "ranking_correct": rank_correct,
        "generation_correct": gen_correct,
        "records": records
    }

    print(
        f"\nRanking accuracy: {rank_correct}/12 "
        f"({rank_correct/12*100:.1f}%)"
    )
    print(
        f"Exact generation: {gen_correct}/12 "
        f"({gen_correct/12*100:.1f}%)"
    )

print("\n" + "=" * 70)
print("FINAL COMPARISON — GRAC-LM v0.4")
print("=" * 70)

old = results["PHASE 4 EXPANDED"]
new = results["PHASE 4B CONTROLLED"]

print(
    f"Answer ranking: "
    f"{old['ranking_correct']}/12 -> {new['ranking_correct']}/12"
)
print(
    f"Exact generation: "
    f"{old['generation_correct']}/12 -> {new['generation_correct']}/12"
)

output_path = base_dir / "phase4b/generalization_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Restore the new model for later use.
load_model(new_path)

print("\nResults saved:", output_path)
print("CELL 71 COMPLETE!")


GRAC-LM v0.4 — CELL 71: PHASE 4B GENERALIZATION TEST

MODEL: PHASE 4 EXPANDED

[1/12] What city serves as the capital of India?
Expected: New Delhi.
Preferred: New Delhi.
Generated: The capital of India is the capital of India.
Ranking: CORRECT | Generation: NO MATCH

[2/12] What city serves as the capital of France?
Expected: Paris.
Preferred: New Delhi.
Generated: The capital of France is the capital of France.
Ranking: INCORRECT | Generation: NO MATCH

[3/12] What city serves as the capital of Japan?
Expected: Tokyo.
Preferred: New Delhi.
Generated: The capital of Japan is the capital of Japan.
Ranking: INCORRECT | Generation: NO MATCH

[4/12] If you add 2 and 2, what do you get?
Expected: 4.
Preferred: A computer is an electronic device that processes data.
Generated: These are the same as you are, you are not able to get yourself. If you are able to get yourself, you can get yourself to get yourself to yourself. If you are able to get yourself to yourself. If you are able to get y

In [ ]:

import torch
import json
import re
from pathlib import Path

print("=" * 74)
print("GRAC-LM v0.4 — CELL 72: PHASE 4C STRESS TEST")
print("=" * 74)

# ==================================================
# 1. CONFIGURATION
# ==================================================

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"
phase4c_dir.mkdir(parents=True, exist_ok=True)

checkpoints = {
    "PHASE 4 EXPANDED":
        base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt",

    "PHASE 4B CONTROLLED":
        base_dir / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt"
}

for name, path in checkpoints.items():
    assert path.exists(), f"Missing checkpoint: {path}"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

context_limit = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")

assert eos_id is not None

# Reuse existing model to conserve GPU memory.
if "model_4b" in globals():
    stress_model = model_4b
elif "eval_model" in globals():
    stress_model = eval_model
else:
    stress_model = GRACLM(GRAC_V04_CONFIG)

stress_model.to(device)
stress_model.eval()

def load_stress_checkpoint(path):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    stress_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    stress_model.eval()
    del checkpoint

# ==================================================
# 2. GENERATION
# ==================================================

@torch.inference_mode()
def stress_generate(question, max_new_tokens=65):
    question = " ".join(question.split()).strip()
    prompt = f"User: {question} Assistant: "

    token_ids = tokenizer.encode(prompt).ids

    generated = []

    for _ in range(max_new_tokens):
        if len(token_ids) >= context_limit:
            break

        x = torch.tensor(
            [token_ids],
            dtype=torch.long,
            device=device
        )

        logits, _ = stress_model(x)

        next_id = int(
            torch.argmax(logits[0, -1]).item()
        )

        if next_id == eos_id:
            break

        token_ids.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# ==================================================
# 3. STRESS TEST QUESTIONS
# ==================================================

tests = [

    # CATEGORY A: HARDER PARAPHRASES
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "India's national government is headquartered in which capital city?",
        "expected": "New Delhi."
    },
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "Identify the French city where the country's national government is based.",
        "expected": "Paris."
    },
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "Which Japanese city functions as the nation's capital?",
        "expected": "Tokyo."
    },
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "Two objects plus another three objects make how many objects?",
        "expected": "5."
    },
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "Which natural process allows green plants to produce sugars using sunlight?",
        "expected": "Plants use sunlight to make food."
    },
    {
        "category": "A_HARD_PARAPHRASE",
        "question": "What physical interaction causes massive bodies to attract each other?",
        "expected": "Gravity is the attraction between objects with mass."
    },

    # CATEGORY B: UNSEEN FACTS
    {
        "category": "B_UNSEEN_FACT",
        "question": "What is the capital of Germany?",
        "expected": "Berlin."
    },
    {
        "category": "B_UNSEEN_FACT",
        "question": "What is the capital of Italy?",
        "expected": "Rome."
    },
    {
        "category": "B_UNSEEN_FACT",
        "question": "What is 7 + 8?",
        "expected": "15."
    },
    {
        "category": "B_UNSEEN_FACT",
        "question": "What is 10 - 4?",
        "expected": "6."
    },
    {
        "category": "B_UNSEEN_FACT",
        "question": "What is the chemical formula of water?",
        "expected": "H2O."
    },
    {
        "category": "B_UNSEEN_FACT",
        "question": "Which planet is known as the Red Planet?",
        "expected": "Mars."
    },

    # CATEGORY C: DISTRACTORS
    {
        "category": "C_DISTRACTOR",
        "question": "My favorite color is green. Anyway, what is the capital of France?",
        "expected": "Paris."
    },
    {
        "category": "C_DISTRACTOR",
        "question": "A student was reading about Japan yesterday. What is the capital of India?",
        "expected": "New Delhi."
    },
    {
        "category": "C_DISTRACTOR",
        "question": "Ignore the number 9 in this sentence. What is 2 + 3?",
        "expected": "5."
    },
    {
        "category": "C_DISTRACTOR",
        "question": "Some people study gravity in school. What is photosynthesis?",
        "expected": "Plants use sunlight to make food."
    },
    {
        "category": "C_DISTRACTOR",
        "question": "A computer was switched on this morning. What is gravity?",
        "expected": "Gravity is the attraction between objects with mass."
    },
    {
        "category": "C_DISTRACTOR",
        "question": "I live near a large city. What is the capital of Japan?",
        "expected": "Tokyo."
    },

    # CATEGORY D: QUESTION SWITCHING
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is the capital of India?",
        "expected": "New Delhi."
    },
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is the capital of France?",
        "expected": "Paris."
    },
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is 2 + 2?",
        "expected": "4."
    },
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is 2 + 3?",
        "expected": "5."
    },
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is photosynthesis?",
        "expected": "Plants use sunlight to make food."
    },
    {
        "category": "D_QUESTION_SWITCH",
        "question": "What is gravity?",
        "expected": "Gravity is the attraction between objects with mass."
    }
]

# ==================================================
# 4. SCORING
# ==================================================

def normalize_answer(text):
    text = text.casefold().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# This is a strict normalized exact-match metric.
# Semantically correct but differently worded answers
# may be marked wrong; review outputs manually.

def is_exact_match(generated, expected):
    return (
        normalize_answer(generated)
        == normalize_answer(expected)
    )

# ==================================================
# 5. RUN BOTH MODELS
# ==================================================

all_results = {}

for model_name, checkpoint_path in checkpoints.items():

    print("\n" + "=" * 74)
    print("EVALUATING:", model_name)
    print("=" * 74)

    load_stress_checkpoint(checkpoint_path)

    records = []
    category_scores = {}

    for i, test in enumerate(tests, 1):
        category = test["category"]
        question = test["question"]
        expected = test["expected"]

        generated = stress_generate(question)

        passed = is_exact_match(generated, expected)

        if category not in category_scores:
            category_scores[category] = {
                "correct": 0,
                "total": 0
            }

        category_scores[category]["total"] += 1
        category_scores[category]["correct"] += int(passed)

        records.append({
            "category": category,
            "question": question,
            "expected": expected,
            "generated": generated,
            "exact_match": passed
        })

        print(f"\n[{i:02d}/{len(tests)}] {category}")
        print("QUESTION:", question)
        print("EXPECTED:", expected)
        print("GENERATED:", generated or "[EMPTY]")
        print("RESULT:", "MATCH" if passed else "NO MATCH")

    all_results[model_name] = {
        "records": records,
        "category_scores": category_scores
    }

    print("\nCATEGORY RESULTS:")

    for category, score in category_scores.items():
        correct = score["correct"]
        total = score["total"]

        print(
            f"  {category}: "
            f"{correct}/{total} "
            f"({100 * correct / total:.1f}%)"
        )

# ==================================================
# 6. FINAL COMPARISON
# ==================================================

print("\n" + "=" * 74)
print("GRAC-LM v0.4 — PHASE 4C STRESS TEST REPORT")
print("=" * 74)

categories = [
    "A_HARD_PARAPHRASE",
    "B_UNSEEN_FACT",
    "C_DISTRACTOR",
    "D_QUESTION_SWITCH"
]

for category in categories:
    print(f"\n{category}")

    for model_name in checkpoints:
        score = all_results[model_name]["category_scores"][category]

        print(
            f"  {model_name}: "
            f"{score['correct']}/{score['total']}"
        )

report_path = phase4c_dir / "cell72_stress_results.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        all_results,
        f,
        indent=2,
        ensure_ascii=False
    )

# Restore Phase 4B checkpoint.
load_stress_checkpoint(checkpoints["PHASE 4B CONTROLLED"])

print("\nReport saved:", report_path)
print("Phase 4B model restored.")
print("\nCELL 72 COMPLETE!")


GRAC-LM v0.4 — CELL 72: PHASE 4C STRESS TEST

EVALUATING: PHASE 4 EXPANDED

[01/24] A_HARD_PARAPHRASE
QUESTION: India's national government is headquartered in which capital city?
EXPECTED: New Delhi.
GENERATED: The state is headquartered in the city. The city is located in the city of India. The city is located in the city of India. The city is located in the city of India. The city is located in the city of India. The city is located in the city of India. The city is located in
RESULT: NO MATCH

[02/24] A_HARD_PARAPHRASE
QUESTION: Identify the French city where the country's national government is based.
EXPECTED: Paris.
GENERATED: The French government is based on the French government.
RESULT: NO MATCH

[03/24] A_HARD_PARAPHRASE
QUESTION: Which Japanese city functions as the nation's capital?
EXPECTED: Tokyo.
GENERATED: The nation's capital is the capital of the capital of the capital.
RESULT: NO MATCH

[04/24] A_HARD_PARAPHRASE
QUESTION: Two objects plus another three objects make

In [ ]:

import torch
import torch.nn.functional as F
import json
import re
import math
from pathlib import Path

print("=" * 74)
print("GRAC-LM v0.4 — CELL 73: CAPABILITY RETENTION AUDIT")
print("=" * 74)

# ==================================================
# 1. CONFIGURATION
# ==================================================

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"
phase4c_dir.mkdir(parents=True, exist_ok=True)

old_path = base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"
new_path = base_dir / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt"

assert old_path.exists(), "Phase 4 checkpoint missing"
assert new_path.exists(), "Phase 4B checkpoint missing"
assert "GRAC_V04_CONFIG" in globals()
assert "tokenizer" in globals()
assert "GRACLM" in globals()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

context_limit = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")
assert eos_id is not None

# Reuse one model to minimize GPU memory usage.
if "model_4b" in globals():
    audit_model = model_4b
elif "stress_model" in globals():
    audit_model = stress_model
else:
    audit_model = GRACLM(GRAC_V04_CONFIG)

audit_model = audit_model.to(device)
audit_model.eval()

def load_audit_model(path):
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )
    audit_model.load_state_dict(ckpt["model_state_dict"])
    audit_model.eval()
    del ckpt

# ==================================================
# 2. GENERAL INSTRUCTION TESTS
# ==================================================

# None of these exact prompts belongs to the
# Phase 4B controlled training set.
#
# These are qualitative probes, not a standardized
# benchmark. We will inspect the answers manually.

general_prompts = [
    "Hello! How are you?",
    "Explain what a scientist does.",
    "What is the difference between a cat and a dog?",
    "Give me three tips for studying.",
    "Why do people need sleep?",
    "Describe how rain forms.",
    "What should someone do when they make a mistake?",
    "Explain why exercise is important.",
    "Write one sentence about a beautiful garden.",
    "What is the purpose of a library?",
    "How does a bicycle work?",
    "What are the benefits of reading books?",
    "Explain why people learn mathematics.",
    "What does friendship mean?",
    "Describe a healthy breakfast."
]

# ==================================================
# 3. GREEDY GENERATION
# ==================================================

@torch.inference_mode()
def audit_generate(question, max_new_tokens=65):
    question = " ".join(question.split()).strip()
    prefix = f"User: {question} Assistant: "

    tokens = tokenizer.encode(prefix).ids
    generated = []

    for _ in range(max_new_tokens):
        if len(tokens) >= context_limit:
            break

        x = torch.tensor(
            [tokens],
            dtype=torch.long,
            device=device
        )

        logits, _ = audit_model(x)
        next_id = int(
            torch.argmax(logits[0, -1]).item()
        )

        if next_id == eos_id:
            break

        tokens.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# ==================================================
# 4. DETECT TRAINED-ANSWER COLLAPSE
# ==================================================

assert "qa_facts" in globals(), (
    "Run Cell 68 first: qa_facts is missing"
)

trained_answers = [
    fact["answer"] for fact in qa_facts
]

def normalize(text):
    return " ".join(
        re.sub(
            r"[^\w\s]",
            "",
            text.casefold()
        ).split()
    )

trained_answer_set = {
    normalize(answer)
    for answer in trained_answers
}

def detect_collapse(response):
    return normalize(response) in trained_answer_set

# ==================================================
# 5. LOAD ORIGINAL EXPANDED VALIDATION DATA
# ==================================================

# This evaluation uses the same Dolly-derived
# expanded validation file created in Phase 4.
# It is NOT the Phase 4B 12-question test set.

validation_path = (
    base_dir / "phase4/expanded_data/validation.jsonl"
)

assert validation_path.exists(), (
    "Expanded validation dataset missing: "
    + str(validation_path)
)

with open(validation_path, encoding="utf-8") as f:
    raw_validation = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Original expanded validation rows:",
      len(raw_validation))

print("Example validation keys:",
      list(raw_validation[0].keys()))

# Accept common Phase 4 dataset field names.
def extract_qa(row):
    question = (
        row.get("instruction")
        or row.get("question")
        or row.get("input")
    )
    answer = (
        row.get("response")
        or row.get("answer")
        or row.get("output")
    )

    if not isinstance(question, str):
        return None
    if not isinstance(answer, str):
        return None

    return (
        " ".join(question.split()).strip(),
        " ".join(answer.split()).strip()
    )

# Deterministic, fixed sample for both checkpoints.
# Avoid looking at the Phase 4B test set here.
validation_examples = []

for row in raw_validation:
    qa = extract_qa(row)

    if qa is None:
        continue

    question, answer = qa
    prefix = f"User: {question} Assistant: "

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(answer).ids

    if not answer_ids:
        continue

    if len(prefix_ids) + len(answer_ids) + 1 > context_limit:
        continue

    validation_examples.append({
        "question": question,
        "answer": answer,
        "prefix_ids": prefix_ids,
        "answer_ids": answer_ids
    })

# Select the same first 100 eligible examples
# for both models, avoiding randomness.
validation_examples = validation_examples[:100]

assert len(validation_examples) > 0, (
    "No eligible validation examples found"
)

print("Evaluation examples:", len(validation_examples))

# ==================================================
# 6. RESPONSE-ONLY VALIDATION LOSS
# ==================================================

@torch.inference_mode()
def validation_token_loss(examples):
    total_nll = 0.0
    total_tokens = 0

    for example in examples:
        prefix_ids = example["prefix_ids"]
        answer_ids = example["answer_ids"]

        ids = prefix_ids + answer_ids + [eos_id]

        x = torch.tensor(
            [ids[:-1]],
            dtype=torch.long,
            device=device
        )

        y = torch.tensor(
            ids[1:],
            dtype=torch.long,
            device=device
        )

        logits, _ = audit_model(x)

        answer_start = len(prefix_ids) - 1

        answer_logits = logits[
            0, answer_start:
        ].float()

        answer_targets = y[
            answer_start:
        ]

        loss_sum = F.cross_entropy(
            answer_logits,
            answer_targets,
            reduction="sum"
        )

        total_nll += loss_sum.item()
        total_tokens += answer_targets.numel()

    return {
        "loss": total_nll / total_tokens,
        "tokens": total_tokens,
        "perplexity": math.exp(total_nll / total_tokens)
    }

# ==================================================
# 7. EVALUATE BOTH CHECKPOINTS
# ==================================================

results_73 = {}

for model_name, path in [
    ("PHASE 4 EXPANDED", old_path),
    ("PHASE 4B CONTROLLED", new_path)
]:
    print("\n" + "=" * 74)
    print("EVALUATING:", model_name)
    print("=" * 74)

    load_audit_model(path)

    generated_records = []
    collapse_count = 0

    for i, question in enumerate(general_prompts, 1):
        answer = audit_generate(question)
        collapsed = detect_collapse(answer)

        collapse_count += int(collapsed)

        generated_records.append({
            "question": question,
            "generated": answer,
            "trained_answer_exact_match": collapsed
        })

        print(f"\n[{i:02d}/{len(general_prompts)}]")
        print("QUESTION:", question)
        print("ANSWER:", answer or "[EMPTY]")
        print("TRAINED-ANSWER MATCH:", collapsed)

    print("\nCalculating response-only validation loss...")
    validation_metrics = validation_token_loss(
        validation_examples
    )

    results_73[model_name] = {
        "general_prompts": generated_records,
        "trained_answer_matches": collapse_count,
        "general_prompt_count": len(general_prompts),
        "validation_metrics": validation_metrics
    }

    print("\nMODEL RESULTS:")
    print(
        "Trained-answer matches:",
        f"{collapse_count}/{len(general_prompts)}"
    )
    print(
        "Expanded validation loss:",
        f"{validation_metrics['loss']:.4f}"
    )
    print(
        "Expanded validation perplexity:",
        f"{validation_metrics['perplexity']:.2f}"
    )

# ==================================================
# 8. FINAL COMPARISON
# ==================================================

print("\n" + "=" * 74)
print("GRAC-LM v0.4 — CELL 73 FINAL COMPARISON")
print("=" * 74)

old = results_73["PHASE 4 EXPANDED"]
new = results_73["PHASE 4B CONTROLLED"]

print(
    "Trained-answer matches:",
    f"{old['trained_answer_matches']}/15",
    "->",
    f"{new['trained_answer_matches']}/15"
)

old_loss = old["validation_metrics"]["loss"]
new_loss = new["validation_metrics"]["loss"]

print(
    "Expanded validation loss:",
    f"{old_loss:.4f} -> {new_loss:.4f}"
)

print(
    "Expanded validation perplexity:",
    f"{old['validation_metrics']['perplexity']:.2f}",
    "->",
    f"{new['validation_metrics']['perplexity']:.2f}"
)

report_path = (
    phase4c_dir / "cell73_retention_audit.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        results_73,
        f,
        indent=2,
        ensure_ascii=False
    )

# Restore Phase 4B model for future work.
load_audit_model(new_path)

print("\nReport saved:", report_path)
print("Phase 4B checkpoint restored.")
print("\nCELL 73 COMPLETE!")


GRAC-LM v0.4 — CELL 73: CAPABILITY RETENTION AUDIT
Original expanded validation rows: 1031
Example validation keys: ['instruction', 'response', 'category']
Evaluation examples: 100

EVALUATING: PHASE 4 EXPANDED

[01/15]
QUESTION: Hello! How are you?
ANSWER: These are you are you are you are you are you are you are you are you are you are you are you are you are you are you are you're not you're not you're not a good person. You are you're not a good person. You are a good person. You are
TRAINED-ANSWER MATCH: False

[02/15]
QUESTION: Explain what a scientist does.
ANSWER: The scientists have been a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was a scientist who was
TRAINED-ANSWER MATCH: False

[03/15]
QUESTION: What is the difference between a cat and a dog?
ANSWER: The cat is a cat, a cat, a cat, a cat, a cat, a cat, 

In [ ]:

import json
import random
from pathlib import Path

print("=" * 72)
print("GRAC-LM v0.4 — CELL 74: MIXED-DATA PREPARATION")
print("=" * 72)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"
phase4c_dir.mkdir(parents=True, exist_ok=True)

general_train_path = (
    base_dir / "phase4/expanded_data/train.jsonl"
)
general_val_path = (
    base_dir / "phase4/expanded_data/validation.jsonl"
)
qa_train_path = (
    base_dir / "phase4b/data/train.jsonl"
)
qa_test_path = (
    base_dir / "phase4b/data/test.jsonl"
)

for path in [
    general_train_path,
    general_val_path,
    qa_train_path,
    qa_test_path
]:
    assert path.exists(), f"Missing: {path}"

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

general_train_raw = load_jsonl(general_train_path)
general_val_raw = load_jsonl(general_val_path)
qa_train_raw = load_jsonl(qa_train_path)
qa_test_raw = load_jsonl(qa_test_path)

def normalize_qa(row):
    question = (
        row.get("instruction")
        or row.get("question")
        or row.get("input")
    )
    answer = (
        row.get("response")
        or row.get("answer")
        or row.get("output")
    )

    if not isinstance(question, str):
        return None

    if not isinstance(answer, str):
        return None

    question = " ".join(question.split()).strip()
    answer = " ".join(answer.split()).strip()

    if not question or not answer:
        return None

    return {
        "instruction": question,
        "response": answer
    }

def prepare_dataset(rows, source):
    prepared = []

    for row in rows:
        item = normalize_qa(row)

        if item is None:
            continue

        prefix = (
            f"User: {item['instruction']} Assistant: "
        )

        prefix_ids = tokenizer.encode(prefix).ids
        answer_ids = tokenizer.encode(
            item["response"]
        ).ids

        if not answer_ids:
            continue

        if (
            len(prefix_ids)
            + len(answer_ids)
            + 1
            > GRAC_V04_CONFIG["context_length"]
        ):
            continue

        item["source"] = source
        prepared.append(item)

    return prepared

general_train = prepare_dataset(
    general_train_raw, "general"
)

general_val = prepare_dataset(
    general_val_raw, "general"
)

qa_train = prepare_dataset(
    qa_train_raw, "structured_qa"
)

qa_test = prepare_dataset(
    qa_test_raw, "structured_qa"
)

# Check for exact question overlap across splits.
train_questions = {
    row["instruction"].casefold()
    for row in general_train + qa_train
}

val_questions = {
    row["instruction"].casefold()
    for row in general_val + qa_test
}

overlap = train_questions & val_questions

print("Exact train/validation question overlaps:", len(overlap))

if overlap:
    print("WARNING: Overlapping questions detected.")
    print("Inspect before training.")

# Store separate pools rather than physically
# duplicating QA examples hundreds of times.
paths = {
    "general_train": phase4c_dir / "general_train.jsonl",
    "general_val": phase4c_dir / "general_val.jsonl",
    "qa_train": phase4c_dir / "qa_train.jsonl",
    "qa_test": phase4c_dir / "qa_test.jsonl"
}

datasets = {
    "general_train": general_train,
    "general_val": general_val,
    "qa_train": qa_train,
    "qa_test": qa_test
}

for name, path in paths.items():
    with open(path, "w", encoding="utf-8") as f:
        for row in datasets[name]:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

print("\nDATASET COUNTS")
print("General training:", len(general_train))
print("General validation:", len(general_val))
print("Structured QA training:", len(qa_train))
print("Structured QA held-out:", len(qa_test))

print("\nSAMPLING PLAN")
print("General instruction probability: 90%")
print("Structured QA probability: 10%")

print("\nFILES SAVED:")
for name, path in paths.items():
    print(name, "->", path)

print("\nCELL 74 COMPLETE!")


GRAC-LM v0.4 — CELL 74: MIXED-DATA PREPARATION
Exact train/validation question overlaps: 22
Inspect before training.

DATASET COUNTS
General training: 9276
General validation: 1031
Structured QA training: 36
Structured QA held-out: 12

SAMPLING PLAN
General instruction probability: 90%
Structured QA probability: 10%

FILES SAVED:
general_train -> /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4c/general_train.jsonl
general_val -> /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4c/general_val.jsonl
qa_train -> /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4c/qa_train.jsonl
qa_test -> /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4c/qa_test.jsonl

CELL 74 COMPLETE!


In [ ]:

import torch
import torch.nn.functional as F
import random
import json
import math
import gc
from pathlib import Path

print("=" * 74)
print("GRAC-LM v0.4 — CELL 75: MIXED-DATA FINE-TUNING")
print("=" * 74)

# ==================================================
# 1. CONFIGURATION
# ==================================================

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"
phase4c_dir.mkdir(parents=True, exist_ok=True)

source_path = (
    base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt"
)

assert source_path.exists(), "Phase 4 checkpoint missing!"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable Colab GPU runtime!"

CFG75 = {
    "max_steps": 600,
    "micro_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-5,
    "weight_decay": 0.01,
    "qa_probability": 0.10,
    "validation_every": 50,
    "general_val_samples": 100,
    "patience": 4,
    "max_general_loss_increase": 0.15,
    "seed": 2026
}

random.seed(CFG75["seed"])
torch.manual_seed(CFG75["seed"])
torch.cuda.manual_seed_all(CFG75["seed"])

print("GPU:", torch.cuda.get_device_name(0))
print("Source:", source_path)

# ==================================================
# 2. LOAD DATA
# ==================================================

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

general_train = read_jsonl(
    phase4c_dir / "general_train.jsonl"
)

general_val = read_jsonl(
    phase4c_dir / "general_val.jsonl"
)

qa_train = read_jsonl(
    phase4c_dir / "qa_train.jsonl"
)

qa_test = read_jsonl(
    phase4c_dir / "qa_test.jsonl"
)

def normalize_question(text):
    return " ".join(text.casefold().split())

# Prevent exact-question overlap across all splits.
reserved_questions = {
    normalize_question(row["instruction"])
    for row in general_val + qa_test
}

# Also prevent the general pool from duplicating
# structured QA questions with conflicting answers.
qa_questions = {
    normalize_question(row["instruction"])
    for row in qa_train
}

reserved_questions.update(qa_questions)

original_count = len(general_train)

general_train = [
    row for row in general_train
    if normalize_question(row["instruction"])
    not in reserved_questions
]

print("\nLEAKAGE CONTROL")
print("Original general training:", original_count)
print("Cleaned general training:", len(general_train))
print("Removed:", original_count - len(general_train))

assert len(general_train) > 0
assert len(qa_train) == 36

# ==================================================
# 3. TOKENIZATION
# ==================================================

eos_id = tokenizer.token_to_id("<eos>")
context_limit = GRAC_V04_CONFIG["context_length"]

def tokenize_qa(row):
    question = " ".join(row["instruction"].split())
    answer = " ".join(row["response"].split())

    prefix_ids = tokenizer.encode(
        f"User: {question} Assistant: "
    ).ids

    answer_ids = tokenizer.encode(answer).ids

    ids = prefix_ids + answer_ids + [eos_id]

    if len(ids) > context_limit:
        return None

    x = ids[:-1]
    y = ids[1:]

    y[:len(prefix_ids)-1] = (
        [-100] * (len(prefix_ids)-1)
    )

    return {
        "input_ids": x,
        "labels": y
    }

def tokenize_pool(rows):
    return [
        item
        for row in rows
        if (item := tokenize_qa(row)) is not None
    ]

train_general = tokenize_pool(general_train)
train_qa = tokenize_pool(qa_train)

# The QA paraphrases are development examples.
dev_qa = tokenize_pool(qa_test)

# Deterministic general validation subset.
rng_val = random.Random(2026)
selected_general_val = rng_val.sample(
    general_val,
    min(CFG75["general_val_samples"], len(general_val))
)

dev_general = tokenize_pool(selected_general_val)

print("\nTOKENIZED DATA")
print("General train:", len(train_general))
print("QA train:", len(train_qa))
print("General validation:", len(dev_general))
print("QA development:", len(dev_qa))

assert train_general and train_qa and dev_general and dev_qa

# ==================================================
# 4. LOAD ORIGINAL PHASE 4 MODEL
# ==================================================

# A fresh model instance prevents accidentally
# modifying the existing Phase 4B model object.
model_75 = GRACLM(GRAC_V04_CONFIG).to(device)

checkpoint = torch.load(
    source_path,
    map_location="cpu",
    weights_only=False
)

model_75.load_state_dict(
    checkpoint["model_state_dict"]
)

del checkpoint
gc.collect()

model_75.train()

optimizer_75 = torch.optim.AdamW(
    model_75.parameters(),
    lr=CFG75["learning_rate"],
    weight_decay=CFG75["weight_decay"]
)

scaler_75 = torch.amp.GradScaler("cuda")

pad_id = eos_id
rng_train = random.Random(CFG75["seed"])

# ==================================================
# 5. BATCHING
# ==================================================

def make_batch(pool, batch_size):
    samples = rng_train.choices(pool, k=batch_size)

    max_len = max(
        len(row["input_ids"])
        for row in samples
    )

    xs, ys = [], []

    for row in samples:
        padding = max_len - len(row["input_ids"])

        xs.append(
            row["input_ids"] + [pad_id] * padding
        )

        ys.append(
            row["labels"] + [-100] * padding
        )

    return (
        torch.tensor(xs, dtype=torch.long, device=device),
        torch.tensor(ys, dtype=torch.long, device=device)
    )

def compute_loss(x, y):
    logits, _ = model_75(x)

    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)).float(),
        y.reshape(-1),
        ignore_index=-100
    )

# ==================================================
# 6. VALIDATION
# ==================================================

@torch.inference_mode()
def evaluate_pool(pool):
    model_75.eval()

    total_loss = 0.0
    total_tokens = 0

    for row in pool:
        x = torch.tensor(
            [row["input_ids"]],
            dtype=torch.long,
            device=device
        )

        y = torch.tensor(
            [row["labels"]],
            dtype=torch.long,
            device=device
        )

        logits, _ = model_75(x)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            y.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        valid_tokens = (y != -100).sum().item()

        total_loss += loss_sum.item()
        total_tokens += valid_tokens

    model_75.train()
    return total_loss / total_tokens

# ==================================================
# 7. BASELINE VALIDATION
# ==================================================

print("\nMEASURING BASELINE...")

baseline_general = evaluate_pool(dev_general)
baseline_qa = evaluate_pool(dev_qa)

print(f"General baseline loss: {baseline_general:.4f}")
print(f"QA baseline loss: {baseline_qa:.4f}")

general_loss_limit = (
    baseline_general
    + CFG75["max_general_loss_increase"]
)

# Choose checkpoints by QA improvement while
# respecting a general-language loss constraint.
best_qa_loss = float("inf")
best_step = None
bad_checks = 0

best_path = (
    phase4c_dir / "GRAC-LM-v0.4-phase4c-best.pt"
)

history = []

# ==================================================
# 8. TRAINING
# ==================================================

print("\nMIXED-DATA TRAINING STARTED!\n")

for step in range(1, CFG75["max_steps"] + 1):
    model_75.train()
    optimizer_75.zero_grad(set_to_none=True)

    train_loss_sum = 0.0

    for _ in range(
        CFG75["gradient_accumulation_steps"]
    ):
        use_qa = (
            rng_train.random()
            < CFG75["qa_probability"]
        )

        pool = train_qa if use_qa else train_general

        x, y = make_batch(
            pool,
            CFG75["micro_batch_size"]
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss = compute_loss(x, y)

        train_loss_sum += loss.item()

        scaler_75.scale(
            loss / CFG75["gradient_accumulation_steps"]
        ).backward()

    scaler_75.unscale_(optimizer_75)

    torch.nn.utils.clip_grad_norm_(
        model_75.parameters(),
        1.0
    )

    scaler_75.step(optimizer_75)
    scaler_75.update()

    if step % CFG75["validation_every"] == 0:
        general_loss = evaluate_pool(dev_general)
        qa_loss = evaluate_pool(dev_qa)

        avg_train_loss = (
            train_loss_sum /
            CFG75["gradient_accumulation_steps"]
        )

        allowed = general_loss <= general_loss_limit

        improved = (
            allowed and qa_loss < best_qa_loss
        )

        if improved:
            best_qa_loss = qa_loss
            best_step = step
            bad_checks = 0

            torch.save(
                {
                    "model_state_dict": model_75.state_dict(),
                    "step": step,
                    "config": GRAC_V04_CONFIG,
                    "source_checkpoint": str(source_path),
                    "training_config": CFG75,
                    "general_val_loss": general_loss,
                    "qa_dev_loss": qa_loss
                },
                best_path
            )
        else:
            bad_checks += 1

        history.append({
            "step": step,
            "train_loss": avg_train_loss,
            "general_val_loss": general_loss,
            "qa_dev_loss": qa_loss,
            "within_general_loss_limit": allowed,
            "saved_best": improved
        })

        print(
            f"Step {step:3d} | "
            f"Train {avg_train_loss:.4f} | "
            f"General {general_loss:.4f} | "
            f"QA {qa_loss:.4f} | "
            f"Allowed {allowed} | "
            f"Best {improved}",
            flush=True
        )

        if bad_checks >= CFG75["patience"]:
            print("\nEARLY STOPPING TRIGGERED")
            break

# ==================================================
# 9. SAVE HISTORY AND FINAL CHECKPOINT
# ==================================================

final_path = (
    phase4c_dir / "GRAC-LM-v0.4-phase4c-final.pt"
)

torch.save(
    {
        "model_state_dict": model_75.state_dict(),
        "step": step,
        "config": GRAC_V04_CONFIG,
        "source_checkpoint": str(source_path),
        "training_config": CFG75
    },
    final_path
)

history_path = phase4c_dir / "cell75_training_history.json"

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "baseline_general_loss": baseline_general,
            "baseline_qa_loss": baseline_qa,
            "general_loss_limit": general_loss_limit,
            "best_step": best_step,
            "history": history
        },
        f,
        indent=2
    )

print("\n" + "=" * 74)
print("CELL 75 TRAINING REPORT")
print("=" * 74)

print("Baseline general loss:", round(baseline_general, 4))
print("General loss limit:", round(general_loss_limit, 4))
print("Best step:", best_step)
print("Best QA development loss:", (
    round(best_qa_loss, 4)
    if best_step is not None else "No qualifying checkpoint"
))
print("Final checkpoint:", final_path)
print("Best checkpoint:", (
    best_path if best_step is not None else "None"
))
print("History:", history_path)

print("\nCELL 75 COMPLETE!")


GRAC-LM v0.4 — CELL 75: MIXED-DATA FINE-TUNING
GPU: Tesla T4
Source: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4/GRAC-LM-v0.4-expanded-best.pt

LEAKAGE CONTROL
Original general training: 9276
Cleaned general training: 9225
Removed: 51

TOKENIZED DATA
General train: 9225
QA train: 36
General validation: 100
QA development: 12

MEASURING BASELINE...
General baseline loss: 3.7415
QA baseline loss: 3.3841

MIXED-DATA TRAINING STARTED!

Step  50 | Train 2.7808 | General 3.7482 | QA 2.2910 | Allowed True | Best True
Step 100 | Train 3.0940 | General 3.7458 | QA 1.5313 | Allowed True | Best True
Step 150 | Train 2.9721 | General 3.7494 | QA 1.1107 | Allowed True | Best True
Step 200 | Train 3.7029 | General 3.7385 | QA 1.0041 | Allowed True | Best True
Step 250 | Train 2.8563 | General 3.7370 | QA 0.7511 | Allowed True | Best True
Step 300 | Train 3.6704 | General 3.7317 | QA 0.6270 | Allowed True | Best True
Step 350 | Train 2.6789 | General 3.7280 | QA 0.5131 | Allowed True | Best True


In [ ]:

import torch
import torch.nn.functional as F
import json
import math
import re
from pathlib import Path

print("=" * 74)
print("GRAC-LM v0.4 — CELL 76: THREE-MODEL COMPARISON")
print("=" * 74)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"

checkpoints = {
    "PHASE 4 EXPANDED":
        base_dir / "phase4/GRAC-LM-v0.4-expanded-best.pt",
    "PHASE 4B CONTROLLED":
        base_dir / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt",
    "PHASE 4C MIXED":
        phase4c_dir / "GRAC-LM-v0.4-phase4c-best.pt"
}

history_path = phase4c_dir / "cell75_training_history.json"

assert history_path.exists(), "Cell 75 history missing!"

with open(history_path, encoding="utf-8") as f:
    training_report = json.load(f)

print("Cell 75 best step:", training_report.get("best_step"))
print("Baseline general loss:",
      training_report.get("baseline_general_loss"))
print("General loss limit:",
      training_report.get("general_loss_limit"))

if training_report.get("best_step") is None:
    raise RuntimeError(
        "Cell 75 did not produce a qualifying best checkpoint. "
        "Inspect its validation history before proceeding."
    )

for name, path in checkpoints.items():
    assert path.exists(), f"Missing checkpoint: {path}"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

eos_id = tokenizer.token_to_id("<eos>")
context_limit = GRAC_V04_CONFIG["context_length"]

# Fresh evaluation model; does not modify Cell 75 model.
eval_76 = GRACLM(GRAC_V04_CONFIG).to(device)
eval_76.eval()

def load_checkpoint_76(path):
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )
    eval_76.load_state_dict(ckpt["model_state_dict"])
    eval_76.eval()
    del ckpt

def read_jsonl_76(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

general_rows = read_jsonl_76(
    phase4c_dir / "general_val.jsonl"
)

qa_rows = read_jsonl_76(
    phase4c_dir / "qa_test.jsonl"
)

# Match Cell 75's deterministic validation selection.
import random
rng = random.Random(2026)

selected_general = rng.sample(
    general_rows,
    min(100, len(general_rows))
)

def tokenize_eval(row):
    question = " ".join(row["instruction"].split())
    answer = " ".join(row["response"].split())

    prefix = tokenizer.encode(
        f"User: {question} Assistant: "
    ).ids

    answer_ids = tokenizer.encode(answer).ids
    ids = prefix + answer_ids + [eos_id]

    if len(ids) > context_limit:
        return None

    labels = ids[1:]
    labels[:len(prefix)-1] = [-100] * (len(prefix)-1)

    return {
        "x": ids[:-1],
        "y": labels
    }

def tokenize_rows(rows):
    return [
        item
        for row in rows
        if (item := tokenize_eval(row)) is not None
    ]

general_eval = tokenize_rows(selected_general)
qa_eval = tokenize_rows(qa_rows)

@torch.inference_mode()
def evaluate_loss_76(dataset):
    total_nll = 0.0
    total_tokens = 0

    for row in dataset:
        x = torch.tensor(
            [row["x"]],
            dtype=torch.long,
            device=device
        )
        y = torch.tensor(
            [row["y"]],
            dtype=torch.long,
            device=device
        )

        logits, _ = eval_76(x)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            y.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_nll += loss.item()
        total_tokens += (y != -100).sum().item()

    return total_nll / total_tokens

@torch.inference_mode()
def generate_76(question, max_tokens=65):
    question = " ".join(question.split())
    ids = tokenizer.encode(
        f"User: {question} Assistant: "
    ).ids

    generated = []

    for _ in range(max_tokens):
        if len(ids) >= context_limit:
            break

        x = torch.tensor(
            [ids], dtype=torch.long, device=device
        )
        logits, _ = eval_76(x)
        next_id = int(logits[0, -1].argmax().item())

        if next_id == eos_id:
            break

        ids.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

test_questions = [
    ("What is the capital of India?", "New Delhi."),
    ("What is the capital of France?", "Paris."),
    ("What is 2 + 3?", "5."),
    ("What is gravity?",
     "Gravity is the attraction between objects with mass."),
    ("What is photosynthesis?",
     "Plants use sunlight to make food."),
    ("What is the capital of Germany?", "Berlin."),
    ("What is 7 + 8?", "15."),
    ("What is the chemical formula of water?", "H2O."),
    ("Why do people need sleep?", None),
    ("Explain what a scientist does.", None),
    ("How does a bicycle work?", None),
    ("What does friendship mean?", None),
    ("Describe a healthy breakfast.", None)
]

trained_answers = {
    re.sub(r"[^\w\s]", "", fact["answer"].casefold()).strip()
    for fact in qa_facts
}

def normalize_76(text):
    return " ".join(
        re.sub(r"[^\w\s]", "", text.casefold()).split()
    )

results_76 = {}

for name, path in checkpoints.items():
    print("\n" + "=" * 74)
    print("MODEL:", name)
    print("=" * 74)

    load_checkpoint_76(path)

    general_loss = evaluate_loss_76(general_eval)
    qa_loss = evaluate_loss_76(qa_eval)

    records = []
    collapse_count = 0

    for question, expected in test_questions:
        answer = generate_76(question)

        collapsed = (
            expected is None
            and normalize_76(answer) in trained_answers
        )

        collapse_count += int(collapsed)

        records.append({
            "question": question,
            "expected": expected,
            "generated": answer,
            "unrelated_trained_answer": collapsed
        })

        print("\nQ:", question)
        print("A:", answer or "[EMPTY]")

    results_76[name] = {
        "general_val_loss": general_loss,
        "qa_dev_loss": qa_loss,
        "general_perplexity": math.exp(general_loss),
        "unrelated_trained_answer_count": collapse_count,
        "records": records
    }

    print("\nGENERAL LOSS:", round(general_loss, 4))
    print("QA DEV LOSS:", round(qa_loss, 4))
    print("GENERAL PERPLEXITY:",
          round(math.exp(general_loss), 2))
    print("UNRELATED TRAINED-ANSWER MATCHES:",
          f"{collapse_count}/5")

print("\n" + "=" * 74)
print("CELL 76 FINAL COMPARISON")
print("=" * 74)

for name, result in results_76.items():
    print(
        f"{name:23s} | "
        f"General {result['general_val_loss']:.4f} | "
        f"QA {result['qa_dev_loss']:.4f} | "
        f"Collapse {result['unrelated_trained_answer_count']}/5"
    )

output_path = phase4c_dir / "cell76_comparison.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results_76, f, indent=2, ensure_ascii=False)

print("\nReport saved:", output_path)
print("CELL 76 COMPLETE!")


GRAC-LM v0.4 — CELL 76: THREE-MODEL COMPARISON
Cell 75 best step: 600
Baseline general loss: 3.741518138173898
General loss limit: 3.8915181381738977

MODEL: PHASE 4 EXPANDED

Q: What is the capital of India?
A: The capital of India is the capital of India. The capital of India is the capital of India. The capital of India is the capital of India.

Q: What is the capital of France?
A: The capital of France is the capital of France.

Q: What is 2 + 3?
A: 1 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3 + 3

Q: What is gravity?
A: The gravity is a very high quality of gravity. It is a great way to be a great way to get a good way to get a good way to get a good way to get a good way. It is a great way to get a good way to get a good way to get a good way

Q: What is photosynthesis?
A: Michosynthesis is a species of photosynthesis.

Q: What is the capital of Germany?
A: The capital of Germany is the capital of Ge

In [ ]:

import torch
import json
import re
from pathlib import Path
from collections import defaultdict

print("=" * 76)
print("GRAC-LM v0.4 — CELL 77: GENERALIZATION & FAILURE ANALYSIS")
print("=" * 76)

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4c_dir = base_dir / "phase4c"

checkpoints_77 = {
    "PHASE 4B": (
        base_dir /
        "phase4b/GRAC-LM-v0.4-phase4b-step300.pt"
    ),
    "PHASE 4C": (
        phase4c_dir /
        "GRAC-LM-v0.4-phase4c-best.pt"
    )
}

for name, path in checkpoints_77.items():
    assert path.exists(), f"Missing checkpoint: {path}"

assert "GRACLM" in globals()
assert "GRAC_V04_CONFIG" in globals()
assert "tokenizer" in globals()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

context_limit = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")

assert eos_id is not None

# Reuse one model object to avoid excessive GPU memory.
if "eval_76" in globals():
    model_77 = eval_76
elif "model_75" in globals():
    model_77 = model_75
else:
    model_77 = GRACLM(GRAC_V04_CONFIG).to(device)

model_77.to(device)
model_77.eval()

def load_77(path):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model_77.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model_77.eval()
    del checkpoint

# ============================================================
# 2. LOAD THE ORIGINAL 24 CELL 72 QUESTIONS
# ============================================================

cell72_path = (
    phase4c_dir / "cell72_stress_results.json"
)

assert cell72_path.exists(), (
    "Cell 72 report missing: " + str(cell72_path)
)

with open(cell72_path, encoding="utf-8") as f:
    cell72_report = json.load(f)

# Recover the questions and expected answers from
# the previously saved Cell 72 report.
source_records = (
    cell72_report["PHASE 4B CONTROLLED"]["records"]
)

original_tests = []

for row in source_records:
    original_tests.append({
        "category": row["category"],
        "question": row["question"],
        "expected": row["expected"],
        "origin": "cell72_reused"
    })

assert len(original_tests) == 24, (
    f"Expected 24 Cell 72 questions, got {len(original_tests)}"
)

# ============================================================
# 3. ADD 20 NEW DIAGNOSTIC QUESTIONS
# ============================================================

new_tests = [

    # E: TEN NEW PARAPHRASES OF KNOWN FACTS
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "In which city is India's central government located?",
        "expected": "New Delhi."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "Where is the French capital situated?",
        "expected": "Paris."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "Which city is the political capital of Japan?",
        "expected": "Tokyo."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "If I have two pencils and get two more, how many pencils do I have?",
        "expected": "4."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "What number results from adding three to two?",
        "expected": "5."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "What do you get when three is added to three?",
        "expected": "6."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "Explain the biological process by which plants make their food.",
        "expected": "Plants use sunlight to make food."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "What makes objects with mass pull on each other?",
        "expected": "Gravity is the attraction between objects with mass."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "What type of device processes information electronically?",
        "expected": "A computer is an electronic device that processes data."
    },
    {
        "category": "E_NEW_PARAPHRASE",
        "question": "Tell me the name of the AI system I am speaking to.",
        "expected": "My name is GRAC."
    },

    # F: TEN ADDITIONAL UNSEEN FACTS
    {
        "category": "F_NEW_FACT",
        "question": "What is the capital of Spain?",
        "expected": "Madrid."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is the capital of Canada?",
        "expected": "Ottawa."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is 9 + 6?",
        "expected": "15."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is 12 - 5?",
        "expected": "7."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is 4 multiplied by 3?",
        "expected": "12."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What gas do humans need for breathing?",
        "expected": "Oxygen."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is the closest star to Earth?",
        "expected": "The Sun."
    },
    {
        "category": "F_NEW_FACT",
        "question": "Which organ pumps blood around the human body?",
        "expected": "The heart."
    },
    {
        "category": "F_NEW_FACT",
        "question": "How many days are there in a week?",
        "expected": "7."
    },
    {
        "category": "F_NEW_FACT",
        "question": "What is the freezing point of water in degrees Celsius?",
        "expected": "0 degrees Celsius."
    }
]

tests_77 = original_tests + [
    {**test, "origin": "cell77_new"}
    for test in new_tests
]

assert len(tests_77) == 44

print("Total questions:", len(tests_77))
print("Cell 72 reused questions:", len(original_tests))
print("New diagnostic questions:", len(new_tests))

# ============================================================
# 4. GENERATION AND SCORING
# ============================================================

def normalize_77(text):
    text = text.casefold().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

@torch.inference_mode()
def generate_77(question, max_new_tokens=65):
    question = " ".join(question.split()).strip()

    ids = tokenizer.encode(
        f"User: {question} Assistant: "
    ).ids

    generated = []

    for _ in range(max_new_tokens):
        if len(ids) >= context_limit:
            break

        x = torch.tensor(
            [ids],
            dtype=torch.long,
            device=device
        )

        logits, _ = model_77(x)

        next_id = int(
            torch.argmax(logits[0, -1]).item()
        )

        if next_id == eos_id:
            break

        ids.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# Exact-match accuracy is strict:
# a correct answer with different wording can be
# marked wrong, so inspect the raw responses too.
def is_match_77(generated, expected):
    return normalize_77(generated) == normalize_77(expected)

# Detect exact reuse of the 12 trained answers.
assert "qa_facts" in globals(), (
    "qa_facts missing — run Cell 68 first"
)

trained_answers_77 = {
    normalize_77(fact["answer"])
    for fact in qa_facts
}

# ============================================================
# 5. EVALUATE BOTH MODELS
# ============================================================

results_77 = {}

for model_name, checkpoint_path in checkpoints_77.items():

    print("\n" + "=" * 76)
    print("EVALUATING:", model_name)
    print("=" * 76)

    load_77(checkpoint_path)

    records = []
    category_stats = defaultdict(
        lambda: {"correct": 0, "total": 0}
    )

    for index, test in enumerate(tests_77, 1):
        question = test["question"]
        expected = test["expected"]
        category = test["category"]

        generated = generate_77(question)

        correct = is_match_77(
            generated, expected
        )

        reused_answer = (
            normalize_77(generated) in trained_answers_77
        )

        category_stats[category]["total"] += 1
        category_stats[category]["correct"] += int(correct)

        records.append({
            **test,
            "generated": generated,
            "exact_match": correct,
            "trained_answer_reused": reused_answer
        })

        print(
            f"[{index:02d}/{len(tests_77)}] "
            f"{category} | "
            f"{'MATCH' if correct else 'NO MATCH'}"
        )
        print("Q:", question)
        print("A:", generated or "[EMPTY]")
        print("EXPECTED:", expected)
        print()

    results_77[model_name] = {
        "records": records,
        "category_stats": dict(category_stats)
    }

# ============================================================
# 6. COMPARISON REPORT
# ============================================================

print("\n" + "=" * 76)
print("GRAC-LM v0.4 — CELL 77 FINAL COMPARISON")
print("=" * 76)

categories = [
    "A_HARD_PARAPHRASE",
    "B_UNSEEN_FACT",
    "C_DISTRACTOR",
    "D_QUESTION_SWITCH",
    "E_NEW_PARAPHRASE",
    "F_NEW_FACT"
]

for category in categories:
    print("\n" + category)

    for model_name in checkpoints_77:
        stat = results_77[model_name]["category_stats"][category]

        print(
            f"  {model_name}: "
            f"{stat['correct']}/{stat['total']}"
        )

print("\nTOTAL SCORES")

for model_name in checkpoints_77:
    records = results_77[model_name]["records"]

    correct = sum(
        int(row["exact_match"])
        for row in records
    )

    reused_wrong = sum(
        int(
            row["trained_answer_reused"]
            and not row["exact_match"]
        )
        for row in records
    )

    print(
        f"{model_name}: "
        f"{correct}/{len(records)} "
        f"({100 * correct / len(records):.1f}%) | "
        f"Wrong trained-answer reuse: {reused_wrong}"
    )

# ============================================================
# 7. SAVE RESULTS
# ============================================================

report_path = (
    phase4c_dir / "cell77_generalization_results.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        results_77,
        f,
        indent=2,
        ensure_ascii=False
    )

# Restore Phase 4C model for the next cell.
load_77(checkpoints_77["PHASE 4C"])

print("\nReport saved:", report_path)
print("Phase 4C checkpoint restored.")
print("\nCELL 77 COMPLETE!")


GRAC-LM v0.4 — CELL 77: GENERALIZATION & FAILURE ANALYSIS
Total questions: 44
Cell 72 reused questions: 24
New diagnostic questions: 20

EVALUATING: PHASE 4B
[01/44] A_HARD_PARAPHRASE | MATCH
Q: India's national government is headquartered in which capital city?
A: New Delhi.
EXPECTED: New Delhi.

[02/44] A_HARD_PARAPHRASE | MATCH
Q: Identify the French city where the country's national government is based.
A: Paris.
EXPECTED: Paris.

[03/44] A_HARD_PARAPHRASE | NO MATCH
Q: Which Japanese city functions as the nation's capital?
A: New Delhi.
EXPECTED: Tokyo.

[04/44] A_HARD_PARAPHRASE | NO MATCH
Q: Two objects plus another three objects make how many objects?
A: Astero.
EXPECTED: 5.

[05/44] A_HARD_PARAPHRASE | MATCH
Q: Which natural process allows green plants to produce sugars using sunlight?
A: Plants use sunlight to make food.
EXPECTED: Plants use sunlight to make food.

[06/44] A_HARD_PARAPHRASE | MATCH
Q: What physical interaction causes massive bodies to attract each other?
A: G

In [ ]:

import json
import random
import re
from pathlib import Path
from collections import Counter

print("=" * 76)
print("GRAC-LM v0.4 — CELL 78: PHASE 4D DATASET CONSTRUCTION")
print("=" * 76)

SEED = 2026
rng = random.Random(SEED)

base_dir = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
phase4d_dir = base_dir / "phase4d"
data_dir = phase4d_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

assert "tokenizer" in globals()
assert "GRAC_V04_CONFIG" in globals()

context_limit = GRAC_V04_CONFIG["context_length"]
eos_id = tokenizer.token_to_id("<eos>")
unk_id = tokenizer.token_to_id("<unk>")

# ============================================================
# 1. FACT BANK
# ============================================================

# Every fact has a unique group ID.
# All question variations of that fact stay in one split.

facts = []

capitals = {
    "India": "New Delhi",
    "France": "Paris",
    "Japan": "Tokyo",
    "Germany": "Berlin",
    "Italy": "Rome",
    "Spain": "Madrid",
    "Canada": "Ottawa",
    "Australia": "Canberra",
    "Brazil": "Brasilia",
    "China": "Beijing",
    "Russia": "Moscow",
    "Egypt": "Cairo",
    "Thailand": "Bangkok",
    "South Korea": "Seoul",
    "Argentina": "Buenos Aires",
    "Mexico": "Mexico City",
    "Portugal": "Lisbon",
    "Greece": "Athens",
    "Norway": "Oslo",
    "Sweden": "Stockholm"
}

for country, capital in capitals.items():
    facts.append({
        "id": "capital_" + country.lower().replace(" ", "_"),
        "category": "geography",
        "subject": country,
        "answer": capital + ".",
        "kind": "capital"
    })

science_facts = [
    ("water_formula", "What is the chemical formula of water?", "H2O."),
    ("oxygen", "What gas do humans need for breathing?", "Oxygen."),
    ("red_planet", "Which planet is known as the Red Planet?", "Mars."),
    ("closest_star", "What is the closest star to Earth?", "The Sun."),
    ("heart", "Which organ pumps blood around the body?", "The heart."),
    ("photosynthesis", "What is photosynthesis?", "Plants use sunlight to make food."),
    ("gravity", "What is gravity?", "Gravity is the attraction between objects with mass."),
    ("evaporation", "What is evaporation?", "Evaporation is the change of liquid into gas."),
    ("condensation", "What is condensation?", "Condensation is the change of gas into liquid."),
    ("freezing", "What is freezing?", "Freezing is the change of liquid into solid."),
    ("melting", "What is melting?", "Melting is the change of solid into liquid."),
    ("electric_current", "What is electric current?", "Electric current is the rate of flow of electric charge."),
    ("voltage", "What is voltage?", "Voltage is the potential difference between two points."),
    ("resistance", "What is electrical resistance?", "Resistance opposes the flow of electric current."),
    ("computer", "What is a computer?", "A computer is an electronic device that processes data."),
    ("artificial_intelligence", "What is artificial intelligence?", "Artificial intelligence is technology that performs tasks associated with human intelligence."),
    ("cpu", "What does CPU stand for?", "Central Processing Unit."),
    ("ram", "What does RAM stand for?", "Random Access Memory."),
    ("binary", "What is the binary number system?", "A number system using zero and one."),
    ("internet", "What is the internet?", "A global network of connected computers.")
]

for fact_id, question, answer in science_facts:
    facts.append({
        "id": fact_id,
        "category": "science_computing",
        "question": question,
        "answer": answer,
        "kind": "definition"
    })

# ============================================================
# 2. ARITHMETIC FACT BANK
# ============================================================

# Different operand pairs are separate groups.
# Commutative equivalents such as 2+3 and 3+2
# share a single group ID to prevent direct leakage.

for a in range(2, 21):
    for b in range(2, 21):
        if a > b:
            continue

        facts.append({
            "id": f"add_{a}_{b}",
            "category": "arithmetic",
            "kind": "addition",
            "a": a,
            "b": b,
            "answer": str(a + b) + "."
        })

for a in range(5, 26):
    for b in range(1, a):
        facts.append({
            "id": f"subtract_{a}_{b}",
            "category": "arithmetic",
            "kind": "subtraction",
            "a": a,
            "b": b,
            "answer": str(a - b) + "."
        })

for a in range(2, 13):
    for b in range(a, 13):
        facts.append({
            "id": f"multiply_{a}_{b}",
            "category": "arithmetic",
            "kind": "multiplication",
            "a": a,
            "b": b,
            "answer": str(a * b) + "."
        })

print("Unique fact/problem groups:", len(facts))
print("Group categories:", Counter(f["category"] for f in facts))

# ============================================================
# 3. SPLIT BY FACT/PROBLEM GROUP
# ============================================================

# Keep geography, science and arithmetic represented
# in all three splits.
by_category = {}

for fact in facts:
    by_category.setdefault(
        fact["category"], []
    ).append(fact)

split_facts = {
    "train": [],
    "validation": [],
    "test": []
}

for category, group in by_category.items():
    group = group.copy()
    rng.shuffle(group)

    n = len(group)
    n_train = int(n * 0.80)
    n_val = int(n * 0.10)

    split_facts["train"].extend(group[:n_train])
    split_facts["validation"].extend(
        group[n_train:n_train+n_val]
    )
    split_facts["test"].extend(
        group[n_train+n_val:]
    )

# ============================================================
# 4. GENERATE QUESTION VARIATIONS
# ============================================================

def make_variations(fact):
    kind = fact["kind"]
    answer = fact["answer"]

    if kind == "capital":
        country = fact["subject"]

        questions = [
            f"What is the capital of {country}?",
            f"Name the capital city of {country}.",
            f"Which city is the capital of {country}?",
            f"What city serves as {country}'s capital?",
            f"I was reading about geography. What is the capital of {country}?"
        ]

    elif kind == "addition":
        a, b = fact["a"], fact["b"]

        questions = [
            f"What is {a} + {b}?",
            f"Calculate {a} plus {b}.",
            f"What is the sum of {a} and {b}?",
            f"If you add {a} and {b}, what do you get?",
            f"Ignore the number 99. What is {a} + {b}?"
        ]

    elif kind == "subtraction":
        a, b = fact["a"], fact["b"]

        questions = [
            f"What is {a} - {b}?",
            f"Calculate {a} minus {b}.",
            f"What is the difference between {a} and {b}?",
            f"Subtract {b} from {a}.",
            f"Ignore the number 99. What is {a} - {b}?"
        ]

    elif kind == "multiplication":
        a, b = fact["a"], fact["b"]

        questions = [
            f"What is {a} times {b}?",
            f"Calculate {a} multiplied by {b}.",
            f"What is the product of {a} and {b}?",
            f"Multiply {a} by {b}.",
            f"Ignore the number 99. What is {a} times {b}?"
        ]

    elif kind == "definition":
        question = fact["question"]

        # Keep the original question and create
        # several controlled instruction variations.
        questions = [
            question,
            f"Please answer: {question}",
            f"Give a short answer. {question}",
            f"I am studying science and computing. {question}",
            f"Answer the following question clearly: {question}"
        ]

    else:
        raise ValueError(f"Unknown kind: {kind}")

    return [
        {
            "id": fact["id"],
            "category": fact["category"],
            "kind": kind,
            "instruction": question,
            "response": answer
        }
        for question in questions
    ]

# ============================================================
# 5. BUILD SPLIT DATASETS
# ============================================================

datasets_78 = {}

for split_name, groups in split_facts.items():
    rows = []

    for fact in groups:
        rows.extend(make_variations(fact))

    rng.shuffle(rows)
    datasets_78[split_name] = rows

# ============================================================
# 6. TOKENIZER AND LENGTH VALIDATION
# ============================================================

def check_row(row):
    prefix = (
        f"User: {row['instruction']} Assistant: "
    )

    prefix_ids = tokenizer.encode(prefix).ids
    answer_ids = tokenizer.encode(row["response"]).ids

    all_ids = prefix_ids + answer_ids + [eos_id]

    return {
        "length": len(all_ids),
        "has_unk": (
            unk_id is not None
            and unk_id in all_ids
        )
    }

validation_report = {}

for split_name, rows in datasets_78.items():
    valid_rows = []
    rejected = Counter()

    for row in rows:
        result = check_row(row)

        if result["length"] > context_limit:
            rejected["too_long"] += 1
            continue

        if result["has_unk"]:
            rejected["unknown_token"] += 1
            continue

        valid_rows.append(row)

    datasets_78[split_name] = valid_rows

    validation_report[split_name] = {
        "rows": len(valid_rows),
        "rejected": dict(rejected)
    }

# ============================================================
# 7. CHECK GROUP LEAKAGE
# ============================================================

split_ids = {
    name: {row["id"] for row in rows}
    for name, rows in datasets_78.items()
}

assert not (
    split_ids["train"] & split_ids["validation"]
), "Train/validation group leakage!"

assert not (
    split_ids["train"] & split_ids["test"]
), "Train/test group leakage!"

assert not (
    split_ids["validation"] & split_ids["test"]
), "Validation/test group leakage!"

# ============================================================
# 8. SAVE DATASETS
# ============================================================

for split_name, rows in datasets_78.items():
    path = data_dir / f"{split_name}.jsonl"

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(row, ensure_ascii=False)
                + "\n"
            )

metadata = {
    "seed": SEED,
    "total_fact_groups": len(facts),
    "split_group_counts": {
        name: len(groups)
        for name, groups in split_facts.items()
    },
    "validation_report": validation_report,
    "group_leakage": False,
    "notes": [
        "Groups are split before paraphrase generation.",
        "Arithmetic commutative duplicates are grouped.",
        "Validation and test facts are not explicitly taught in Phase 4D training.",
        "Earlier pretraining may contain some of these facts.",
        "Test set should remain untouched until final evaluation."
    ]
}

metadata_path = data_dir / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ============================================================
# 9. FINAL REPORT
# ============================================================

print("\n" + "=" * 76)
print("PHASE 4D DATASET REPORT")
print("=" * 76)

for split_name, rows in datasets_78.items():
    print(
        f"{split_name.upper():10s}: "
        f"{len(rows):5d} examples | "
        f"{len(split_ids[split_name]):4d} unique groups"
    )

    print(
        "  Rejected:",
        validation_report[split_name]["rejected"]
    )

print("\nSample training examples:")

for row in datasets_78["train"][:5]:
    print("Q:", row["instruction"])
    print("A:", row["response"])
    print()

print("Metadata saved:", metadata_path)
print("Group leakage: NONE")
print("\nCELL 78 COMPLETE!")


GRAC-LM v0.4 — CELL 78: PHASE 4D DATASET CONSTRUCTION
Unique fact/problem groups: 590
Group categories: Counter({'arithmetic': 550, 'geography': 20, 'science_computing': 20})

PHASE 4D DATASET REPORT
TRAIN     :  2360 examples |  472 unique groups
  Rejected: {}
VALIDATION:   295 examples |   59 unique groups
  Rejected: {}
TEST      :   295 examples |   59 unique groups
  Rejected: {}

Sample training examples:
Q: Calculate 5 plus 20.
A: 25.

Q: I was reading about geography. What is the capital of Norway?
A: Oslo.

Q: What is 9 + 17?
A: 26.

Q: What is the capital of Brazil?
A: Brasilia.

Q: Calculate 15 minus 8.
A: 7.

Metadata saved: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4d/data/metadata.json
Group leakage: NONE

CELL 78 COMPLETE!


In [ ]:

import torch
import torch.nn.functional as F
import json, random, math, gc
from pathlib import Path

print("=" * 72)
print("GRAC-LM v0.4 — CELL 79: PHASE 4D TRAINING")
print("=" * 72)

BASE = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
OUT = BASE / "phase4d"
DATA = OUT / "data"
OUT.mkdir(parents=True, exist_ok=True)

CFG79 = {
    "max_steps": 250,
    "micro_batch": 4,
    "accumulation": 4,
    "learning_rate": 1e-5,
    "qa_probability": 0.70,
    "validation_every": 25,
    "save_every": 50,
    "patience": 5,
    "general_loss_tolerance": 0.20,
    "seed": 2026
}

SOURCE = BASE / "phase4c/GRAC-LM-v0.4-phase4c-best.pt"
BEST = OUT / "GRAC-LM-v0.4-phase4d-best.pt"
RESUME = OUT / "GRAC-LM-v0.4-phase4d-resume.pt"
HISTORY = OUT / "cell79_history.json"

assert SOURCE.exists(), "Phase 4C checkpoint missing."
assert torch.cuda.is_available(), "Cell 79 requires CUDA GPU."

device = torch.device("cuda")
torch.manual_seed(CFG79["seed"])
random.seed(CFG79["seed"])
rng79 = random.Random(CFG79["seed"])

def read79(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

qa_train = read79(DATA / "train.jsonl")
qa_val_all = read79(DATA / "validation.jsonl")
general_train = read79(BASE / "phase4c/general_train.jsonl")
general_val_all = read79(BASE / "phase4c/general_val.jsonl")

# Keep validation quick but representative:
# one variation per held-out Phase 4D group.
qa_val_by_id = {}
for row in qa_val_all:
    qa_val_by_id.setdefault(row["id"], row)
qa_val = list(qa_val_by_id.values())

# Fixed general validation subset for every checkpoint.
general_val = random.Random(2026).sample(
    general_val_all,
    min(60, len(general_val_all))
)

eos = tokenizer.token_to_id("<eos>")
limit = GRAC_V04_CONFIG["context_length"]

def encode79(row):
    question = " ".join(row["instruction"].split())
    answer = " ".join(row["response"].split())

    prefix = tokenizer.encode(
        f"User: {question} Assistant: "
    ).ids
    answer_ids = tokenizer.encode(answer).ids
    ids = prefix + answer_ids + [eos]

    if len(ids) > limit:
        return None

    x = ids[:-1]
    y = ids[1:].copy()
    y[:len(prefix)-1] = [-100] * (len(prefix)-1)

    return (x, y)

def prepare79(rows):
    return [
        result
        for row in rows
        if (result := encode79(row)) is not None
    ]

train_qa = prepare79(qa_train)
train_general = prepare79(general_train)
val_qa = prepare79(qa_val)
val_general = prepare79(general_val)

assert all([
    train_qa, train_general, val_qa, val_general
])

print("QA training:", len(train_qa))
print("General training:", len(train_general))
print("QA validation groups:", len(val_qa))
print("General validation examples:", len(val_general))

# ------------------------------------------------------------
# Batch construction
# ------------------------------------------------------------

def batch79(rows):
    chosen = [
        rng79.choice(rows)
        for _ in range(CFG79["micro_batch"])
    ]

    max_len = max(len(x) for x, y in chosen)

    bx, by = [], []

    for x, y in chosen:
        pad = max_len - len(x)
        bx.append(x + [eos] * pad)
        by.append(y + [-100] * pad)

    return (
        torch.tensor(bx, dtype=torch.long, device=device),
        torch.tensor(by, dtype=torch.long, device=device)
    )

# ------------------------------------------------------------
# Model and optimizer
# ------------------------------------------------------------

model79 = GRACLM(GRAC_V04_CONFIG).to(device)

optimizer79 = torch.optim.AdamW(
    model79.parameters(),
    lr=CFG79["learning_rate"],
    weight_decay=0.01
)

scaler79 = torch.amp.GradScaler("cuda")

start_step = 0
best_step = None
best_qa = float("inf")
bad_checks = 0
history79 = []

# Resume automatically if an interrupted run exists.
if RESUME.exists():
    state = torch.load(
        RESUME, map_location="cpu", weights_only=False
    )
    assert state["config"] == CFG79, (
        "Resume configuration differs. "
        "Restore the original CFG79 or archive the resume file."
    )

    model79.load_state_dict(state["model"])
    optimizer79.load_state_dict(state["optimizer"])
    scaler79.load_state_dict(state["scaler"])

    start_step = state["step"]
    best_step = state["best_step"]
    best_qa = state["best_qa"]
    bad_checks = state["bad_checks"]
    history79 = state["history"]

    rng79.setstate(state["rng_state"])
    torch.set_rng_state(state["torch_rng"])
    torch.cuda.set_rng_state_all(state["cuda_rng"])

    baseline_general = state["baseline_general"]
    general_limit = state["general_limit"]

    print("RESUMED from step:", start_step)

else:
    state = torch.load(
        SOURCE, map_location="cpu", weights_only=False
    )
    model79.load_state_dict(state["model_state_dict"])
    del state

# ------------------------------------------------------------
# Validation: response-token-weighted loss
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate79(rows):
    model79.eval()
    total_loss = 0.0
    total_tokens = 0

    for x, y in rows:
        tx = torch.tensor(
            [x], dtype=torch.long, device=device
        )
        ty = torch.tensor(
            [y], dtype=torch.long, device=device
        )

        logits, _ = model79(tx)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            ty.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_loss += loss.item()
        total_tokens += (ty != -100).sum().item()

    return total_loss / max(total_tokens, 1)

if start_step == 0:
    print("\nEvaluating Phase 4C starting checkpoint...")
    baseline_general = evaluate79(val_general)
    baseline_qa = evaluate79(val_qa)
    general_limit = (
        baseline_general +
        CFG79["general_loss_tolerance"]
    )

    print("Baseline general loss:", round(baseline_general, 4))
    print("Baseline Phase 4D QA loss:", round(baseline_qa, 4))
    print("Maximum allowed general loss:", round(general_limit, 4))

# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------

def save_resume79(step):
    torch.save({
        "model": model79.state_dict(),
        "optimizer": optimizer79.state_dict(),
        "scaler": scaler79.state_dict(),
        "step": step,
        "best_step": best_step,
        "best_qa": best_qa,
        "bad_checks": bad_checks,
        "history": history79,
        "rng_state": rng79.getstate(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "baseline_general": baseline_general,
        "general_limit": general_limit,
        "config": CFG79
    }, RESUME)

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

print("\nSTARTING PHASE 4D TRAINING\n")

for step in range(start_step + 1, CFG79["max_steps"] + 1):

    model79.train()
    optimizer79.zero_grad(set_to_none=True)

    running_loss = 0.0

    for _ in range(CFG79["accumulation"]):

        rows = (
            train_qa
            if rng79.random() < CFG79["qa_probability"]
            else train_general
        )

        x, y = batch79(rows)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            logits, _ = model79(x)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)).float(),
                y.reshape(-1),
                ignore_index=-100
            )

        running_loss += loss.item()

        scaler79.scale(
            loss / CFG79["accumulation"]
        ).backward()

    scaler79.unscale_(optimizer79)

    torch.nn.utils.clip_grad_norm_(
        model79.parameters(), 1.0
    )

    scaler79.step(optimizer79)
    scaler79.update()

    if step % CFG79["validation_every"] == 0:

        qa_loss = evaluate79(val_qa)
        general_loss = evaluate79(val_general)

        acceptable = general_loss <= general_limit
        improved = acceptable and qa_loss < best_qa - 0.001

        if improved:
            best_qa = qa_loss
            best_step = step
            bad_checks = 0

            torch.save({
                "model_state_dict": model79.state_dict(),
                "step": step,
                "qa_val_loss": qa_loss,
                "general_val_loss": general_loss,
                "config": CFG79
            }, BEST)
        else:
            bad_checks += 1

        record = {
            "step": step,
            "train_loss": running_loss / CFG79["accumulation"],
            "qa_val_loss": qa_loss,
            "general_val_loss": general_loss,
            "acceptable_general": acceptable,
            "best_step": best_step
        }

        history79.append(record)

        print(
            f"STEP {step:03d} | "
            f"Train {record['train_loss']:.4f} | "
            f"QA {qa_loss:.4f} | "
            f"General {general_loss:.4f} | "
            f"{'BEST' if improved else 'no improvement'}"
        )

        if step % CFG79["save_every"] == 0:
            save_resume79(step)
            print("  Resume checkpoint saved.")

        if bad_checks >= CFG79["patience"]:
            print("\nEARLY STOPPING triggered.")
            save_resume79(step)
            break

    elif step % CFG79["save_every"] == 0:
        save_resume79(step)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

with open(HISTORY, "w", encoding="utf-8") as f:
    json.dump({
        "configuration": CFG79,
        "baseline_general": baseline_general,
        "general_limit": general_limit,
        "best_step": best_step,
        "best_qa_loss": (
            best_qa if best_step is not None else None
        ),
        "history": history79
    }, f, indent=2)

print("\n" + "=" * 72)
print("CELL 79 COMPLETE")
print("=" * 72)
print("Best step:", best_step)
print("Best QA validation loss:",
      round(best_qa, 4) if best_step is not None else "NONE")
print("Original general loss:", round(baseline_general, 4))
print("Allowed general loss:", round(general_limit, 4))
print("Best checkpoint:", BEST if best_step else "NONE")
print("Resume checkpoint:", RESUME)
print("History:", HISTORY)


GRAC-LM v0.4 — CELL 79: PHASE 4D TRAINING
QA training: 2360
General training: 9276
QA validation groups: 59
General validation examples: 60

Evaluating Phase 4C starting checkpoint...
Baseline general loss: 3.6755
Baseline Phase 4D QA loss: 2.586
Maximum allowed general loss: 3.8755

STARTING PHASE 4D TRAINING

STEP 025 | Train 2.5547 | QA 1.3279 | General 3.7126 | BEST
STEP 050 | Train 1.9649 | QA 1.3146 | General 3.6969 | BEST
  Resume checkpoint saved.
STEP 075 | Train 2.3905 | QA 1.2580 | General 3.7033 | BEST
STEP 100 | Train 1.4845 | QA 1.2414 | General 3.6971 | BEST
  Resume checkpoint saved.
STEP 125 | Train 1.0245 | QA 1.2126 | General 3.6966 | BEST
STEP 150 | Train 2.3649 | QA 1.1939 | General 3.6952 | BEST
  Resume checkpoint saved.
STEP 175 | Train 1.7701 | QA 1.2035 | General 3.6953 | no improvement
STEP 200 | Train 2.3478 | QA 1.2083 | General 3.6939 | no improvement
  Resume checkpoint saved.
STEP 225 | Train 1.1477 | QA 1.1860 | General 3.6930 | BEST
STEP 250 | Train 1.

In [ ]:

import torch
import json
import re
from pathlib import Path
from collections import defaultdict

print("=" * 72)
print("GRAC-LM v0.4 — CELL 80: HELD-OUT QA ACCURACY")
print("=" * 72)

BASE80 = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
DATA80 = BASE80 / "phase4d/data"

CHECKPOINTS80 = {
    "PHASE 4C": BASE80 / "phase4c/GRAC-LM-v0.4-phase4c-best.pt",
    "PHASE 4D": BASE80 / "phase4d/GRAC-LM-v0.4-phase4d-best.pt"
}

for name, path in CHECKPOINTS80.items():
    assert path.exists(), f"Missing {name}: {path}"

with open(DATA80 / "validation.jsonl", encoding="utf-8") as f:
    all_rows80 = [json.loads(line) for line in f if line.strip()]

# One held-out question per fact/problem group.
groups80 = {}
for row in all_rows80:
    groups80.setdefault(row["id"], row)

questions80 = list(groups80.values())
assert len(questions80) == 59

device80 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model80 = GRACLM(GRAC_V04_CONFIG).to(device80)
model80.eval()

eos80 = tokenizer.token_to_id("<eos>")
limit80 = GRAC_V04_CONFIG["context_length"]

def normalize80(s):
    return " ".join(
        re.sub(r"[^\w\s]", "", s.casefold()).split()
    )

@torch.inference_mode()
def generate80(question, max_new_tokens=35):
    ids = tokenizer.encode(
        f"User: {' '.join(question.split())} Assistant: "
    ).ids

    generated = []

    for _ in range(max_new_tokens):
        if len(ids) >= limit80:
            break

        x = torch.tensor(
            [ids], dtype=torch.long, device=device80
        )

        logits, _ = model80(x)
        next_id = int(logits[0, -1].argmax())

        if next_id == eos80:
            break

        ids.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

results80 = {}

for name, path in CHECKPOINTS80.items():
    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )
    model80.load_state_dict(checkpoint["model_state_dict"])
    model80.eval()
    del checkpoint

    stats = defaultdict(lambda: {"correct": 0, "total": 0})
    records = []

    print("\n" + "=" * 72)
    print("EVALUATING", name)
    print("=" * 72)

    for row in questions80:
        answer = generate80(row["instruction"])
        correct = normalize80(answer) == normalize80(row["response"])

        category = row["category"]
        stats[category]["total"] += 1
        stats[category]["correct"] += int(correct)

        records.append({
            "id": row["id"],
            "category": category,
            "kind": row["kind"],
            "question": row["instruction"],
            "expected": row["response"],
            "generated": answer,
            "exact_match": correct
        })

        print(
            "✓" if correct else "✗",
            row["instruction"],
            "| Expected:", row["response"],
            "| GRAC:", answer[:100]
        )

    total_correct = sum(r["exact_match"] for r in records)

    results80[name] = {
        "correct": total_correct,
        "total": len(records),
        "accuracy": total_correct / len(records),
        "categories": dict(stats),
        "records": records
    }

print("\n" + "=" * 72)
print("CELL 80 — FINAL COMPARISON")
print("=" * 72)

for name, result in results80.items():
    print(
        f"\n{name}: {result['correct']}/{result['total']} "
        f"({100 * result['accuracy']:.1f}%)"
    )

    for category, s in result["categories"].items():
        print(
            f"  {category}: {s['correct']}/{s['total']}"
        )

output80 = BASE80 / "phase4d/cell80_validation_accuracy.json"

with open(output80, "w", encoding="utf-8") as f:
    json.dump(results80, f, indent=2, ensure_ascii=False)

print("\nSaved:", output80)
print("Reserved test set: NOT USED")
print("CELL 80 COMPLETE!")


GRAC-LM v0.4 — CELL 80: HELD-OUT QA ACCURACY

EVALUATING PHASE 4C
✗ What is the difference between 19 and 18? | Expected: 1. | GRAC: 4.
✗ Ignore the number 99. What is 21 - 19? | Expected: 2. | GRAC: 1. The number 99.
✗ What is 11 - 3? | Expected: 8. | GRAC: 6.
✗ Calculate 22 minus 20. | Expected: 2. | GRAC: 6.
✗ Ignore the number 99. What is 18 - 16? | Expected: 2. | GRAC: 4.
✗ What is the product of 6 and 8? | Expected: 48. | GRAC: 6.
✗ Calculate 9 minus 6. | Expected: 3. | GRAC: 6.
✗ Ignore the number 99. What is 6 + 12? | Expected: 18. | GRAC: 4.
✗ Ignore the number 99. What is 21 - 9? | Expected: 12. | GRAC: 4.
✗ What is 4 times 4? | Expected: 16. | GRAC: 5.
✗ Subtract 12 from 20. | Expected: 8. | GRAC: 6.
✗ Multiply 5 by 8. | Expected: 40. | GRAC: 6.
✗ What is 3 times 6? | Expected: 18. | GRAC: 6.
✗ What is 25 - 5? | Expected: 20. | GRAC: 5.
✗ What is 19 - 16? | Expected: 3. | GRAC: 6.
✗ Calculate 7 plus 15. | Expected: 22. | GRAC: 6.
✗ Ignore the number 99. What is 13 - 1? | Exp

In [ ]:

import torch
import json
import random
import re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

print("=" * 72)
print("GRAC-LM v0.4 — CELL 81: FINAL DIAGNOSTIC & SESSION SAVE")
print("=" * 72)

BASE81 = Path("/content/drive/MyDrive/GRAC/GRAC-LM-v0.4")
OUT81 = BASE81 / "phase4d"
DATA81 = OUT81 / "data"

CHECKPOINT81 = OUT81 / "GRAC-LM-v0.4-phase4d-best.pt"

assert CHECKPOINT81.exists(), "Phase 4D checkpoint missing!"
assert "GRACLM" in globals()
assert "tokenizer" in globals()
assert "GRAC_V04_CONFIG" in globals()

def read81(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train81 = read81(DATA81 / "train.jsonl")
val81 = read81(DATA81 / "validation.jsonl")

# Sample 60 training groups and use all 59 validation groups.
rng81 = random.Random(2026)

train_groups81 = {}
val_groups81 = {}

for row in train81:
    train_groups81.setdefault(row["id"], []).append(row)

for row in val81:
    val_groups81.setdefault(row["id"], []).append(row)

sampled_train_ids81 = rng81.sample(
    list(train_groups81),
    min(60, len(train_groups81))
)

# Use two question variations for each selected group.
train_questions81 = []
for group_id in sampled_train_ids81:
    variations = train_groups81[group_id]
    train_questions81.extend(
        rng81.sample(variations, min(2, len(variations)))
    )

val_questions81 = [
    rows[0]
    for rows in val_groups81.values()
]

device81 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model81 = GRACLM(GRAC_V04_CONFIG).to(device81)

state81 = torch.load(
    CHECKPOINT81,
    map_location="cpu",
    weights_only=False
)
model81.load_state_dict(state81["model_state_dict"])
model81.eval()
del state81

eos81 = tokenizer.token_to_id("<eos>")
limit81 = GRAC_V04_CONFIG["context_length"]

def normalize81(text):
    return " ".join(
        re.sub(r"[^\w\s]", "", text.casefold()).split()
    )

@torch.inference_mode()
def generate81(question, max_new_tokens=25):
    ids = tokenizer.encode(
        f"User: {' '.join(question.split())} Assistant: "
    ).ids

    generated = []

    for _ in range(max_new_tokens):
        if len(ids) >= limit81:
            break

        x = torch.tensor(
            [ids], dtype=torch.long, device=device81
        )

        logits, _ = model81(x)
        next_id = int(logits[0, -1].argmax().item())

        if next_id == eos81:
            break

        ids.append(next_id)
        generated.append(next_id)

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

def evaluate81(rows):
    records = []
    by_kind = defaultdict(lambda: {"correct": 0, "total": 0})
    outputs = Counter()

    for row in rows:
        generated = generate81(row["instruction"])
        correct = normalize81(generated) == normalize81(row["response"])

        by_kind[row["kind"]]["total"] += 1
        by_kind[row["kind"]]["correct"] += int(correct)

        outputs[normalize81(generated)] += 1

        records.append({
            "id": row["id"],
            "kind": row["kind"],
            "question": row["instruction"],
            "expected": row["response"],
            "generated": generated,
            "correct": correct
        })

    return {
        "correct": sum(r["correct"] for r in records),
        "total": len(records),
        "by_kind": dict(by_kind),
        "top_outputs": outputs.most_common(10),
        "records": records
    }

print("\nEvaluating training questions...")
train_result81 = evaluate81(train_questions81)

print("Evaluating held-out validation questions...")
val_result81 = evaluate81(val_questions81)

print("\n" + "=" * 72)
print("ACCURACY REPORT")
print("=" * 72)

for label, result in [
    ("TRAIN", train_result81),
    ("VALIDATION", val_result81)
]:
    accuracy = 100 * result["correct"] / result["total"]

    print(
        f"{label}: {result['correct']}/{result['total']} "
        f"({accuracy:.1f}%)"
    )

    for kind, stats in sorted(result["by_kind"].items()):
        print(
            f"  {kind}: "
            f"{stats['correct']}/{stats['total']}"
        )

    print("  Most frequent outputs:", result["top_outputs"][:5])

# ------------------------------------------------------------
# TOKENIZER NUMBER AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("NUMBER TOKENIZER AUDIT")
print("=" * 72)

token_audit81 = {}

for number in ["1", "6", "11", "18", "21", "25", "48", "99"]:
    ids = tokenizer.encode(number).ids
    decoded = tokenizer.decode(ids)

    token_audit81[number] = {
        "ids": ids,
        "decoded": decoded
    }

    print(
        f"{number:>3s} -> IDs {ids} -> {decoded!r}"
    )

# ------------------------------------------------------------
# VERIFY SAVED CHECKPOINTS
# ------------------------------------------------------------

checkpoint_paths81 = {
    "Phase 4 Expanded":
        BASE81 / "phase4/GRAC-LM-v0.4-expanded-best.pt",
    "Phase 4B":
        BASE81 / "phase4b/GRAC-LM-v0.4-phase4b-step300.pt",
    "Phase 4C":
        BASE81 / "phase4c/GRAC-LM-v0.4-phase4c-best.pt",
    "Phase 4D":
        BASE81 / "phase4d/GRAC-LM-v0.4-phase4d-best.pt",
    "Phase 4D resume":
        BASE81 / "phase4d/GRAC-LM-v0.4-phase4d-resume.pt"
}

checkpoint_status81 = {}

for name, path in checkpoint_paths81.items():
    exists = path.is_file()
    size_mb = round(path.stat().st_size / (1024**2), 2) if exists else None

    checkpoint_status81[name] = {
        "exists": exists,
        "size_mb": size_mb,
        "path": str(path)
    }

    print(
        f"{name}: "
        f"{'SAVED' if exists else 'MISSING'} "
        f"{size_mb if exists else ''} MB"
    )

# ------------------------------------------------------------
# SAVE SESSION REPORT
# ------------------------------------------------------------

session81 = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project": "GRAC-LM v0.4",
    "phase": "4D",
    "last_completed_cell": 81,
    "train_diagnostic": train_result81,
    "validation_diagnostic": val_result81,
    "tokenizer_number_audit": token_audit81,
    "checkpoint_status": checkpoint_status81,
    "next_steps": [
        "Review training versus held-out accuracy.",
        "Determine whether arithmetic failure is underlearning or memorization.",
        "Review numeric tokenization.",
        "Design Phase 4E without starting from scratch.",
        "Preserve the Phase 4C and Phase 4D checkpoints."
    ]
}

report81 = OUT81 / "cell81_session_report.json"

with open(report81, "w", encoding="utf-8") as f:
    json.dump(session81, f, indent=2, ensure_ascii=False)

print("\nSession report saved:", report81)

missing81 = [
    name for name, status in checkpoint_status81.items()
    if not status["exists"]
]

if missing81:
    print("WARNING — Missing checkpoints:", missing81)
else:
    print("ALL IMPORTANT CHECKPOINTS VERIFIED!")

print("\nCELL 81 COMPLETE — SESSION READY TO CLOSE.")


GRAC-LM v0.4 — CELL 81: FINAL DIAGNOSTIC & SESSION SAVE

Evaluating training questions...
Evaluating held-out validation questions...

ACCURACY REPORT
TRAIN: 10/120 (8.3%)
  addition: 3/48
  capital: 2/4
  definition: 1/6
  multiplication: 0/10
  subtraction: 4/52
  Most frequent outputs: [('1', 43), ('18', 23), ('22', 15), ('21', 11), ('11', 9)]
VALIDATION: 3/59 (5.1%)
  addition: 0/14
  capital: 0/2
  definition: 1/2
  multiplication: 0/8
  subtraction: 2/33
  Most frequent outputs: [('1', 27), ('18', 12), ('11', 6), ('21', 5), ('16', 2)]

NUMBER TOKENIZER AUDIT
  1 -> IDs [20] -> '1'
  6 -> IDs [25] -> '6'
 11 -> IDs [20, 20] -> '11'
 18 -> IDs [20, 27] -> '18'
 21 -> IDs [21, 20] -> '21'
 25 -> IDs [21, 24] -> '25'
 48 -> IDs [23, 27] -> '48'
 99 -> IDs [28, 28] -> '99'
Phase 4 Expanded: SAVED 208.65 MB
Phase 4B: SAVED 208.65 MB
Phase 4C: SAVED 208.65 MB
Phase 4D: SAVED 208.65 MB
Phase 4D resume: SAVED 625.97 MB

Session report saved: /content/drive/MyDrive/GRAC/GRAC-LM-v0.4/phase4